# 🩸 Скрининг риска анемии по фото глаза — конъюнктива + склера (версия 10fix — исправлена по аудиту)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhansaya-afk/jjghg/blob/main/notebooks/anemia_screening_v10_fixed.ipynb)

> **Что исправлено по сравнению с v10** (полный список — `docs/v10_audit.md` в репозитории): точность теперь считается вложенной перекрёстной проверкой (порог HHR, набор признаков, модель и калибровка выбираются без проверяемых пациентов); промахи автопоиска конъюнктивы входят в проверку; несколько фото одного человека — одна группа; свет моделируется в линейном RGB (реально 2700–7500 K); персональная калибровка — отдельно для каждого человека; Hb в ячейке 5 можно вводить в г/л; байесовский пересчёт учитывает долю анемии в обучающих данных; ферритин у детей до 5 лет — порог 12 мкг/л. Старый файл `anemia_models_v10.joblib` не подходит — модели обучатся заново.

**Это скрининг, а не диагноз.** Система оценивает гемоглобин по цвету конъюнктивы и склеры и показывает риск анемии. Точное значение Hb, ферритин и тип анемии даёт только анализ крови (ячейки 5 и 8).

### Как пользоваться (без Google Диска)
1. **Первый раз:** перетащите zip **CP-AnemiC** в панель «Файлы» слева (нужен для модели детей до 5 лет) → «Среда выполнения → Выполнить все». В ячейке 1 нажмите «Отмена». Обучение ≈ 30–40 мин, один раз; в конце скачается файл `anemia_models_v10fix.joblib` — **сохраните его**.
2. **Дальше:** «Выполнить все» → в ячейке 1 выберите `anemia_models_v10fix.joblib` (и `local_data.zip`, если добавляли фото в ячейке 7) → в ячейке 4 выберите фото глаза. ≈ 1 мин.

### Что честно показывает система
- **Чем определено и почему:** у конъюнктивы и склеры своя ожидаемая ошибка (проверка на невиденных пациентах при обычном и другом свете × качество снимка). Согласуются — общий итог; расходятся — берётся метод с меньшей ошибкой, причина пишется.
- **Свет и камера:** каждый обучающий снимок «пересниман» при разных лампах (2700–7500 K), экспозиции и обработке телефона; ячейка 3 печатает ошибку «при обычном / при другом свете» и «как было до v10». Белый лист бумаги в кадре (необязательно) — самый точный эталон цвета.
- **Как найдена конъюнктива:** моделью, обученной на разметке врачей, — только если на проверке она совпадает с врачами лучше правил; иначе правилами.
- **Точность метода на проверке** (±г/л на пациентах, которых модель не видела).
- **«Оценка ненадёжна»**, если снимок сильно не похож на обучающие; **«степень по фото не определить»**, если сигнал слабый.

### Как снимать
Глаз крупно, нижнее веко оттянуто, белок виден, дневной свет без вспышки и фильтров. Лучше 2–3 снимка.

| Ячейка | Что делает |
|---|---|
| 1 | Настройка (Диск или файл моделей) |
| 2 | Функции (поиск глаза, конъюнктивы, склеры, цветовые индексы, нормы ВОЗ) |
| 3 | Датасеты и модели (обучение один раз) |
| 4 | **📷 Анализ фото** — пол и возраст в полях справа |
| 5 | (необяз.) Hb из анализа крови + персональная калибровка |
| 8 | **Этап 2:** профиль возможной причины анемии по анализу крови (или анкета: что сдать) |
| 6 | (для защиты) Точность: ЧЕСТНО (вложенная проверка) и «оптимистично» как в v10 |
| 7 | Добавить фото с известным Hb (повышает точность для вашего телефона) |


In [ ]:
#@title ЯЧЕЙКА 1 — НАСТРОЙКА (Google Диск или работа без него) { display-mode: "form" }
#@markdown **Google Диск.** Если доступ к Диску нельзя давать каждый раз (например, аккаунт под родительским
#@markdown контролем) — снимите галочку. Тогда модели хранятся в файле `anemia_models_v10.joblib` на вашем компьютере.
Использовать_Google_Диск = False  #@param {type:"boolean"}
#@markdown *(Необязательно)* ссылка на файл моделей, открытый «для всех, у кого есть ссылка» — тогда он скачается сам:
Ссылка_на_файл_моделей = ''  #@param {type:"string"}

import os, sys, glob, shutil, subprocess, warnings, time
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 100
SEED = 42
OUT = 'results'
os.makedirs(OUT, exist_ok=True)
MODEL_NAME = 'anemia_models_v10fix.joblib'   # исправленная версия (аудит): старый файл v10 не подходит

DRIVE_DIR, ON_DRIVE = None, False
if Использовать_Google_Диск:
    try:
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        if os.path.ismount('/content/drive'):
            DRIVE_DIR, ON_DRIVE = '/content/drive/MyDrive/anemia_poc', True
    except Exception as e:
        print('⚠️ Google Диск не подключился:', str(e)[:150])
if DRIVE_DIR is None:
    DRIVE_DIR = 'anemia_local'
os.makedirs(DRIVE_DIR, exist_ok=True)
MODEL_PATH = f'{DRIVE_DIR}/{MODEL_NAME}'

if not ON_DRIVE:
    # файлы, перетащенные в панель «Файлы» Colab (модели, архив CP-AnemiC), — в рабочую папку
    for f in glob.glob('*.joblib') + glob.glob('*.zip'):
        dst = MODEL_PATH if f.endswith('.joblib') else f'{DRIVE_DIR}/{os.path.basename(f)}'
        if f.endswith('.joblib') and os.path.basename(f) != MODEL_NAME:
            print(f'⚠️ {f} — файл моделей другой версии; если он устарел, ячейка 3 обучит {MODEL_NAME}.')
        if not os.path.exists(dst):
            shutil.move(f, dst)
    # модели по ссылке (Google Диск «для всех, у кого есть ссылка», без входа в аккаунт)
    if not os.path.exists(MODEL_PATH) and Ссылка_на_файл_моделей.strip():
        try:
            import gdown
            gdown.download(Ссылка_на_файл_моделей.strip(), MODEL_PATH, quiet=True, fuzzy=True)
        except Exception as e:
            print('⚠️ Не удалось скачать модели по ссылке:', str(e)[:120])
        if os.path.exists(MODEL_PATH) and os.path.getsize(MODEL_PATH) < 10_000:
            os.remove(MODEL_PATH)                      # скачалась страница ошибки, а не файл
            print('⚠️ По ссылке нет файла моделей — проверьте, что доступ открыт «для всех, у кого есть ссылка».')
    # иначе — попросить файл с компьютера (одно нажатие)
    if not os.path.exists(MODEL_PATH):
        try:
            from google.colab import files
            print(f'📂 Выберите файл {MODEL_NAME} на компьютере (его скачала ячейка 3 в прошлый раз).\n'
                  '   Можно сразу выбрать и local_data.zip (ваши фото с известным Hb из ячейки 7).\n'
                  '   Первый запуск и файла ещё нет? Нажмите «Отмена» — модели обучатся заново (≈30–40 мин, один раз).')
            for name in files.upload().keys():
                if name.endswith('.joblib'):
                    if name != MODEL_NAME:
                        print(f'⚠️ {name} — файл моделей СТАРОЙ версии; ячейка 3 обучит новую ({MODEL_NAME}).')
                    shutil.move(name, MODEL_PATH)
                elif name.endswith('.zip'):
                    shutil.move(name, f'{DRIVE_DIR}/{name}')
        except Exception:
            pass

# ваши фото с известным Hb (ячейка 7), сохранённые в прошлый раз как local_data.zip
_lz = f'{DRIVE_DIR}/local_data.zip'
if os.path.exists(_lz) and not os.path.exists(f'{DRIVE_DIR}/local_data/features.csv'):
    import zipfile
    with zipfile.ZipFile(_lz) as z:
        z.extractall(DRIVE_DIR)
    print('📷 Ваши фото с известным Hb восстановлены из local_data.zip')

print(f'   OpenCV {cv2.__version__}')
print(f"✅ Ячейка 1 готова. Режим: {'Google Диск' if ON_DRIVE else 'без Google Диска'} · папка: {DRIVE_DIR}")
if os.path.exists(MODEL_PATH):
    print('📦 Модели найдены — обучение НЕ понадобится (ячейка 3 займёт секунды).')
else:
    print('🆕 Моделей нет — ячейка 3 обучит их (один раз, ≈30–40 мин).'
          + ('' if ON_DRIVE else f' В конце ячейка 3 скачает {MODEL_NAME} на компьютер — СОХРАНИТЕ его.'))


In [ ]:
# ============================================================
# ЯЧЕЙКА 2 — БИБЛИОТЕКА ФУНКЦИЙ (только определения, выполняется за секунды)
# ============================================================
import os, re, glob, json, math, warnings, shutil
import numpy as np, pandas as pd, cv2
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')
try:
    cv2.utils.logging.setLogLevel(cv2.utils.logging.LOG_LEVEL_ERROR)   # без служебных предупреждений OpenCV 5
except Exception:
    pass
EPS = 1e-8
HUE_GRID = list(range(300, 360, 5))
TARGET_R = 60                       # радиус радужки после нормализации масштаба (px)

# ---------------------------------------------------------------- чтение изображений
def resize_max(img, max_side, nearest=False):
    h, w = img.shape[:2]
    s = max_side / max(h, w)
    if s >= 1:
        return img
    return cv2.resize(img, (int(w * s), int(h * s)),
                      interpolation=cv2.INTER_NEAREST if nearest else cv2.INTER_AREA)

def load_image(path, max_side=2400):
    """RGB float32 [0,1]. Поддержка JPG/PNG/HEIC/WEBP, учёт поворота из EXIF."""
    bgr = cv2.imread(path, cv2.IMREAD_COLOR)
    if bgr is None:
        try:
            from PIL import Image, ImageOps
            try:
                from pillow_heif import register_heif_opener
            except ImportError:
                import subprocess, sys
                subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pillow-heif'], check=False)
                from pillow_heif import register_heif_opener
            register_heif_opener()
            rgb8 = np.array(ImageOps.exif_transpose(Image.open(path)).convert('RGB'))
        except Exception as e:
            raise ValueError(f'Не удалось открыть файл {os.path.basename(path)}: {e}')
    else:
        rgb8 = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    return resize_max(rgb8.astype(np.float32) / 255.0, max_side)

def load_segment(path, max_side=800):
    """Ручная разметка конъюнктивы: ткань на прозрачном/чёрном фоне -> (RGB, маска)."""
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im is None:
        raise FileNotFoundError(path)
    if im.ndim == 2:
        im = cv2.cvtColor(im, cv2.COLOR_GRAY2BGR)
    if im.shape[2] == 4:
        mask, im = im[..., 3] > 0, im[..., :3]
    else:
        mask = im.max(axis=2) > 15
    if im.dtype != np.uint8:
        im = (im / 257).astype(np.uint8)
    rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB).astype(np.float32) / 255
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    return resize_max(rgb, max_side), resize_max(mask, max_side, nearest=True) > 0

# ---------------------------------------------------------------- цвет
def lab_of(rgb):
    lab = cv2.cvtColor(rgb.astype(np.float32), cv2.COLOR_RGB2Lab)
    L, A, B = lab[..., 0], lab[..., 1], lab[..., 2]
    return L, A, B, np.sqrt(A ** 2 + B ** 2)

def rgb_to_hsi(rgb):
    """HSI: H — оттенок (почти не зависит от яркости света), S — насыщенность, I — яркость."""
    R, G, B = rgb[..., 0], rgb[..., 1], rgb[..., 2]
    I = (R + G + B) / 3
    S = 1 - np.minimum(np.minimum(R, G), B) / (I + EPS)
    num = 0.5 * ((R - G) + (R - B))
    den = np.sqrt((R - G) ** 2 + (R - B) * (G - B)) + EPS
    th = np.degrees(np.arccos(np.clip(num / den, -1, 1)))
    return np.where(B <= G, th, 360 - th), np.clip(S, 0, 1), I

def erythema_map(rgb):
    """Индекс эритемы EI = log10(R) - log10(G): Hb поглощает зелёный, почти не поглощает красный."""
    return np.log10(rgb[..., 0] * 255 + 1) - np.log10(rgb[..., 1] * 255 + 1)

def valid_tissue(rgb, mask):
    """Пиксели ткани без бликов, теней и бесцветных участков."""
    H, S, I = rgb_to_hsi(rgb)
    hsv = cv2.cvtColor(rgb.astype(np.float32), cv2.COLOR_RGB2HSV)
    glare = (hsv[..., 2] > 0.92) & (hsv[..., 1] < 0.20)
    glare = cv2.dilate(glare.astype(np.uint8), np.ones((5, 5), np.uint8)) > 0
    inner = cv2.erode(mask.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    return inner & ~glare & (I > 0.08) & (S > 0.03)

def compute_indices(rgb, mask):
    """Цветовые индексы конъюнктивы (EI, HHR, PVM, CIELAB, энтропия). None — мало ткани."""
    valid = valid_tissue(rgb, mask)
    if valid.sum() < 200:
        return None
    H, S, _ = rgb_to_hsi(rgb)
    L, A, Bb, _ = lab_of(rgb)
    EI = erythema_map(rgb)
    R8, G8 = rgb[..., 0] * 255, rgb[..., 1] * 255
    dist = cv2.distanceTransform(mask.astype(np.uint8), cv2.DIST_L2, 5)
    core = valid & (dist >= np.median(dist[valid]))
    core = core if core.sum() >= 50 else valid
    gray = cv2.cvtColor((np.clip(rgb, 0, 1) * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    p = np.bincount(gray[valid], minlength=256).astype(float); p = p[p > 0] / p.sum()
    ys = np.nonzero(valid)[0]
    lower = valid & (np.arange(valid.shape[0])[:, None] >= np.median(ys))
    f = {'EI': float(EI[valid].mean()),
         'EI_lower': float(EI[lower].mean()) if lower.sum() > 50 else float(EI[valid].mean()),
         'a_star': float(A[valid].mean()), 'L_star': float(L[valid].mean()), 'b_star': float(Bb[valid].mean()),
         'S': float(S[valid].mean()),
         'r_norm': float((rgb[..., 0] / (rgb.sum(2) + EPS))[valid].mean()),
         'g_norm': float((rgb[..., 1] / (rgb.sum(2) + EPS))[valid].mean()),
         'PVM_R': float(np.median(R8[core])), 'PVM_G': float(np.median(G8[core])),
         'PVM_EI': float(np.median(EI[core])), 'PVM_a': float(np.median(A[core])),
         'entropy': float(-(p * np.log2(p)).sum()),
         'n_px': int(valid.sum())}
    for T in HUE_GRID:
        f[f'HHR_{T}'] = float(np.mean(H[valid] >= T))
    return f

# ---------------------------------------------------------------- поиск радужки, белка, конъюнктивы
def sclera_like(rgb, b_max=6):
    """Белок глаза: почти бесцветный (малая хрома, b*≈0). У кожи b* заметно выше."""
    L, A, B, C = lab_of(rgb)
    S = cv2.cvtColor(rgb.astype(np.float32), cv2.COLOR_RGB2HSV)[..., 1]
    return (S < 0.22) & (np.abs(A) < 7) & (B < b_max) & (L > 35)

_ANG_R = np.linspace(-np.pi / 4, np.pi / 4, 24)
_ANG_L = _ANG_R + np.pi
_ANG_ALL = np.linspace(0, 2 * np.pi, 36, endpoint=False)

def _samp(img, cx, cy, radii, ang):
    h, w = img.shape
    rr = np.asarray(radii)[:, None]
    x = np.clip((cx + rr * np.cos(ang)).astype(int), 0, w - 1)
    y = np.clip((cy + rr * np.sin(ang)).astype(int), 0, h - 1)
    return img[y, x]

def circle_score(L, W, cx, cy, r):
    """«Это радужка?» — контраст края на боковых дугах × белок с обеих сторон − белок внутри."""
    con = float(min(np.median(_samp(L, cx, cy, [1.1 * r, 1.25 * r, 1.4 * r], a))
                    - np.median(_samp(L, cx, cy, [0.6 * r, 0.75 * r, 0.9 * r], a)) for a in (_ANG_R, _ANG_L)))
    wr = _samp(W, cx, cy, [1.15 * r, 1.3 * r, 1.45 * r], _ANG_R).mean()
    wl = _samp(W, cx, cy, [1.15 * r, 1.3 * r, 1.45 * r], _ANG_L).mean()
    win = _samp(W, cx, cy, [0.3 * r, 0.5 * r, 0.7 * r], _ANG_ALL).mean()
    return con * (0.2 + wl + wr) - 60 * win, con

def _refine(L, W, cx, cy, r):
    best = (circle_score(L, W, cx, cy, r)[0], cx, cy, r)
    for (r_lo, r_hi, nr), span, steps in (((0.6, 2.0, 8), 0.5, 5), ((0.9, 1.1, 3), 0.12, 5)):
        _, bx, by, br = best
        for rr in np.linspace(r_lo * br, r_hi * br, nr):
            for dx in np.linspace(-span * br, span * br, steps):
                for dy in np.linspace(-span * br, span * br, steps):
                    sc = circle_score(L, W, bx + dx, by + dy, rr)[0]
                    if sc > best[0]:
                        best = (sc, bx + dx, by + dy, rr)
    return best

def find_iris(rgb, white):
    h, w = rgb.shape[:2]
    L = cv2.GaussianBlur(lab_of(rgb)[0], (0, 0), 1.5)
    m = min(h, w)
    k = max(3, int(m / 80)) | 1
    ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
    cands = []
    n, lbl, st, _ = cv2.connectedComponentsWithStats(white.astype(np.uint8))
    for j in np.argsort(-st[1:, cv2.CC_STAT_AREA])[:6] + 1:
        if st[j, cv2.CC_STAT_AREA] < 0.0005 * h * w:
            break
        comp = (lbl == j).astype(np.uint8)
        hull = np.zeros_like(comp); cv2.fillConvexPoly(hull, cv2.convexHull(cv2.findNonZero(comp)), 1)
        hole = cv2.morphologyEx(((hull > 0) & (comp == 0) & (L < np.percentile(L[comp > 0], 50))).astype(np.uint8),
                                cv2.MORPH_OPEN, ker)
        m2, l2, s2, c2 = cv2.connectedComponentsWithStats(hole)
        for i in range(1, m2):
            if s2[i, cv2.CC_STAT_AREA] > 0.0003 * h * w:
                cands.append((c2[i][0], c2[i][1], max(s2[i, cv2.CC_STAT_WIDTH], s2[i, cv2.CC_STAT_HEIGHT]) / 2))
                # Внутри «проёма глаза» радужка — самый тёмный компактный круг: центр = максимум
                # карты расстояний тёмной области (ресницы тонкие и убираются морфологией)
                reg = l2 == i
                dk = (reg & (L < np.percentile(L[reg], 35))).astype(np.uint8)
                kk = max(3, int(s2[i, cv2.CC_STAT_WIDTH] / 12)) | 1
                dk = cv2.morphologyEx(dk, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kk, kk)))
                if dk.sum() > 30:
                    dt = cv2.distanceTransform(dk, cv2.DIST_L2, 5)
                    yx = np.unravel_index(np.argmax(dt), dt.shape)
                    cands.append((float(yx[1]), float(yx[0]), float(dt.max())))
    kd = max(3, int(m / 40)) | 1
    dark = cv2.morphologyEx((L < np.percentile(L, 8)).astype(np.uint8), cv2.MORPH_OPEN,
                            cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kd, kd)))
    n3, _, s3, c3 = cv2.connectedComponentsWithStats(dark)
    for i in np.argsort(-s3[1:, cv2.CC_STAT_AREA])[:5] + 1:
        bw, bh, ar = s3[i, cv2.CC_STAT_WIDTH], s3[i, cv2.CC_STAT_HEIGHT], s3[i, cv2.CC_STAT_AREA]
        if ar > 0.0003 * h * w and 0.4 < bw / max(bh, 1) < 2.5 and ar / (bw * bh) > 0.5:
            cands.append((c3[i][0], c3[i][1], np.sqrt(ar / np.pi)))
    hc = cv2.HoughCircles((L * 2.55).astype(np.uint8), cv2.HOUGH_GRADIENT, dp=1.5, minDist=m / 12,
                          param1=80, param2=18, minRadius=max(3, int(0.03 * m)), maxRadius=int(0.3 * m))
    if hc is not None:
        cands += [tuple(c) for c in hc[0][:12]]
    W = white.astype(np.float32)
    cands = [c for c in cands if c[2] >= 4]
    if not cands:
        return None, []
    pre = [max(circle_score(L, W, cx, cy, k * r)[0] for k in (0.7, 1.0, 1.5)) for cx, cy, r in cands]
    cands = [cands[i] for i in np.argsort(pre)[::-1][:10]]
    scored = []
    for cx, cy, r in cands:
        score, cx, cy, r = _refine(L, W, cx, cy, r)
        con = circle_score(L, W, cx, cy, r)[1]
        if con < 5 or cx - r < 0 or cx + r > w or cy - 0.8 * r < 0 or cy + r > h:
            continue
        scored.append((score, cx, cy, r))
    if not scored:
        return None, []
    scored.sort(reverse=True)
    return scored[0][1:], scored[:6]

def _find(rgb, b_max, iris_override=None, xlim=None):
    """Белок -> радужка -> полоса под ними до линии ресниц -> конъюнктива."""
    h, w = rgb.shape[:2]
    L, A, B, C = lab_of(rgb)
    S = cv2.cvtColor(rgb.astype(np.float32), cv2.COLOR_RGB2HSV)[..., 1]
    k = max(3, int(min(h, w) / 100)) | 1
    white = cv2.morphologyEx(sclera_like(rgb, b_max).astype(np.uint8), cv2.MORPH_OPEN,
                             cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))) > 0
    if iris_override is not None:
        iris, top = tuple(iris_override), [(0.0,) + tuple(iris_override)]
    else:
        iris, top = find_iris(rgb, white)
    dbg = {'white': white, 'iris_candidates': top, 'iris': iris, 'zone': None, 'sclera': None,
           'iris_score': top[0][0] if top else -1e9}
    if iris is None:
        return None, dbg, 'не найдена радужка'
    cx, cy, r = iris
    yy, xx = np.mgrid[0:h, 0:w]
    dd = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    cand_s = white & (dd < 3.0 * r) & (np.abs(yy - cy) < 1.3 * r)
    # Эталон цвета склеры — белые пиксели сразу слева/справа от радужки (это точно белок).
    side = cand_s & (dd > 1.05 * r) & (dd < 2.2 * r) & (np.abs(yy - cy) < 0.5 * r)
    a_ref = float(np.median(A[side])) if side.sum() >= 30 else 0.0
    # Ткань «розовее белка» — это конъюнктива, даже если она бледная (анемия!), а не склера.
    # Медианное сглаживание убирает тонкие сосуды на белке (их краснота не делает белок конъюнктивой),
    # а сплошная розовая ткань конъюнктивы — даже бледная — остаётся розовой.
    kb = max(5, int(r / 5)) | 1
    As = cv2.medianBlur(np.clip(A + 64, 0, 255).astype(np.uint8), min(kb, 15)).astype(np.float32) - 64
    pinker = (As > a_ref + 3.0) | ((yy > cy + 1.2 * r) & (As > a_ref + 1.5))
    # Белок с сосудами: такой же светлый, как эталон белка, и лишь слегка розовый — это всё ещё склера
    if side.sum() >= 30:
        L_ref = float(np.median(L[side]))
        Ls = cv2.medianBlur(np.clip(L * 2.55, 0, 255).astype(np.uint8), min(kb, 15)).astype(np.float32) / 2.55
        pinker &= ~((Ls > L_ref - 6) & (As < a_ref + 8))
    cand_s &= ~pinker
    ring = (dd > 0.95 * r) & (dd < 1.5 * r) & (np.abs(yy - cy) < 0.7 * r)
    ns, ls, _, _ = cv2.connectedComponentsWithStats(cand_s.astype(np.uint8))
    touch = np.unique(ls[cand_s & ring]); touch = touch[touch > 0]
    sclera = np.isin(ls, touch) if len(touch) else np.zeros_like(cand_s)
    sclera &= dd > 1.03 * r                             # внутрь радужки склера не заходит
    dbg['sclera'] = sclera
    white = white & ~pinker                            # бледная конъюнктива больше не считается «белым»
    lash = (L < 45) & (C < 10)                         # ресницы: тёмные И бесцветные
    zone = np.zeros((h, w), bool)
    xa, xb = int(max(0, cx - 1.8 * r)), int(min(w, cx + 1.8 * r))
    if xlim is not None:
        xa, xb = max(xa, int(xlim[0])), min(xb, int(xlim[1]))
    for x in range(xa, xb):
        ys = np.flatnonzero(sclera[:, x]); ys = ys[ys > cy - 0.3 * r]
        if abs(x - cx) <= 0.9 * r:
            y0 = int(cy + np.sqrt(max(r * r - (x - cx) ** 2, 0)))
            if len(ys):
                y0 = max(y0, ys.max())
        elif len(ys):
            y0 = ys.max()
        else:
            y0 = int(cy + 0.5 * r)
        y1 = min(h, y0 + int(1.8 * r))
        # Линия ресниц = участок из ≥3 тёмных бесцветных пикселей подряд, не ближе 0.45 r от начала.
        # (Тонкая тень на границе белка и века раньше обрывала поиск слишком рано.)
        col = lash[y0:y1, x].astype(np.int8)
        run = np.convolve(col, np.ones(3, np.int8), mode='valid') == 3 if len(col) >= 3 else np.zeros(0, bool)
        stop = np.flatnonzero(run); stop = stop[stop > 0.45 * r]
        zone[y0:(y0 + stop[0]) if len(stop) else y1, x] = True
    dbg['zone'] = zone
    # «Самые красные» пиксели НЕ отбираем — иначе бледная (анемичная) ткань выпадала бы из измерения
    cand = zone & ~white & (A > 2) & (C > 4) & (L > 15) & (S < 0.8)
    k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    mm = cv2.morphologyEx(cv2.morphologyEx(cand.astype(np.uint8), cv2.MORPH_OPEN, k5), cv2.MORPH_CLOSE, k5)
    n, lbl2, st2, _ = cv2.connectedComponentsWithStats(mm)
    if n <= 1:
        return None, dbg, 'под глазом не найдена розовая ткань'
    mm = lbl2 == 1 + np.argmax(st2[1:, cv2.CC_STAT_AREA])
    # Отделяем пальпебральную конъюнктиву (красная ткань века) от белёсой полосы над ней
    # (глазное яблоко под конъюнктивой — по цвету это ещё белок). Граница — середина между цветом белка
    # и самой красной частью найденной ткани. При бледной (анемичной) конъюнктиве контраст мал,
    # и уточнение не применяется — бледная ткань НЕ выбрасывается.
    a_hi = float(np.percentile(As[mm], 90))
    if a_hi - a_ref > 6:
        thr_a = a_ref + 0.45 * (a_hi - a_ref)
        ref = (mm & (As >= thr_a)).astype(np.uint8)
        ref = cv2.morphologyEx(cv2.morphologyEx(ref, cv2.MORPH_OPEN, k5), cv2.MORPH_CLOSE, k5)
        n3, l3, s3, _ = cv2.connectedComponentsWithStats(ref)
        if n3 > 1:
            ref = l3 == 1 + np.argmax(s3[1:, cv2.CC_STAT_AREA])
            if ref.sum() >= 0.3 * mm.sum():
                dbg['conj_raw'] = mm
                mm = ref
    return mm, dbg, 'ok'

def find_conjunctiva_auto(rgb):
    """3 порога «белизны» (тёплый свет желтит белок); берём самую уверенную радужку.
    Если первый же порог дал уверенный результат — остальные не считаем (в 2–3 раза быстрее)."""
    results = []
    for b in (6, 10, 14):
        r = _find(rgb, b)
        results.append(r)
        if r[0] is not None and r[0].sum() >= 300 and r[1]['iris_score'] > 20:
            break
    ok = [r for r in results if r[0] is not None and r[0].sum() >= 300]
    return max(ok if ok else results, key=lambda r: r[1]['iris_score'])

def _find_with_iris(rgb, iris, xlim=None):
    """Три порога белизны; берём вариант с наибольшей найденной конъюнктивой (а не первый попавшийся)."""
    res = [_find(rgb, b, iris_override=iris, xlim=xlim) for b in (6, 10, 14)]
    ok = [r for r in res if r[0] is not None and r[0].sum() >= 150]
    return max(ok, key=lambda r: r[0].sum()) if ok else res[0]

def highlight_gains(crop, iris=None):
    """Оценка цвета освещения по самым светлым НЕ пересвеченным пикселям рядом с радужкой
    (белок и блик роговицы отражают цвет источника света). Поправка фон Криса; ≈1 при нейтральном свете."""
    lum = crop.mean(axis=2)
    ok = crop.max(axis=2) < 0.985
    if iris is not None:                               # только кольцо вокруг радужки по бокам — там белок
        cx, cy, r = iris
        yy, xx = np.mgrid[0:crop.shape[0], 0:crop.shape[1]]
        dd = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
        ok &= (dd > 1.05 * r) & (dd < 2.5 * r) & (np.abs(yy - cy) < 0.9 * r)
    if ok.sum() < 100:
        return np.ones(3, np.float32)
    thr = np.percentile(lum[ok], 80 if iris is not None else 97)
    px = crop[ok & (lum >= thr)]
    ref = np.median(px, axis=0)
    # ИСПРАВЛЕНО (аудит v10): было clip(0.65, 1.55) с комментарием «до ≈2700 K», но в гамма-кодированном sRGB
    # белый при 2700 K (баланс белого D65) требует усиления синего ≈1.9 — старый предел покрывал только ≈3700 K.
    return np.clip(ref.mean() / (ref + EPS), 0.5, 2.0).astype(np.float32)

# ---------------------------------------------------------------- конъюнктива: модель, обученная на разметке врачей
# Правила (цвет + геометрия) находят область примерно. Ячейка 3 обучает по ~200 фото с ручной разметкой
# врачей (Eyes-defy-anemia) попиксельный классификатор «пальпебральная конъюнктива / не она» и проверяет его
# на пациентах, которых модель не видела. Используется ТОЛЬКО если на проверке он совпадает с врачами
# лучше правил; иначе остаются правила. CONJ_SEG заполняет ячейка 3.
CONJ_SEG = globals().get('CONJ_SEG')
SEG_STEP = 2                        # признаки считаются на вырезе, уменьшенном в 2 раза (радужка 30 px)

def _ref_colors(L, A, B, ic, white):
    """Эталоны цвета в этом же кадре: белок (рядом с радужкой) и кожа (дальше от глаза).
    Признаки ОТНОСИТЕЛЬНО них почти не зависят от освещения и камеры."""
    cx, cy, r = ic
    h, w = L.shape
    yy, xx = np.mgrid[0:h, 0:w]
    dd = np.hypot(xx - cx, yy - cy)
    side = white & (dd > 1.05 * r) & (dd < 2.2 * r) & (np.abs(yy - cy) < 0.5 * r)
    if side.sum() < 20:
        C = np.hypot(A, B)
        side = (L > np.percentile(L, 80)) & (C < np.percentile(C, 30))
        if side.sum() < 20:
            side = np.ones_like(L, bool)
    skin = dd > 3.0 * r
    if skin.sum() < 50:
        skin = np.ones_like(L, bool)
    return ([float(np.median(v[side])) for v in (L, A, B)],
            [float(np.median(v[skin])) for v in (L, A, B)])

def seg_features(crop, ic, white=None, zone=None, conj_rule=None):
    """Признаки каждого пикселя выреза глаза (радужка 60 px) для модели конъюнктивы.
    Возвращает (X, roi, (h, w)): X — строки только для пикселей ниже центра радужки (roi)."""
    k = 1.0 / SEG_STEP
    sm = cv2.resize(crop, None, fx=k, fy=k, interpolation=cv2.INTER_AREA)
    h, w = sm.shape[:2]
    cx, cy, r = ic[0] * k, ic[1] * k, max(ic[2] * k, 3.0)
    L, A, B, C = lab_of(sm)
    hue = np.degrees(np.arctan2(B, A))
    def rs(m):
        if m is None:
            return np.zeros((h, w), np.float32)
        return cv2.resize(np.asarray(m).astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST).astype(np.float32)
    wht, zn, cr = rs(white), rs(zone), rs(conj_rule)
    (Lw, Aw, Bw), (Lk, Ak, Bk) = _ref_colors(L, A, B, (cx, cy, r), wht > 0)
    s1, s2 = max(r / 8, 1.0), max(r / 3, 2.0)
    g = lambda x, s: cv2.GaussianBlur(x.astype(np.float32), (0, 0), s)
    A1, L1, B1 = g(A, s1), g(L, s1), g(B, s1)
    A2, L2, B2 = g(A, s2), g(L, s2), g(B, s2)
    std = np.sqrt(np.maximum(g(L * L, s1) - L1 ** 2, 0))
    sh = max(int(round(r / 2)), 1)
    up = lambda x: np.vstack([np.repeat(x[:1], sh, 0), x[:-sh]])        # значение на r/2 ВЫШЕ пикселя
    dn = lambda x: np.vstack([x[sh:], np.repeat(x[-1:], sh, 0)])        # значение на r/2 НИЖЕ пикселя
    yy, xx = np.mgrid[0:h, 0:w].astype(np.float32)
    dy, dx = (yy - cy) / r, np.abs(xx - cx) / r
    feats = [L, A, B, C, hue,
             A - Aw, L - Lw, B - Bw, A - Ak, L - Lk, B - Bk,
             A1 - Aw, L1 - Lw, B1 - Bw, A2 - Aw, L2 - Lw, B2 - Bw, std,
             up(A1) - Aw, up(L1) - Lw, dn(A1) - Aw, dn(L1) - Lw, up(g(wht, s1)), dn(g(wht, s1)),
             dy, dx, np.hypot(dx, dy), g(zn, s1), cr, g(wht, s1)]
    roi = (dy > -0.3) & (dy < 4.6)
    X = np.stack([f[roi] for f in feats], axis=1).astype(np.float32)
    return X, roi, (h, w)

def _seg_post(p, thr, r_s):
    """Карта вероятности -> маска: сглаживание, порог, одна связная область (конъюнктива — сплошная полоса)."""
    m = (cv2.GaussianBlur(p, (0, 0), 1.0) >= thr).astype(np.uint8)
    k3 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    m = cv2.morphologyEx(cv2.morphologyEx(m, cv2.MORPH_OPEN, k3), cv2.MORPH_CLOSE, k3)
    n, lbl, st, _ = cv2.connectedComponentsWithStats(m)
    if n <= 1:
        return None
    j = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    if st[j, cv2.CC_STAT_AREA] < 0.15 * r_s ** 2:
        return None
    return lbl == j

def seg_probability(crop, ic, white, zone, conj_rule, model):
    X, roi, (h, w) = seg_features(crop, ic, white, zone, conj_rule)
    p = np.zeros((h, w), np.float32)
    if len(X):
        p[roi] = model.predict_proba(X)[:, 1]
    return p

def conj_learned(crop, ic, white, zone, conj_rule, seg=None):
    """Маска конъюнктивы от обученной модели (в координатах выреза) или None."""
    seg = seg or CONJ_SEG
    p = seg_probability(crop, ic, white, zone, conj_rule, seg['model'])
    m = _seg_post(p, seg['thr'], ic[2] / SEG_STEP)
    if m is None:
        return None
    return cv2.resize(m.astype(np.uint8), (crop.shape[1], crop.shape[0]), interpolation=cv2.INTER_NEAREST) > 0

def combine_conj(m_rule, m_learn):
    """Маска модели принимается, только если она лежит там же, где правила нашли оттянутое веко
    (иначе это не уточнение, а другая область — тогда честно остаются правила)."""
    if m_rule is None or m_learn is None:
        return m_rule
    inter = float((m_rule & m_learn).sum())
    if inter < 0.3 * m_learn.sum():
        return m_rule
    return m_learn

def analyze_eye(rgb, iris, corners=None, gains=None, hidden=False):
    """Вырез вокруг глаза в ЕДИНОМ масштабе (радиус радужки = 60 px) и поиск конъюнктивы/склеры в нём.
    Один и тот же масштаб для крупного плана и для фото всего лица."""
    h, w = rgb.shape[:2]
    cx, cy, r = iris
    x0, x1 = int(max(0, cx - 4 * r)), int(min(w, cx + 4 * r))
    y0, y1 = int(max(0, cy - 2.5 * r)), int(min(h, cy + 4.5 * r))
    s = TARGET_R / r
    crop = cv2.resize(rgb[y0:y1, x0:x1], None, fx=s, fy=s,
                      interpolation=cv2.INTER_CUBIC if s > 1 else cv2.INTER_AREA)
    crop = np.clip(crop, 0, 1).astype(np.float32)
    # Цвет освещения: лист бумаги (если найден) надёжнее; иначе — по бликам/белку глаза.
    ic = ((cx - x0) * s, (cy - y0) * s, r * s)
    if gains is None:
        gains = highlight_gains(crop, ic)
        if np.abs(gains - 1).max() < 0.05:         # свет почти нейтральный — не трогаем
            gains = np.ones(3, np.float32)
    crop = np.clip(crop * gains, 0, 1).astype(np.float32)
    xlim = None
    if corners is not None:
        xs = sorted([(corners[0][0] - x0) * s, (corners[1][0] - x0) * s])
        xlim = (xs[0], xs[1])
    m, dbg, info = _find_with_iris(crop, ic, xlim)
    eye = {'crop': crop, 'conj': m, 'conj_rule': m, 'conj_method': 'правила (цвет и геометрия)', 'sclera': dbg['sclera'], 'white': dbg['white'], 'zone': dbg['zone'],
           'iris_crop': ic, 'iris': (cx, cy, r), 'box': (x0, y0, x1, y1), 'scale': s, 'r_px': r, 'info': info,
           'sharp': sharpness(rgb, iris), 'gains': gains, 'corners': corners}
    # Проверка: это действительно радужка? (тёмный круг, по бокам белок, внутри не белок)
    Lc = cv2.GaussianBlur(lab_of(crop)[0], (0, 0), 1.5)
    Wc = dbg['white'].astype(np.float32)
    q, con = circle_score(Lc, Wc, *ic)
    wr = float(_samp(Wc, ic[0], ic[1], [1.15 * ic[2], 1.3 * ic[2], 1.45 * ic[2]], _ANG_R).mean())
    wl = float(_samp(Wc, ic[0], ic[1], [1.15 * ic[2], 1.3 * ic[2], 1.45 * ic[2]], _ANG_L).mean())
    eye['iris_q'] = {'score': float(q), 'con': float(con), 'white_l': wl, 'white_r': wr}
    # белок должен быть С ОБЕИХ сторон радужки (белая стена/платок только с одной стороны — не белок глаза)
    eye['iris_ok'] = con >= 8 and max(wl, wr) >= 0.2 and min(wl, wr) >= 0.08 and (q >= 12 or con >= 15)
    # Вокруг настоящей радужки (1.6–4 радиуса) есть веки, ресницы, кожа — «белое» там не сплошное.
    # Если вокруг круга почти всё «белое» — это родинка/тень на коже, которая при холодном свете
    # теряет цвет и выглядит как белок. Такой круг — не глаз.
    yy_, xx_ = np.mgrid[0:crop.shape[0], 0:crop.shape[1]]
    dd_ = np.hypot(xx_ - ic[0], yy_ - ic[1])
    ann = (dd_ > 1.6 * ic[2]) & (dd_ < 4.0 * ic[2])
    vis = float(ann.sum()) / (np.pi * (16 - 2.56) * ic[2] ** 2)   # какая часть кольца попала в кадр
    if ann.sum() >= 100:
        wf = float(dbg['white'][ann].mean())
        eye['iris_q'].update(white_around=wf, ring_visible=vis)
        if wf >= 0.9 and vis >= 0.6:                   # (у крупного плана кольцо выходит за кадр — не судим)
            eye['iris_ok'] = False
    eye['iris_hidden'] = False
    if hidden:                                         # радужка закрыта веком: глаз найден по конъюнктиве
        eye['iris_hidden'] = True
        eye['iris_ok'] = m is not None and m.sum() >= 0.6 * ic[2] ** 2
    ch, cw = crop.shape[:2]
    if m is not None and m.sum() > 0:
        L, A, _, _ = lab_of(crop)
        yy, xx = np.mgrid[0:ch, 0:cw]
        dd = np.sqrt((xx - ic[0]) ** 2 + (yy - ic[1]) ** 2)
        skin = (dd > 2.5 * ic[2]) & (dd < 4 * ic[2]) & ~m & (L > 30)
        a_skin = float(np.median(A[skin])) if skin.sum() > 50 else 0.0
        eye['open_score'] = m.sum() / (ic[2] ** 2) * max(float(A[m].mean()) - a_skin, 0.5)
    else:
        eye['open_score'] = 0.0
    # Выбор глаза и проверка «веко оттянуто» — по правилам (как раньше). Модель по разметке врачей
    # только УТОЧНЯЕТ границы конъюнктивы внутри уже найденного века (если на проверке она точнее правил).
    seg = globals().get('CONJ_SEG')
    if m is not None and seg is not None and seg.get('use'):
        try:
            ml = conj_learned(crop, ic, dbg['white'], dbg['zone'], m, seg)
            mc = combine_conj(m, ml)
            if mc is not m:
                eye['conj'], eye['conj_method'] = mc, 'модель, обученная на разметке врачей'
        except Exception:
            pass
    return eye

def sharpness(rgb, iris):
    """Резкость (дисперсия Лапласиана) на ИСХОДНОМ разрешении: крупные глаза уменьшаем до радужки 20 px,
    мелкие не увеличиваем — иначе увеличение «размывает» и ложно занижает резкость."""
    cx, cy, r = iris
    H, W = rgb.shape[:2]
    x0, x1 = int(max(0, cx - 3 * r)), int(min(W, cx + 3 * r))
    y0, y1 = int(max(0, cy - 2 * r)), int(min(H, cy + 3 * r))
    if x1 - x0 < 5 or y1 - y0 < 5:
        return 0.0
    reg = rgb[y0:y1, x0:x1]
    k = min(1.0, 20 / max(r, 1))
    if k < 1:
        reg = cv2.resize(reg, None, fx=k, fy=k, interpolation=cv2.INTER_AREA)
    g = cv2.cvtColor((np.clip(reg, 0, 1) * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
    return float(cv2.Laplacian(g, cv2.CV_64F).var())

def to_full(mask_crop, eye, shape):
    """Маска из выреза -> координаты исходного фото (для показа)."""
    full = np.zeros(shape[:2], bool)
    if mask_crop is None:
        return full
    x0, y0, x1, y1 = eye['box']
    full[y0:y1, x0:x1] = cv2.resize(mask_crop.astype(np.uint8), (x1 - x0, y1 - y0),
                                    interpolation=cv2.INTER_NEAREST) > 0
    return full

def _cascades():
    """Детекторы Хаара (хранятся в модуле cv2, чтобы функции можно было запускать параллельно)."""
    if not hasattr(cv2, '_anm_face'):
        try:
            cv2._anm_face = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
            cv2._anm_eye = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_eye.xml')
            if cv2._anm_face.empty() or cv2._anm_eye.empty():
                raise RuntimeError('файлы детекторов не найдены')
        except Exception:                          # OpenCV 5: детекторов Хаара нет — работает YuNet / поиск по кадру
            cv2._anm_face = cv2._anm_eye = None
    return cv2._anm_face, cv2._anm_eye

def _best_detection(img):
    """Поиск радужки на исходных цветах; если не вышло (сильный цветной свет) — после нейтрализации цвета.
    Возвращает (результат, использованное изображение)."""
    _, d1, _ = find_conjunctiva_auto(img)
    if d1 and d1.get('iris') and d1['iris_score'] > 8:
        return d1, img
    g = highlight_gains(img)
    if np.abs(g - 1).max() < 0.05:
        return d1, img
    img2 = np.clip(img * g, 0, 1).astype(np.float32)
    _, d2, _ = find_conjunctiva_auto(img2)
    if d2 and d2.get('iris') and (not (d1 and d1.get('iris')) or d2['iris_score'] > d1['iris_score']):
        return d2, img2
    return d1, img

def _iris_in_box(rgb, box):
    """Уточняем радужку внутри рамки глаза (вырез приводится к ширине 400 px)."""
    H, W = rgb.shape[:2]
    x, y, w, h = box
    x0, x1 = max(0, int(x - 0.3 * w)), min(W, int(x + 1.3 * w))
    y0, y1 = max(0, int(y - 0.2 * h)), min(H, int(y + 1.5 * h))
    if x1 - x0 < 8 or y1 - y0 < 8:
        return None
    s = 400 / (x1 - x0)
    crop = cv2.resize(rgb[y0:y1, x0:x1], None, fx=s, fy=s, interpolation=cv2.INTER_CUBIC if s > 1 else cv2.INTER_AREA)
    crop = np.clip(crop, 0, 1).astype(np.float32)
    d, _ = _best_detection(crop)
    if not d or not d.get('iris'):
        return None
    cx, cy, r = d['iris']
    return {'iris': (x0 + cx / s, y0 + cy / s, r / s),
            'corners': (np.array([x, y + h / 2]), np.array([x + w, y + h / 2]))}

def _get_yunet():
    """Нейросетевой детектор лица YuNet (входит в OpenCV 4.8+ и 5.x): находит лицо и центры глаз при любом
    масштабе и наклоне головы. Файл модели (230 КБ) скачивается один раз; копия хранится рядом с моделями."""
    if not hasattr(cv2, '_anm_yunet'):
        cv2._anm_yunet = False
        name = 'face_detection_yunet_2023mar.onnx'
        keep = os.path.join(globals().get('DRIVE_DIR', '.'), name)
        path = keep if os.path.exists(keep) else name
        urls = ['https://media.githubusercontent.com/media/opencv/opencv_zoo/main/models/face_detection_yunet/' + name,
                'https://github.com/opencv/opencv_zoo/raw/main/models/face_detection_yunet/' + name,
                'https://huggingface.co/opencv/face_detection_yunet/resolve/main/' + name]
        if not (os.path.exists(path) and os.path.getsize(path) > 100_000):
            import urllib.request
            for u in urls:
                try:
                    urllib.request.urlretrieve(u, name)
                    if os.path.getsize(name) > 100_000:      # настоящий файл модели, а не страница ошибки
                        path = name
                        break
                except Exception:
                    pass
        try:
            if os.path.getsize(path) > 100_000:
                cv2._anm_yunet = cv2.FaceDetectorYN.create(path, '', (320, 320), 0.6, 0.3, 5000)
                if path != keep and os.path.isdir(os.path.dirname(keep) or '.'):
                    try:
                        shutil.copy(path, keep)
                    except Exception:
                        pass
        except Exception as e:
            print('ℹ️ Детектор лица YuNet недоступен:', str(e)[:100])
            cv2._anm_yunet = False
        if cv2._anm_yunet is False:
            print('ℹ️ Детектор лица YuNet не загрузился — фото всего лица распознаются хуже; снимайте глаз крупно.')
    return cv2._anm_yunet or None

def _yunet_eye_boxes(small, s):
    det = _get_yunet()
    if det is None:
        return []
    bgr = np.ascontiguousarray(cv2.cvtColor((small * 255).astype(np.uint8), cv2.COLOR_RGB2BGR))
    det.setInputSize((bgr.shape[1], bgr.shape[0]))
    _, faces = det.detect(bgr)
    if faces is None or len(faces) == 0:
        return []
    f = max(faces, key=lambda r: r[2] * r[3])
    eyes = [(f[4], f[5]), (f[6], f[7])]                 # центры правого и левого глаза
    d = max(np.hypot(eyes[0][0] - eyes[1][0], eyes[0][1] - eyes[1][1]), 4)
    return [((ex - 0.3 * d) / s, (ey - 0.22 * d) / s, 0.6 * d / s, 0.44 * d / s) for ex, ey in eyes]

def _closeup_candidates(rgb, variants=(0,)):
    """Крупный план одного глаза: поиск радужки по всему кадру. variants: 0 — исходные цвета,
    1 — нейтрализованные (считается, только если исходный вариант не дал настоящего глаза)."""
    H = rgb.shape[0]
    sm = resize_max(rgb, 800); k = sm.shape[0] / H
    g = highlight_gains(sm)
    imgs = [sm] if np.abs(g - 1).max() < 0.05 else [sm, np.clip(sm * g, 0, 1).astype(np.float32)]
    out = []
    for vi, img in enumerate(imgs):                    # проверку «это радужка?» делает analyze_eye
        if vi not in variants:
            continue
        _, d, _ = find_conjunctiva_auto(img)
        # если «белым» оказалась большая часть кадра (стена, фон) — это не крупный план глаза
        if d and d.get('iris') and sclera_like(img).mean() < 0.5 and d['iris'][2] < 0.3 * min(sm.shape[:2]):
            cx, cy, r = d['iris']
            out.append({'iris': (cx / k, cy / k, r / k), 'corners': None})
    return out

def detect_eyes(rgb, use_face=True):
    """Кандидаты глаз на фото любого масштаба:
    1) YuNet (лицо + центры глаз); 2) детекторы Хаара (лицо -> глаза, + геометрия лица);
    3) глаза по кадру; 4) крупный план одного глаза (с проверкой правдоподобия)."""
    H, W = rgb.shape[:2]
    small = resize_max(rgb, 1000)
    s = small.shape[0] / H
    boxes, source = [], ''
    if not use_face:                                  # режим «крупный план» (фото датасета) — сразу к глазу
        return _closeup_candidates(rgb), 'крупный план'
    try:
        boxes = _yunet_eye_boxes(small, s)
        source = 'лицо (YuNet)' if boxes else ''
    except Exception:
        boxes = []
    if not boxes:
        g = cv2.equalizeHist(cv2.cvtColor((small * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY))
        m = min(g.shape)
        _FACE, _EYE = _cascades()
        faces = _FACE.detectMultiScale(g, 1.1, 3, minSize=(int(0.1 * m),) * 2) if _FACE is not None else []
        if _FACE is None:
            source = 'крупный план'
        elif len(faces):
            fx, fy, fw, fh = [v / s for v in max(faces, key=lambda f: f[2] * f[3])]
            fx0, fy0 = int(fx), int(fy)
            face = rgb[fy0:int(fy + 0.65 * fh), fx0:int(fx + fw)]
            k = 500 / max(face.shape[1], 1)
            fg = cv2.equalizeHist(cv2.cvtColor((cv2.resize(face, None, fx=k, fy=k) * 255).astype(np.uint8),
                                               cv2.COLOR_RGB2GRAY))
            es = _EYE.detectMultiScale(fg, 1.1, 4, minSize=(60, 60), maxSize=(225, 225))
            boxes = [(fx0 + x / k, fy0 + y / k, w / k, h / k) for x, y, w, h in es]
            boxes += [(fx + 0.13 * fw, fy + 0.22 * fh, 0.32 * fw, 0.25 * fh),   # геометрия лица (запас)
                      (fx + 0.55 * fw, fy + 0.22 * fh, 0.32 * fw, 0.25 * fh)]
            source = 'лицо (Хаар)'
        else:
            es = _EYE.detectMultiScale(g, 1.1, 4, minSize=(int(0.15 * m),) * 2)
            boxes = [tuple(v / s for v in e) for e in es]
            source = 'глаз'
    cands = []
    for b in sorted(boxes, key=lambda b: -b[2] * b[3])[:6]:
        c = _iris_in_box(rgb, b)
        if c:
            cands.append(c)
    if not cands:                                     # крупный план одного глаза
        cands = _closeup_candidates(rgb)
        source = 'крупный план'
    uniq = []
    for c in cands:
        if all(np.hypot(c['iris'][0] - u['iris'][0], c['iris'][1] - u['iris'][1]) > u['iris'][2] for u in uniq):
            uniq.append(c)
    return uniq, source

def find_by_conjunctiva(rgb):
    """Запасной поиск, когда радужку не видно (человек смотрит вверх, веко её закрывает):
    ищем красно-розовую влажную ткань (оттянутое веко), над которой лежит белок глаза.
    Возвращает кандидатов с «условной радужкой» для масштаба."""
    H = rgb.shape[0]
    sm = resize_max(rgb, 900); k = sm.shape[0] / H
    L, A, B, C = lab_of(sm)
    hue = np.degrees(np.arctan2(B, A))
    skin = (L > 30) & (C > 8)
    if skin.sum() < 500:
        return []
    a_s, h_s = float(np.median(A[skin])), float(np.median(hue[skin]))
    red = (A > a_s + 8) & (hue < h_s - 6) & (L > 20) & (L < 88)
    k5 = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
    red = cv2.morphologyEx(cv2.morphologyEx(red.astype(np.uint8), cv2.MORPH_OPEN, k5), cv2.MORPH_CLOSE, k5)
    L_s, C_s = float(np.median(L[skin])), float(np.median(C[skin]))
    # белок у пожилых/в тени бывает серым — главный признак: он гораздо МЕНЕЕ цветной, чем кожа
    whitish = (C < 0.6 * C_s) & (L > 0.55 * L_s)
    dark = L < 0.6 * L_s                                   # ресницы / край века / радужка
    n, lbl, st, _ = cv2.connectedComponentsWithStats(red)
    out = []
    for j in range(1, n):
        x, y, w, h, area = st[j]
        if area < 0.002 * red.size or w < 1.3 * h:          # конъюнктива — вытянутый «полумесяц»
            continue
        y_up = max(0, y - h)
        above = whitish[y_up:y, x:x + w]                     # полоса строго НАД тканью
        wf = float(above.mean()) if above.size else 0.0
        if wf < 0.15:                                       # над ней должен быть белок глаза
            continue
        lid = dark[max(0, y - 3 * h):y_up + 1, x:x + w]      # ещё выше — тёмный край века/ресницы
        if lid.size == 0 or lid.mean() < 0.05:             # (у губ над «белым» зубов этого нет)
            continue
        r = w / 5.0
        out.append((area * wf, {'iris': ((x + w / 2) / k, (y - 1.0 * r) / k, r / k),
                                'corners': (np.array([x / k, (y + h / 2) / k]), np.array([(x + w) / k, (y + h / 2) / k]))}))
    out.sort(key=lambda t: -t[0])
    return [c for _, c in out[:3]]

def _research_in_crop(rgb, eye):
    """Радужка-кандидат не прошла проверку — ищем настоящую радужку заново внутри выреза вокруг глаза."""
    d, _ = _best_detection(eye['crop'])
    if not d or not d.get('iris'):
        return None
    cx, cy, r = d['iris']
    x0, y0 = eye['box'][:2]
    s = eye['scale']
    return (x0 + cx / s, y0 + cy / s, r / s)

def _conj_big(e):
    """Открыта заметная полоска века, а не пара пикселей (по правилам — выбор глаза от модели не зависит)."""
    m = None if e is None else e.get('conj_rule', e['conj'])
    return m is not None and m.sum() >= 0.6 * e['iris_crop'][2] ** 2

def normalize_exposure(rgb):
    """Тёмный снимок (вечер, тень): линейно поднимаем яркость, чтобы самые светлые участки стали ≈0.9.
    Линейное усиление не меняет соотношений цветов (EI, оттенок), но позволяет найти белок и радужку."""
    hi = float(np.percentile(rgb.max(axis=2), 99))
    if hi >= 0.6 or hi <= 0.02:
        return rgb, 1.0
    g = min(0.9 / hi, 4.0)
    return np.clip(rgb * g, 0, 1).astype(np.float32), g

def best_eye(rgb, gains=None, use_face=True, require_conj=True):
    """Анализирует кандидатов, ПРОВЕРЯЕТ, что найдена именно радужка (тёмный круг с белком по бокам),
    при ошибке ищет заново; из настоящих глаз выбирает тот, где оттянуто веко."""
    rgb, exp_gain = normalize_exposure(rgb)
    e, source, n = _best_eye(rgb, gains, use_face, require_conj)
    wb = None
    if not _conj_big(e) and gains is None and use_face:   # (фото датасета сняты при ровном свете прибора)
        # Глаз не найден ИЛИ найден без оттянутого века. При сильном цветном свете (лампа накаливания,
        # холодный LED) белок перестаёт быть «белым», а бледное веко сливается с белком — ищем ещё раз
        # после выравнивания цвета по коже (её оттенок у людей почти одинаков) и берём лучший вариант.
        g = skin_balance_gains(rgb)
        if g is not None:
            e2, s2, n2 = _best_eye(np.clip(rgb * g, 0, 1).astype(np.float32), None, use_face, require_conj)
            if e2 is not None and (e is None or (_conj_big(e2) and not _conj_big(e))):
                e, source, n, wb = e2, s2, n2, g
    if e is not None:
        e['exposure_gain'] = exp_gain
        e['wb_fallback'] = wb
    return e, source, n

def skin_balance_gains(rgb):
    """Поправка цвета освещения по коже: средние доли G/R и B/R кожи при дневном свете ≈ 0.80 и 0.68.
    Возвращает усиления каналов или None, если цвет и так нормальный.
    ИСПРАВЛЕНО (аудит v10): раньше бралась медиана ВСЕХ пикселей кадра средней яркости (фон, одежда, белок),
    а не кожи. Теперь — только кожеподобные пиксели (тёплый оттенок, умеренная хрома). Ограничение остаётся:
    доли 0.80/0.68 типичны для светлой кожи; для тёмной кожи поправка неточна — это запасной режим."""
    sm = resize_max(rgb, 400)
    lum = sm.mean(axis=2)
    L, A, B, C = lab_of(sm)
    hue = np.degrees(np.arctan2(B, A))
    ok = (lum > 0.2) & (lum < 0.9) & (C > 6) & (C < 45) & (hue > 15) & (hue < 85) & ~sclera_like(sm, 14)
    if ok.sum() < 500:
        return None
    med = np.median(sm[ok], axis=0) + EPS
    tgt = np.array([1.0, 0.80, 0.68]) * med[0]
    gn = tgt / med
    gn = gn / gn.mean()
    if np.abs(gn - 1).max() < 0.08:
        return None
    return np.clip(gn, 0.5, 2.0).astype(np.float32)

def _best_eye(rgb, gains=None, use_face=True, require_conj=True):
    cands, source = detect_eyes(rgb, use_face)
    eyes, tried = [], []
    def _add(iris, corners):
        if any(np.hypot(iris[0] - t[0], iris[1] - t[1]) < 0.5 * t[2] for t in tried):
            return None
        tried.append(iris)
        e = analyze_eye(rgb, iris, corners, gains)
        return e
    for c in cands:
        e = _add(c['iris'], c['corners'])
        if e is None:
            continue
        if not e['iris_ok']:                              # не радужка — ищем заново в этом же месте
            iris2 = _research_in_crop(rgb, e)
            e2 = _add(iris2, c['corners']) if iris2 else None
            e = e2 if e2 is not None else e
        eyes.append(e)
    if not any(e['iris_ok'] and _conj_big(e) for e in eyes):
        # последняя попытка: поиск по всему кадру (для крупного плана — с нейтрализованными цветами)
        for c in _closeup_candidates(rgb, (1,) if source == 'крупный план' else (0, 1)):
            e = _add(c['iris'], None)
            if e is not None:
                if not e['iris_ok']:
                    iris2 = _research_in_crop(rgb, e)
                    e2 = _add(iris2, None) if iris2 else None
                    e = e2 if e2 is not None else e
                eyes.append(e)
    good = [e for e in eyes if e['iris_ok'] and _conj_big(e)]
    best = max(good, key=lambda e: e['open_score']) if good else None
    # Радужку не видно или найден «глаз» с почти невидимой конъюнктивой (часто это ложный круг в углу глаза,
    # когда человек смотрит вверх) — ищем глаз по самой конъюнктиве и берём более убедительный вариант
    if best is None or best['open_score'] < 8:
        for c in find_by_conjunctiva(rgb):
            e = analyze_eye(rgb, c['iris'], c['corners'], gains, hidden=True)
            if e['iris_ok'] and _conj_big(e) and (best is None or e['open_score'] > best['open_score']):
                best, source = e, 'по конъюнктиве (радужка закрыта веком)'
    if best is not None:
        return best, source, len(cands)
    iris_only = [e for e in eyes if e['iris_ok']]
    if iris_only and not require_conj:                    # веко не оттянуто — глаз годен для склеры
        return max(iris_only, key=lambda e: int(e['sclera'].sum()) if e['sclera'] is not None else 0), source, len(cands)
    return None, source, len(cands)                       # лучше попросить переснять, чем ошибиться


def quality_check_sclera(eye):
    """Пригодность снимка для оценки ТОЛЬКО по склере."""
    problems = []
    if eye['r_px'] < 7:
        problems.append('глаз слишком мелкий на фото — снимите крупнее')
    if eye.get('sharp', 0) < 6:
        problems.append('снимок сильно размыт')
    sc = eye.get('sclera')
    rc = eye['iris_crop'][2]
    if sc is None or sc.sum() < 0.5 * rc ** 2:
        problems.append('белок глаза виден слишком мало — откройте глаз шире, взгляд чуть вверх')
    return problems

def conj_crop_from_file(path):
    """Фото — это уже ВЫРЕЗАННАЯ конъюнктива (прозрачный или чёрный фон, как в датасетах)?
    Возвращает (rgb, маска, описание) или None."""
    im = cv2.imread(path, cv2.IMREAD_UNCHANGED)
    if im is None or im.ndim != 3:
        return None
    if im.shape[2] == 4 and (im[..., 3] < 250).mean() > 0.05 and (im[..., 3] > 0).mean() > 0.02:
        rgb, mask = load_segment(path)
        return rgb, mask, 'вырезанная конъюнктива (прозрачный фон)'
    bgr = im[..., :3]
    black = bgr.max(axis=2) < 15
    if 0.15 < black.mean() < 0.95:
        rgb, mask = load_segment(path)
        _, A, _, _ = lab_of(rgb)
        if mask.sum() > 300 and np.median(A[mask]) > 5:
            return rgb, mask, 'вырезанная конъюнктива (чёрный фон)'
    return None

def conj_tissue_crop(rgb):
    """Глаз на фото не найден: проверяем, не крупный ли это кадр одной конъюнктивы без фона
    (почти весь кадр — влажная розово-красная ткань). Кожа сюда не проходит: она желтее (больше b*)."""
    sm = resize_max(rgb, 800)
    L, A, B, C = lab_of(sm)
    hue = np.degrees(np.arctan2(B, A))
    tissue = (A > 8) & (hue < 38) & (L > 20) & (L < 92)
    if tissue.mean() < 0.6:
        return None
    full = np.ones(tissue.shape, bool)
    return sm, full, 'крупный кадр конъюнктивы'

# ---------------------------------------------------------------- склера
def sclera_features(rgb, sclera, iris):
    """Признаки склеры: цвет (голубизна/желтизна/краснота) + сосуды. None — склера слишком мала."""
    if sclera is None or iris is None or sclera.sum() < 150:
        return None, None, None
    L, A, B, C = lab_of(rgb)
    cx, cy, r = iris
    h, w = sclera.shape
    yy, xx = np.mgrid[0:h, 0:w]
    dd = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    skin = (dd > 2.2 * r) & (dd < 3.5 * r) & (C > 8) & (L > 30) & ~sclera
    inner = cv2.erode(sclera.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    if inner.sum() < 100:
        return None, None, None
    G8 = (np.clip(rgb[..., 1], 0, 1) * 255).astype(np.uint8)
    k = max(3, int(r / 5)) | 1
    bh = cv2.morphologyEx(G8, cv2.MORPH_BLACKHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))).astype(np.float32)
    g_bg = float(np.median(G8[inner])) + EPS
    vessel = inner & (bh > 0.06 * g_bg)            # сосуды: тонкие и темнее фона в зелёном канале
    bg = inner & ~vessel
    if bg.sum() < 100:
        return None, None, None
    nv = int(vessel.sum())
    f = {'scl_L': float(L[bg].mean()), 'scl_a': float(A[bg].mean()), 'scl_b': float(B[bg].mean()),
         'scl_blue_rel': float(B[bg].mean() - np.median(B[skin])) if skin.sum() > 100 else 0.0,
         'scl_blue_ratio': float((rgb[..., 2] / (rgb.sum(2) + EPS))[bg].mean()),
         'vessel_density': float(nv / inner.sum()),
         'vessel_contrast': float(np.median(bh[vessel]) / g_bg) if nv > 20 else 0.0,
         'vessel_redness': float(A[vessel].mean() - A[bg].mean()) if nv > 20 else 0.0}
    return f, vessel, bg

# ---------------------------------------------------------------- устойчивость к освещению и камере
def conj_rel_features(rgb, conj, sclera):
    """Цвет конъюнктивы ОТНОСИТЕЛЬНО белка того же глаза на том же снимке. Свет и камера меняют оба
    одинаково, поэтому разница/отношение почти не зависят от лампы, экспозиции и обработки телефона.
    None — склеры для сравнения слишком мало."""
    if conj is None or sclera is None or sclera.sum() < 150:
        return None
    vc = valid_tissue(rgb, conj)
    vs = (cv2.erode(sclera.astype(np.uint8), np.ones((3, 3), np.uint8)) > 0) & (rgb.max(axis=2) < 0.98)
    if vc.sum() < 200 or vs.sum() < 100:
        return None
    L, A, B, _ = lab_of(rgb)
    EI = erythema_map(rgb)
    rc, rs = np.median(rgb[vc], axis=0) + EPS, np.median(rgb[vs], axis=0) + EPS
    return {'rel_dA': float(np.median(A[vc]) - np.median(A[vs])),
            'rel_dB': float(np.median(B[vc]) - np.median(B[vs])),
            'rel_L': float(np.median(L[vc]) / max(np.median(L[vs]), 1.0)),
            'rel_dEI': float(np.median(EI[vc]) - np.median(EI[vs])),
            'rel_R': float(rc[0] / rs[0]), 'rel_G': float(rc[1] / rs[1]),
            'rel_RG': float(np.log(rc[0] / rc[1]) - np.log(rs[0] / rs[1]))}

def eye_conj_features(crop, conj, sclera):
    """Все признаки конъюнктивы на фото глаза: абсолютные индексы + относительно белка."""
    f = compute_indices(crop, conj)
    if f is None:
        return None
    rel = conj_rel_features(crop, conj, sclera)
    if rel:
        f.update(rel)
    return f

def crop_regions(crop, ic, seg=None):
    """Конъюнктива и склера на уже выровненном вырезе — ровно как в analyze_eye (правила + модель)."""
    m, dbg, _ = _find_with_iris(crop, ic)
    mc = m
    if m is not None and seg is not None and seg.get('use'):
        try:
            mc = combine_conj(m, conj_learned(crop, ic, dbg['white'], dbg['zone'], m, seg))
        except Exception:
            mc = m
    return {'conj': mc, 'conj_rule': m, 'sclera': dbg['sclera'], 'white': dbg['white'], 'zone': dbg['zone']}

# ИСПРАВЛЕНО (аудит v10): свет моделируется в ЛИНЕЙНОМ RGB планковским источником (баланс белого камеры на D65).
# Было: множители exp(±0.28/0.32) к гамма-кодированному sRGB. Это ≈ возведение множителя в степень 2.2 в линейном
# пространстве: тёплый край ≈3500 K (а не 2700 K), холодный — далеко за 7500 K (синее любого реального света).
_XYZ2LIN = np.array([[3.2404542, -1.5371385, -0.4985314],
                     [-0.9692660, 1.8760108, 0.0415560],
                     [0.0556434, -0.2040259, 1.0572252]])

def srgb_to_lin(x):
    x = np.clip(x, 0, 1)
    return np.where(x <= 0.04045, x / 12.92, ((x + 0.055) / 1.055) ** 2.4)

def lin_to_srgb(x):
    x = np.clip(x, 0, 1)
    return np.where(x <= 0.0031308, 12.92 * x, 1.055 * x ** (1 / 2.4) - 0.055)

def _planck_xy(T):
    """Цветность планковского излучателя (Kim et al. 2002), 1667–25000 K."""
    if T <= 4000:
        x = -0.2661239e9 / T ** 3 - 0.2343589e6 / T ** 2 + 0.8776956e3 / T + 0.179910
    else:
        x = -3.0258469e9 / T ** 3 + 2.1070379e6 / T ** 2 + 0.2226347e3 / T + 0.240390
    if T <= 2222:
        y = -1.1063814 * x ** 3 - 1.34811020 * x ** 2 + 2.18555832 * x - 0.20219683
    elif T <= 4000:
        y = -0.9549476 * x ** 3 - 1.37418593 * x ** 2 + 2.09137015 * x - 0.16748867
    else:
        y = 3.0817580 * x ** 3 - 5.87338670 * x ** 2 + 3.75112997 * x - 0.37001483
    return x, y

def kelvin_gains(T, ref=6500):
    """Множители линейных R, G, B для источника T относительно ref (G = 1)."""
    def white(t):
        x, y = _planck_xy(t)
        v = _XYZ2LIN @ np.array([x / y, 1.0, (1 - x - y) / y])
        return v / v[1]
    return (white(T) / white(ref)).astype(np.float32)

def light_perturb(rgb, rng, strength=1.0):
    """Тот же снимок «при другом свете и другой камерой»: лампа 2700–7500 K (равномерно по 1/T), лёгкий
    зелёно-пурпурный оттенок, экспозиция ×0.6–1.6 — всё в линейном RGB; затем тональная кривая телефона
    (гамма), насыщенность, шум."""
    inv = rng.uniform(1 / 7500, 1 / 2700)
    T = 1 / (1 / 6500 + strength * (inv - 1 / 6500))
    gains = kelvin_gains(T) * np.array([1.0, np.exp(rng.uniform(-0.04, 0.04) * strength), 1.0], np.float32)
    k = float(np.exp(rng.uniform(np.log(0.6), np.log(1.6)) * strength))
    x = lin_to_srgb(srgb_to_lin(rgb) * gains * k)
    gamma = float(np.exp(rng.uniform(-0.15, 0.15) * strength))
    sat = float(1 + rng.uniform(-0.2, 0.2) * strength)
    gray = x.mean(axis=2, keepdims=True)
    x = np.clip(gray + sat * (x - gray), 0, 1) ** gamma
    x = x + rng.normal(0, 0.006 * strength, x.shape)
    return np.clip(x, 0, 1).astype(np.float32)

def renormalize_crop(crop, ic):
    """То же выравнивание, что делает приложение с каждым снимком: экспозиция + цвет света по белку."""
    x, _ = normalize_exposure(crop)
    g = highlight_gains(x, ic)
    if np.abs(g - 1).max() < 0.05:
        return x
    return np.clip(x * g, 0, 1).astype(np.float32)

# ---------------------------------------------------------------- лист белой бумаги (калибровка цвета)
def white_reference_gains(rgb, exclude_box=None):
    """Ищет в кадре лист белой бумаги: крупный, очень светлый, почти бесцветный и ПРЯМОУГОЛЬНЫЙ участок
    (бледная кожа сюда не проходит). Возвращает поправку фон Криса или None."""
    small = resize_max(rgb, 600); s = small.shape[0] / rgb.shape[0]
    hsv = cv2.cvtColor(small.astype(np.float32), cv2.COLOR_RGB2HSV)
    cand = ((hsv[..., 1] < 0.08) & (hsv[..., 2] > 0.70) & (hsv[..., 2] < 0.99)).astype(np.uint8)
    if exclude_box is not None:
        x0, y0, x1, y1 = [int(v * s) for v in exclude_box]
        cand[max(0, y0):y1, max(0, x0):x1] = 0
    cand = cv2.morphologyEx(cand, cv2.MORPH_OPEN, np.ones((7, 7), np.uint8))
    n, lbl, st, _ = cv2.connectedComponentsWithStats(cand)
    for j in np.argsort(-st[1:, cv2.CC_STAT_AREA])[:3] + 1:
        area, bw, bh = st[j, cv2.CC_STAT_AREA], st[j, cv2.CC_STAT_WIDTH], st[j, cv2.CC_STAT_HEIGHT]
        if area < 0.04 * cand.size:
            break
        cnt = cv2.findContours((lbl == j).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)[0][0]
        rect_fill = area / max(cv2.contourArea(cv2.boxPoints(cv2.minAreaRect(cnt))), 1)
        if rect_fill > 0.85:                              # лист бумаги — почти прямоугольник
            ref = np.median(small[lbl == j], axis=0)
            return np.clip(ref.mean() / (ref + EPS), 0.7, 1.4).astype(np.float32)
    return None

# ---------------------------------------------------------------- проверка качества снимка
def quality_check(eye):
    """(критические проблемы, предупреждения, резкость). Переснять просим ТОЛЬКО если анализ
    действительно невозможен; всё остальное — предупреждение и более широкий интервал погрешности."""
    problems, warns = [], []
    r, sharp = eye['r_px'], eye.get('sharp', 0.0)
    crop, conj = eye['crop'], eye['conj']
    if conj is None:
        return ['конъюнктива не видна — оттяните нижнее веко сильнее'], warns, sharp
    if r < 7:
        problems.append(f'глаз слишком мелкий на фото (радужка {r:.0f} px) — снимите крупнее')
    elif r < 15:
        warns.append('глаз мелкий на фото — точность ниже')
    if sharp < 6:
        problems.append('снимок сильно размыт — держите телефон неподвижно')
    elif sharp < 25:
        warns.append('снимок нерезкий — точность ниже')
    rc = eye['iris_crop'][2]
    area_rel = conj.sum() / (rc ** 2)
    if area_rel < 0.3:
        problems.append('видна слишком маленькая полоска века — оттяните нижнее веко сильнее')
    elif area_rel < 0.8:
        warns.append('веко оттянуто слабо — точность ниже')
    v_mean = float(crop.max(axis=2)[conj].mean())
    if v_mean < 0.08:
        problems.append('слишком темно — снимайте при хорошем освещении')
    elif v_mean < 0.18:
        warns.append('темновато — точность ниже')
    clipped = float(np.mean(crop[conj].max(axis=1) > 0.98))
    if clipped > 0.5:
        problems.append('пересвет — уберите вспышку/прямое солнце')
    elif clipped > 0.2:
        warns.append('частичный пересвет — точность ниже')
    valid = valid_tissue(crop, conj)
    if valid.sum() < 200:
        problems.append('на конъюнктиве почти нет чистой ткани (блики/тени) — смените угол к свету')
    elif valid.sum() / max(conj.sum(), 1) < 0.35:
        warns.append('много бликов — точность ниже')
    return problems, warns, sharp

# ---------------------------------------------------------------- медицинские нормы (ВОЗ)
def who_cutoff(sex, age, pregnant):
    """Порог анемии по ВОЗ 2011, г/дл (возраст в годах). Для детей младше 6 мес. порога ВОЗ нет —
    используется порог 6–59 мес. (ориентировочно). ВОЗ 2024 уточнила пороги (6–23 мес. — 10.5 г/дл,
    беременные во II триместре — 10.5 г/дл, поправки на высоту и курение); здесь оставлены нормы 2011,
    по которым размечены обучающие датасеты."""
    if pregnant:
        return 11.0
    if age < 5:
        return 11.0
    if age < 12:
        return 11.5
    if age < 15:
        return 12.0
    return 13.0 if sex == 'M' else 12.0

def who_grade(hb, sex, age, pregnant):
    cut = who_cutoff(sex, age, pregnant)
    if hb >= cut:
        return 'Норма', '#2e7d32'
    mild = 10.0 if (pregnant or age < 5) else 11.0
    mod = 7.0 if (pregnant or age < 5) else 8.0
    if hb >= mild:
        return 'Лёгкая анемия', '#f9a825'
    if hb >= mod:
        return 'Умеренная анемия', '#ef6c00'
    return 'Тяжёлая анемия', '#c62828'

PRIORS = {'woman_15_49': 0.30, 'pregnant': 0.37, 'child_6_59m': 0.40, 'other': 0.10}  # ВОЗ 2019; 'other' — допущение
SIGN_LR = {   # LR+ при осмотре (Hb < 11 г/дл): McGee, Evidence-Based Physical Diagnosis
    'palmar_crease_pallor': (7.9, 'Бледные складки ладони'),
    'palmar_pallor': (5.6, 'Бледные ладони'),
}

def interpret_labs(sex, age, pregnant, Hb, MCV=None, RBC=None, RDW=None, ferritin=None, CRP=None, B12=None, folate=None):
    """Hb г/дл, MCV фл, RBC ×10¹²/л, RDW %, ферритин мкг/л, CRP мг/л, B12 пг/мл, фолат нг/мл."""
    out = []
    grade, _ = who_grade(Hb, sex, age, pregnant)
    out.append(f'Hb {Hb * 10:.0f} г/л ({Hb:.1f} г/дл), порог ВОЗ {who_cutoff(sex, age, pregnant) * 10:.0f} г/л → {grade}')
    inflamed = CRP is not None and CRP > 5
    if ferritin is not None:
        cut = (30 if age < 5 else 70) if inflamed else (12 if age < 5 else 15)
        out.append(f'Ферритин {ferritin} мкг/л ' + ('< ' if ferritin < cut else '≥ ') + f'{cut} → '
                   + ('запасы железа истощены' if ferritin < cut else 'дефицит железа по ВОЗ не подтверждается'))
    if MCV is not None and grade != 'Норма':
        if MCV < 80:
            out.append(f'MCV {MCV} фл < 80 → микроцитарная анемия (часто дефицит железа)')
            if RBC:
                mi = MCV / RBC
                out.append(f'Индекс Ментцера {mi:.1f}: ' + ('> 13 → скорее дефицит железа' if mi > 13
                                                          else '< 13 → возможно носительство талассемии, обсудите с врачом'))
            if RDW is not None and RDW > 14.5:
                out.append(f'RDW {RDW}% > 14.5 → типично для дефицита железа')
        elif MCV > 100:
            out.append(f'MCV {MCV} фл > 100 → макроцитарная анемия (частые причины — дефицит B12/фолата)')
            if B12 is not None and B12 < 200:
                out.append(f'B12 {B12} пг/мл — ниже типичного референса')
            if folate is not None and folate < 3:
                out.append(f'Фолат {folate} нг/мл — ниже типичного референса')
        else:
            out.append(f'MCV {MCV} фл — нормоцитарная анемия: нужна оценка врача')
    return grade, out

print('✅ Ячейка 2: функции готовы (версия 5.0-fix: исправления аудита — свет в линейном RGB, баланс по коже)')


In [ ]:
# ============================================================
# ЯЧЕЙКА 3 — ДАТАСЕТЫ И МОДЕЛИ (версия 10: устойчивость к свету и камере)
# Первый запуск (~10 мин): исследовательские фото копируются на Google Диск, обучаются модели.
# Дальше: модели загружаются с Диска за секунды.
# Что нового для ТОЧНОСТИ:
#   1) модель конъюнктивы учится на областях, найденных ТЕМ ЖЕ алгоритмом, что и на вашем фото
#      (раньше — на ручной разметке врачей: обучение и применение не совпадали);
#   2) снятие «сжатия к среднему»: тяжёлые случаи больше не превращаются в «лёгкие», а нормальные — в «лёгкую анемию»;
#   3) границы степеней анемии и вероятность подбираются по данным (честная перекрёстная проверка);
#   4) веса редких значений Hb (тяжёлая/умеренная анемия встречается реже — модель её «не замечала»);
#   5) ваши собственные фото с известным Hb (ячейка 7) добавляются в обучение с повышенным весом;
#   6) v9: область конъюнктивы ищет модель, обученная на ручной разметке врачей (проверка на пациентах,
#      которых она не видела); если она НЕ точнее правил — честно остаются правила;
#   7) v10: каждый обучающий снимок «пересъёмется» при 4 случайных светах/камерах (лампа 2700–7500 K,
#      экспозиция, тональная кривая телефона); модели учатся на всех вариантах, а набор признаков
#      выбирается по ошибке на ДРУГИХ светах у пациентов, которых модель не видела (стресс-тест).
# ИСПРАВЛЕНО ПО АУДИТУ (v10fix, список ошибок — v10_audit.md):
#   A) все выборы (порог HHR, набор признаков, Ridge/бустинг, калибровка, порог сегментации) делаются
#      ВНУТРИ обучающих фолдов; цифры точности — вложенная перекрёстная проверка (раньше выбор делался на тех же
#      пациентах, на которых считалась «точность на невиденных» → завышение);
#   B) пациенты, у которых автопоиск промахнулся мимо разметки врача, исключаются только из ОБУЧЕНИЯ,
#      а не из проверки (в приложении разметки врача нет — промахи должны входить в ошибку);
#   C) несколько фото одного человека (ячейка 7) — одна группа в GroupKFold (раньше каждое фото — отдельный «пациент»);
#   D) IoU сегментации считается по всем фото с разметкой (ненайденная конъюнктива = IoU 0), с долей найденных;
#   E) бустинг без sample_weight (sklearn ≥ 1.9 с весами в ~30 раз медленнее) — веса заменены оверсэмплингом.
# ============================================================
import joblib, glob, re, zipfile, shutil, sklearn, json
from joblib import Parallel, delayed
from scipy.stats import norm
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LogisticRegression, LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor, HistGradientBoostingClassifier
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score

VERSION = 'v10fix'  # v10 + исправления аудита (честная вложенная проверка, свет в линейном RGB)
CONJ_SEG = None  # пока модели не готовы — поиск конъюнктивы по правилам
DS_DIR = f'{DRIVE_DIR}/datasets'
LOCAL_DIR = f'{DRIVE_DIR}/local_data'          # ваши фото с известным Hb (ячейка 7)
STATE_FILE = f'{DRIVE_DIR}/anemia_models_{VERSION}.joblib'
F_SEG = f'{DRIVE_DIR}/features_eda_conj_v6.csv'     # ручная разметка (как в v6 — переиспользуется)
F_AUTO = f'{DRIVE_DIR}/features_eda_auto_v9.csv'     # поиск глаза не менялся с v9 — переиспользуется
F_CP = f'{DRIVE_DIR}/features_cp_conj_v6.csv'
CROP_DIR = f'{DS_DIR}/eye_crops_v9'                  # вырезы глаз датасета (для обучения поиска и стресс-теста)
N_AUG, N_STRESS = 3, 2          # вариантов света на снимок: для обучения / для проверки (другие случайные)
N_SEG_AUG = 2                   # вариантов света на снимок для обучения поиска конъюнктивы (+1 для проверки)
# Ускорение: если от прошлых версий уже есть такие же признаки (v5/v6) — берём их, а не считаем заново
for _dst in (F_SEG, F_CP):
    if not os.path.exists(_dst):
        _old = _dst.replace('_v6.csv', '_v5.csv')
        if os.path.exists(_old):
            shutil.copy(_old, _dst)
            print('⚡ Использую готовые признаки из прошлой версии:', os.path.basename(_old))
CONJ_BASE = ['EI', 'EI_lower', 'a_star', 'L_star', 'b_star', 'S', 'r_norm', 'g_norm',
             'PVM_R', 'PVM_G', 'PVM_EI', 'PVM_a', 'entropy']
SCL_BASE = ['scl_L', 'scl_a', 'scl_b', 'scl_blue_rel', 'scl_blue_ratio',
            'vessel_density', 'vessel_contrast', 'vessel_redness']
ADULT_BOUNDS = (11.0, 8.0)          # ВОЗ, взрослые небеременные: лёгкая от 11, умеренная 8–10.9, тяжёлая < 8
CHILD_BOUNDS = (10.0, 7.0)          # ВОЗ, дети 6–59 мес и беременные

# ---------------------------------------------------------------- Eyes-defy-anemia -> копия на Диске
def _norm_id(x):
    m = re.search(r'\d+', str(x))
    return str(int(m.group())) if m else None

def _country(p):
    p = p.lower()
    return 'India' if 'india' in p else 'Italy' if 'italy' in p else 'NA'

def _find_col(cols, pats):
    for c in cols:
        if any(re.search(p, str(c).strip().lower()) for p in pats):
            return c
    return None

def _copy_eda_one(row, dst):
    """Копирует на Диск разметку конъюнктивы (как есть) и исходное фото (уменьшенное до 1600 px, JPEG)."""
    try:
        seg_dst = f"{dst}/{row['key']}_conj.png"
        if not os.path.exists(seg_dst):
            shutil.copy(row['segment'], seg_dst)
        org_dst = None
        if isinstance(row['original'], str):
            org_dst = f"{dst}/{row['key']}_eye.jpg"
            if not os.path.exists(org_dst):
                bgr = cv2.imread(row['original'], cv2.IMREAD_COLOR)
                h, w = bgr.shape[:2]; s = min(1.0, 1600 / max(h, w))
                if s < 1:
                    bgr = cv2.resize(bgr, (int(w * s), int(h * s)), interpolation=cv2.INTER_AREA)
                cv2.imwrite(org_dst, bgr, [cv2.IMWRITE_JPEG_QUALITY, 93])
        return {**row, 'segment': seg_dst, 'original': org_dst}
    except Exception:
        return None

def load_eda():
    idx = f'{DS_DIR}/eda/index.csv'
    if os.path.exists(idx):
        return pd.read_csv(idx, dtype={'pid': str})
    print(f'   Скачиваю Eyes-defy-anemia (Kaggle) и сохраняю в {DS_DIR} (один раз)...')
    import kagglehub
    root = kagglehub.dataset_download('harshwardhanfartale/eyes-defy-anemia')
    recs = []
    for t in glob.glob(f'{root}/**/*', recursive=True):
        if not t.lower().endswith(('.xlsx', '.xls', '.csv')):
            continue
        sheets = {'csv': pd.read_csv(t)} if t.lower().endswith('.csv') else pd.read_excel(t, sheet_name=None)
        for _, df in sheets.items():
            hb = _find_col(df.columns, [r'^hg?b', r'hgb', r'hemoglob', r'haemoglob', r'emoglob'])
            pid = _find_col(df.columns, [r'^n\.?$', r'^no\.?$', r'^number', r'^id', r'patient', r'numero']) or df.columns[0]
            sex = _find_col(df.columns, [r'sex', r'gender', r'sesso'])
            if hb is None:
                continue
            for _, r in df.iterrows():
                try:
                    hbv = float(str(r[hb]).replace(',', '.'))
                except ValueError:
                    continue
                sx = str(r[sex]).strip().lower()[:1] if sex else ''
                recs.append({'country': _country(t), 'pid': _norm_id(r[pid]), 'Hb': hbv,
                             'sex': 'M' if sx == 'm' else 'F' if sx in ('f', 'w', 'd') else None})
    tab = pd.DataFrame(recs).dropna(subset=['pid', 'Hb'])
    folders = {}
    for p in glob.glob(f'{root}/**/*', recursive=True):
        if p.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')):
            folders.setdefault(os.path.dirname(p), []).append(p)
    rows = []
    for folder, files in folders.items():
        low = {f: os.path.basename(f).lower() for f in files}
        palp = [f for f, n in low.items() if 'palpebral' in n and 'forniceal' not in n]
        both = [f for f, n in low.items() if 'palpebral' in n and 'forniceal' in n]
        orig = [f for f, n in low.items() if 'palpebral' not in n and 'forniceal' not in n]
        if palp or both:
            rows.append({'country': _country(folder), 'pid': _norm_id(os.path.basename(folder)),
                         'segment': (palp or both)[0], 'original': orig[0] if orig else None})
    imgs = pd.DataFrame(rows)
    keys = ['country', 'pid'] if (tab.country != 'NA').any() else ['pid']
    d = imgs.merge(tab.drop_duplicates(keys), on=keys, how='inner').reset_index(drop=True)
    d['thr'] = np.where(d.sex == 'M', 13.0, 12.0)                      # ВОЗ, взрослые
    d['key'] = 'EDA_' + d.country.astype(str) + '_' + d.pid.astype(str)
    dst = f'{DS_DIR}/eda'; os.makedirs(dst, exist_ok=True)
    out = Parallel(n_jobs=-1)(delayed(_copy_eda_one)(r, dst) for r in d.to_dict('records'))
    d = pd.DataFrame([o for o in out if o])
    d.to_csv(idx, index=False)
    print(f'   ✅ Eyes-defy-anemia: {len(d)} пациентов сохранены в {dst}')
    return d

# ---------------------------------------------------------------- CP-AnemiC -> копия на Диске
def load_cp():
    idx = f'{DS_DIR}/cp/index.csv'
    if os.path.exists(idx):
        return pd.read_csv(idx, dtype={'pid': str})
    for zp in [z for z in glob.glob(f'{DRIVE_DIR}/*.zip') if zipfile.is_zipfile(z)]:
        tmp = 'cp_tmp'
        shutil.rmtree(tmp, ignore_errors=True)
        with zipfile.ZipFile(zp) as z:
            z.extractall(tmp)
        for inner in glob.glob(f'{tmp}/**/*.zip', recursive=True):
            with zipfile.ZipFile(inner) as z:
                z.extractall(os.path.dirname(inner))
        xl = glob.glob(f'{tmp}/**/Cp-Anaemic Dataset.xlsx', recursive=True)
        if not xl:
            continue                                                    # это не CP-AnemiC
        root = os.path.dirname(xl[0])
        meta = pd.read_excel(xl[0], sheet_name=0)
        dst = f'{DS_DIR}/cp'; os.makedirs(dst, exist_ok=True)
        rows = []
        for _, r in meta.iterrows():
            src = os.path.join(root, 'All Categories', str(r['File_Name']))
            if os.path.exists(src):
                key = f"CP_{r['Image_ID']}"
                shutil.copy(src, f'{dst}/{key}.png')
                rows.append({'key': key, 'pid': str(r['Image_ID']), 'Hb': float(r['Hb_g_dl']),
                             'segment': f'{dst}/{key}.png', 'sex': None, 'thr': 11.0})   # ВОЗ, 6–59 мес
        d = pd.DataFrame(rows)
        d.to_csv(idx, index=False)
        shutil.rmtree(tmp, ignore_errors=True)
        print(f'   ✅ CP-AnemiC: {len(d)} детей сохранены в {dst}')
        return d
    print(f'   ℹ️ Архив CP-AnemiC не найден в папке {DRIVE_DIR} — модель для детей до 5 лет не обучается.\n'
          '      (Без Google Диска: перетащите zip в панель «Файлы» слева и снова запустите ячейки 1 и 3.)')
    return None


# ---------------------------------------------------------------- извлечение признаков (параллельно, с контрольными точками)
def _conj_one(key, segment):
    try:
        rgb, mask = load_segment(segment)
        f = compute_indices(rgb, mask)
        return {'key': key, **f} if f else None
    except Exception:
        return None

def _to_crop(full_mask, eye, shape):
    """Маска на всём фото -> координаты выреза глаза."""
    x0, y0, x1, y1 = eye['box']
    sub = full_mask[y0:y1, x0:x1].astype(np.uint8)
    return cv2.resize(sub, (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST) > 0

def _eye_one(key, original, segment=None):
    """ТОТ ЖЕ путь, что для фото пользователя: поиск глаза -> конъюнктива и склера -> признаки.
    Для датасета дополнительно: разметка врача в координатах выреза и сам вырез (для обучения поиска)."""
    try:
        if not isinstance(original, str):
            return None
        rgb = load_image(original, 1200)
        eye, _, _ = best_eye(rgb, use_face=segment is None, require_conj=False)
        if eye is None:
            return None
        out = {'key': key}
        crop = eye['crop']
        save = {'crop': (np.clip(crop, 0, 1) * 255).astype(np.uint8), 'ic': np.array(eye['iris_crop'], np.float32)}
        for k in ('conj', 'white', 'zone', 'sclera'):
            if eye.get(k) is not None:
                save[k] = eye[k].astype(bool)
        if isinstance(segment, str):
            try:
                _, dm = load_segment(segment, 100000)
                if abs(dm.shape[0] / dm.shape[1] - rgb.shape[0] / rgb.shape[1]) < 0.02:
                    dm = cv2.resize(dm.astype(np.uint8), (rgb.shape[1], rgb.shape[0]),
                                    interpolation=cv2.INTER_NEAREST) > 0
                    save['doc'] = _to_crop(dm, eye, crop.shape)
            except Exception:
                pass
        os.makedirs(CROP_DIR, exist_ok=True)
        np.savez_compressed(f'{CROP_DIR}/{key}.npz', **save)
        if eye['conj'] is not None:
            fc = eye_conj_features(crop, eye['conj'], eye.get('sclera'))
            if fc:
                out.update({f'c_{k}': v for k, v in fc.items()})
        fs, _, _ = sclera_features(crop, eye['sclera'], eye['iris_crop'])
        if fs:
            out.update({f's_{k}': v for k, v in fs.items()})
        return out                 # даже без признаков: глаз найден, вырез сохранён (модель поиска может найти веко)
    except Exception:
        return None

# ---------------------------------------------------------------- обучение поиска конъюнктивы по разметке врачей
def _load_crop(key):
    p = f'{CROP_DIR}/{key}.npz'
    if not os.path.exists(p):
        return None
    with np.load(p) as z:
        d = {k: z[k] for k in z.files}
    d['crop'] = d['crop'].astype(np.float32) / 255
    d['ic'] = tuple(float(v) for v in d['ic'])
    return d

def _small(m, shape):
    return cv2.resize(m.astype(np.uint8), (shape[1], shape[0]), interpolation=cv2.INTER_NEAREST) > 0

def _iou(a, b):
    if a is None or b is None:
        return 0.0
    u = (a | b).sum()
    return float((a & b).sum() / u) if u else 0.0

def _seg_xy(d, rng, n_pos=700, n_neg=1400):
    """Пиксели одного фото для обучения: конъюнктива врача (+), рядом с ней (−, трудные) и дальше (−)."""
    X, roi, shp = seg_features(d['crop'], d['ic'], d.get('white'), d.get('zone'), d.get('conj'))
    doc = _small(d['doc'], shp).astype(np.uint8)
    r = d['ic'][2] / SEG_STEP
    near = cv2.dilate(doc, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (int(r) | 1,) * 2)) > 0
    y = doc[roi] > 0
    pos, nn, nf = np.flatnonzero(y), np.flatnonzero(near[roi] & ~y), np.flatnonzero(~near[roi])
    pick = lambda a, n: rng.choice(a, min(n, len(a)), replace=False) if len(a) else a
    idx = np.concatenate([pick(pos, n_pos), pick(nn, n_neg // 2), pick(nf, n_neg // 2)]).astype(int)
    return X[idx], y[idx]

def _seg_variant_one(key, seed):
    """Вырез глаза при N_SEG_AUG случайных светах (для обучения) + 1 свет для проверки (сохраняется на диск).
    Правила поиска пересчитываются на каждом варианте — как это будет в приложении."""
    try:
        d = _load_crop(key)
        rng = np.random.default_rng(seed)
        xy = []
        for i in range(N_SEG_AUG + 1):
            x = renormalize_crop(light_perturb(d['crop'], rng), d['ic'])
            rg = crop_regions(x, d['ic'])
            v = {'crop': x, 'ic': d['ic'], 'white': rg['white'], 'zone': rg['zone'], 'doc': d['doc']}
            if rg['conj'] is not None:
                v['conj'] = rg['conj']
            if i < N_SEG_AUG:
                xy.append(_seg_xy(v, rng))
            else:
                save = {'crop': (x * 255).astype(np.uint8), 'ic': np.array(d['ic'], np.float32), 'doc': d['doc']}
                for kk in ('white', 'zone', 'conj'):
                    if v.get(kk) is not None:
                        save[kk] = v[kk].astype(bool)
                np.savez_compressed(f'{CROP_DIR}/light_{key}.npz', **save)
        return xy, None
    except Exception:
        return [], None

def _load_light(key):
    return _load_crop('light_' + key)

def _seg_fit(XY):
    X = np.vstack([a for a, _ in XY]); y = np.concatenate([b for _, b in XY])
    # баланс классов 50/50 оверсэмплингом меньшего класса (не sample_weight — см. E в шапке)
    rng = np.random.default_rng(SEED)
    pos, neg = np.flatnonzero(y), np.flatnonzero(~y)
    small, big = (pos, neg) if len(pos) < len(neg) else (neg, pos)
    idx = np.concatenate([big, small, rng.choice(small, len(big) - len(small), replace=True)]) if len(small) else big
    m = HistGradientBoostingClassifier(max_iter=150, learning_rate=0.1, max_leaf_nodes=31,
                                       l2_regularization=1.0, random_state=SEED)
    return m.fit(X[idx], y[idx])

def train_conj_segmenter(keys):
    """Попиксельная модель конъюнктивы + честная проверка на НЕвиденных пациентах (5 частей).
    Сравнение с правилами на тех же фото. Возвращает (состояние модели, маски вне обучения)."""
    T = []
    for k in keys:
        d = _load_crop(k)
        if d is not None and 'doc' in d and d['doc'].sum() >= 0.3 * d['ic'][2] ** 2:
            T.append(k)
    if len(T) < 40:
        print(f'   Поиск конъюнктивы: фото с разметкой врача в кадре только {len(T)} — модель не обучается, остаются правила.')
        return None, {}
    t = time.time()
    rng = np.random.default_rng(SEED)
    XY = [_seg_xy(_load_crop(k), rng) for k in T]
    # те же снимки при другом свете: правила пересчитываются на «пересъёмке», разметка врача та же
    var = Parallel(n_jobs=-1)(delayed(_seg_variant_one)(k, SEED + 104729 * i) for i, k in enumerate(T))
    XYa = [v[0] for v in var]
    probs, probs_l = {}, {}
    for tr, te in GroupKFold(n_splits=5).split(T, groups=T):
        m = _seg_fit([XY[i] for i in tr] + [xy for i in tr for xy in XYa[i]])
        for i in te:
            d = _load_crop(T[i])
            probs[T[i]] = seg_probability(d['crop'], d['ic'], d.get('white'), d.get('zone'), d.get('conj'), m)
            dl = _load_light(T[i])
            if dl is not None:
                probs_l[T[i]] = (seg_probability(dl['crop'], dl['ic'], dl.get('white'), dl.get('zone'), dl.get('conj'), m),
                                 _small(dl['conj'], probs[T[i]].shape) if 'conj' in dl else None)
    docs, rules = {}, {}
    for k in T:
        d = _load_crop(k)
        shp = probs[k].shape
        docs[k] = _small(d['doc'], shp)
        rules[k] = _small(d['conj'], shp) if 'conj' in d else None
    rs = {k: _load_crop(k)['ic'][2] / SEG_STEP for k in T}
    # оценивается ровно то, что работает в приложении: модель уточняет веко, найденное правилами
    KL = [k for k in T if k in probs_l]
    THRS = np.arange(0.3, 0.76, 0.05)
    def _iou_at(thr, ks):
        a = [_iou(combine_conj(rules[k], _seg_post(probs[k], thr, rs[k])), docs[k]) for k in ks]
        b = [_iou(combine_conj(probs_l[k][1], _seg_post(probs_l[k][0], thr, rs[k])), docs[k]) for k in ks if k in probs_l]
        return a, b
    def _pick_thr(ks):
        sc = []
        for thr in THRS:
            a, b = _iou_at(thr, ks)
            sc.append(0.5 * np.mean(a) + 0.5 * (np.mean(b) if b else np.mean(a)))
        return float(THRS[int(np.argmax(sc))])
    # ИСПРАВЛЕНО: порог подбирается на ДРУГИХ фолдах и применяется к этому (раньше — на тех же фото, где считался IoU)
    iou_l, iou_ll = [], []
    for tr, te in GroupKFold(n_splits=5).split(T, groups=T):
        t_ = _pick_thr([T[i] for i in tr])
        a, b = _iou_at(t_, [T[i] for i in te])
        iou_l += a; iou_ll += b
    thr = _pick_thr(T)                                    # порог для итоговой модели — по всем фото
    iou_r = [_iou(rules[k], docs[k]) for k in T]
    iou_rl = [_iou(probs_l[k][1], docs[k]) for k in KL]
    model = _seg_fit(XY + [xy for v in XYa for xy in v])
    use = float(np.median(iou_l)) >= float(np.median(iou_r)) + 0.03
    st = {'model': model, 'thr': thr, 'use': bool(use), 'n': len(T),
          'iou_learned': float(np.median(iou_l)), 'iou_rule': float(np.median(iou_r)),
          'iou_learned_light': float(np.median(iou_ll)) if iou_ll else None,
          'iou_rule_light': float(np.median(iou_rl)) if iou_rl else None,
          'iou_learned_mean': float(np.mean(iou_l)), 'iou_rule_mean': float(np.mean(iou_r)),
          'found_learned': float(np.mean([v > 0 for v in iou_l])), 'found_rule': float(np.mean([v > 0 for v in iou_r]))}
    lt = (f"; при другом свете — модель {st['iou_learned_light']:.0%}, правила {st['iou_rule_light']:.0%}"
          if iou_ll else '')
    print(f"   Поиск конъюнктивы (проверка на {len(T)} невиденных фото, {time.time() - t:.0f} с): совпадение с врачами "
          f"(IoU, медиана) — модель {st['iou_learned']:.0%}, правила {st['iou_rule']:.0%}{lt} → "
          f"{'используется МОДЕЛЬ' if use else 'модель не лучше — остаются ПРАВИЛА'}")
    oof = {k: _seg_post(probs[k], thr, rs[k]) for k in T}
    return st, oof

def auto_conj_features(fauto, seg, oof):
    """Признаки конъюнктивы датасета из той области, которую найдёт приложение на фото пользователя
    (для фото с разметкой — маска модели, обученной БЕЗ этого пациента)."""
    rows, masks = [], {}
    for k in fauto.key.astype(str):
        d = _load_crop(k)
        if d is None:
            continue
        crop, r = d['crop'], d['ic'][2]
        m = d.get('conj')                                 # веко, найденное правилами (как в приложении)
        if m is not None and seg is not None and seg['use']:
            if k in oof:
                ml = None if oof[k] is None else _small(oof[k], crop.shape)
            else:
                ml = conj_learned(crop, d['ic'], d.get('white'), d.get('zone'), m, seg)
            m = combine_conj(m, ml)
        row = {'key': k}
        if m is not None and m.sum() >= 0.3 * r ** 2:
            masks[k] = m
            fc = eye_conj_features(crop, m, d.get('sclera'))
            if fc:
                row.update({f'c_{kk}': v for kk, v in fc.items()})
                if 'doc' in d:
                    inter = float((m & d['doc']).sum())
                    row['overlap'] = inter / max(m.sum(), 1)
                    row['iou'] = inter / max((m | d['doc']).sum(), 1)
        rows.append(row)
    new = pd.DataFrame(rows)
    keep = fauto[[c for c in fauto.columns if not c.startswith('c_') and c not in ('overlap', 'iou')]].copy()
    keep['key'] = keep.key.astype(str)
    return keep.merge(new, on='key', how='left'), masks

# ---------------------------------------------------------------- «другой свет, другая камера»: обучение и стресс-тест
def _aug_eye_one(key, m, seed, seg=None):
    """Признаки одного глаза датасета при N_AUG + N_STRESS случайных светах (после того же выравнивания,
    что делает приложение). Области конъюнктивы и склеры — те же."""
    try:
        d = _load_crop(key)
        if d is None:
            return []
        rows = []
        for part, n, off in (('aug', N_AUG, 0), ('stress', N_STRESS, 500_000)):
            rng = np.random.default_rng(seed + off)
            for i in range(n):
                x = renormalize_crop(light_perturb(d['crop'], rng), d['ic'])
                row = {'key': key, 'part': part}
                # области ищутся ЗАНОВО на «пересъёмке» (как в приложении): ошибка поиска при другом свете
                # тоже входит в проверку. Признаки — только если на исходном снимке веко было найдено.
                rg = crop_regions(x, d['ic'], seg)
                mc = rg['conj']
                if m is not None and mc is not None and mc.sum() >= 0.3 * d['ic'][2] ** 2:
                    fc = eye_conj_features(x, mc, rg['sclera'])
                    if fc:
                        row.update({f'c_{kk}': v for kk, v in fc.items()})
                fs, _, _ = sclera_features(x, rg['sclera'], d['ic'])
                if fs:
                    row.update({f's_{kk}': v for kk, v in fs.items()})
                rows.append(row)
        return rows
    except Exception:
        return []

def _aug_seg_one(key, segment, seed):
    """То же для уже вырезанной конъюнктивы (белка рядом нет — свет выровнять нечем)."""
    try:
        rgb, mask = load_segment(segment)
        rows = []
        for part, n, off in (('aug', N_AUG, 0), ('stress', N_STRESS, 500_000)):
            rng = np.random.default_rng(seed + off)
            for i in range(n):
                f = compute_indices(light_perturb(rgb, rng), mask)
                if f:
                    rows.append({'key': key, 'part': part, **f})
        return rows
    except Exception:
        return []

def _aug_table(fn, items, label, **kw):
    t = time.time()
    out = Parallel(n_jobs=-1)(delayed(fn)(*it, SEED + 7919 * i, **kw) for i, it in enumerate(items))
    df = pd.DataFrame([r for rows in out for r in rows])
    print(f'   {label}: {len(df)} вариантов снимков при другом свете ({time.time() - t:.0f} с)')
    return df

def _extract(fn, items, path, label, chunk=40):
    if os.path.exists(path):
        return pd.read_csv(path)
    part = path + '.part.csv'
    done = pd.read_csv(part) if os.path.exists(part) else pd.DataFrame({'key': []})
    tried = set(pd.read_csv(part + '.tried')['key'].astype(str)) if os.path.exists(part + '.tried') else set()
    todo = [it for it in items if str(it[0]) not in tried]
    t = time.time()
    print(f'   {label}: осталось обработать {len(todo)} из {len(items)}')
    for i in range(0, len(todo), chunk):
        batch = todo[i:i + chunk]
        out = Parallel(n_jobs=-1)(delayed(fn)(*it) for it in batch)
        new = pd.DataFrame([o for o in out if o])
        if len(new):
            done = pd.concat([done, new], ignore_index=True)
        done.to_csv(part, index=False)
        tried |= {str(it[0]) for it in batch}
        pd.DataFrame({'key': sorted(tried)}).to_csv(part + '.tried', index=False)
        print(f'      … {min(i + chunk, len(todo))}/{len(todo)}  ({time.time() - t:.0f} с)', flush=True)
    done.to_csv(path, index=False)
    for f_ in (part, part + '.tried'):
        if os.path.exists(f_):
            os.remove(f_)
    return done

# ---------------------------------------------------------------- модель + калибровка
def _balance_weights(y):
    """Веса против дисбаланса: редкие значения Hb (тяжёлая/умеренная анемия) весят больше."""
    bins = np.digitize(y, [8.0, 11.0, 12.5, 14.0])
    cnt = np.bincount(bins, minlength=5).astype(float)
    w = len(y) / (np.count_nonzero(cnt) * cnt[bins])
    return np.clip(w, 0.5, 4.0)

def _model(kind):
    if kind == 'ridge':
        return make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 21)))
    return HistGradientBoostingRegressor(max_iter=300, learning_rate=0.05, max_depth=3,
                                         min_samples_leaf=15, l2_regularization=1.0, random_state=SEED)

def _fit_w(kind, X, y, w):
    m = _model(kind)
    if kind == 'ridge':
        m.fit(X, y, ridgecv__sample_weight=w)
    else:                                   # веса -> повторы строк (sample_weight в HGB sklearn ≥ 1.9 очень медленный)
        rep_ = np.maximum(np.round(np.asarray(w) / max(np.min(w), 1e-6) * 1.0), 1).astype(int)
        rep_ = np.minimum(rep_, 12)
        m.fit(np.repeat(X, rep_, axis=0), np.repeat(y, rep_))
    return m

def _groups(df):
    """Группа для GroupKFold: человек (несколько фото одного человека — одна группа)."""
    if 'grp' in df.columns:
        return df['grp'].fillna(df['key']).astype(str).values
    return df['key'].astype(str).values

def _oof(kind, X, y, w, groups, aug=None, stress=None, keys=None):
    """Прогноз для каждого пациента моделью, которая его НЕ видела. aug = (X, y, w, группы) — варианты
    обучающих снимков при другом свете (добавляются к обучению только для пациентов обучающей части);
    stress = (X, группы) — снимки проверяемых пациентов при ДРУГИХ светах (ошибка «при другом свете»)."""
    oof = np.zeros(len(y))
    oof_s = np.full(len(stress[0]), np.nan) if stress is not None else None
    keys = groups if keys is None else keys
    for tr, te in GroupKFold(n_splits=5).split(X, y, groups):
        Xt, yt, wt = X[tr], y[tr], w[tr]
        if aug is not None:
            ma = np.isin(aug[3], keys[tr])
            Xt, yt, wt = np.vstack([Xt, aug[0][ma]]), np.concatenate([yt, aug[1][ma]]), np.concatenate([wt, aug[2][ma]])
        m = _fit_w(kind, Xt, yt, wt)
        oof[te] = m.predict(X[te])
        if stress is not None:
            ms = np.isin(stress[1], keys[te])
            if ms.any():
                oof_s[ms] = m.predict(stress[0][ms])
    return oof, oof_s

def _band_mae(y, p):
    """Средняя ошибка, посчитанная ОТДЕЛЬНО по диапазонам Hb и усреднённая — честнее для редких групп."""
    bands = np.digitize(y, [8.0, 11.0, 12.5, 14.0])
    return float(np.mean([np.abs(p[bands == b] - y[bands == b]).mean() for b in np.unique(bands)]))

def _youden(score, positive, lo, hi):
    """Порог t в [lo, hi]: «положительный», если score < t; максимум чувствительность + специфичность.
    Диапазон ограничен (±1 г/дл от границы ВОЗ), чтобы малые данные не уводили границы в абсурд."""
    if positive.sum() < 5 or (~positive).sum() < 5:
        return None
    cand = np.linspace(lo, hi, 26)
    return float(max(cand, key=lambda t: ((score < t) & positive).sum() / positive.sum()
                     + ((score >= t) & ~positive).sum() / (~positive).sum()))

def _calibration(y, pred, thr, bounds, d_floor):
    """1) частично снимает сжатие к среднему (наклон d не ниже d_floor: при слабом сигнале
       сильная «растяжка» только усилила бы шум); 2) уточняет границы степеней в пределах ±1 г/дл
       от границ ВОЗ; 3) калибрует вероятность анемии (логистическая регрессия)."""
    mild_lo, mod_lo = bounds
    d = float(np.polyfit(y, pred, 1)[0])
    d = float(np.clip(d, d_floor, 1.0))
    c = float(np.mean(pred) - d * np.mean(y))
    hb = (pred - c) / d
    cal = {'c': c, 'd': d, 'd_floor': d_floor, 'sigma': float(np.std(y - hb, ddof=1))}
    rel = hb - thr
    dn = _youden(rel, y < thr, -1.0, 1.5)
    cal['cut_norm'] = dn if dn is not None else 0.0
    cm = _youden(hb, y < mild_lo, mild_lo - 1, mild_lo + 1)
    cs = _youden(hb, y < mod_lo, mod_lo - 1, mod_lo + 1)
    cal['cut_mod'] = cm if cm is not None else mild_lo
    cal['cut_sev'] = min(cs if cs is not None else mod_lo, cal['cut_mod'] - 0.5)
    an = (y < thr).astype(int)
    if 0 < an.sum() < len(an):
        lr = LogisticRegression().fit(rel.reshape(-1, 1), an)
        cal['p_a'], cal['p_b'] = float(lr.intercept_[0]), float(lr.coef_[0][0])
    return cal

def apply_cal(cal, raw, thr):
    hb = (raw - cal['c']) / cal['d']
    z = hb - thr
    p = 1 / (1 + np.exp(-(cal['p_a'] + cal['p_b'] * z))) if 'p_a' in cal else norm.cdf(-z / cal['sigma'])
    return hb, p

def grade_cal(cal, hb, thr, bounds):
    """Степень анемии по откалиброванным границам."""
    if hb >= thr + cal['cut_norm']:
        return 0
    if hb >= cal['cut_mod']:
        return 1
    if hb >= cal['cut_sev']:
        return 2
    return 3

GRADE_NAMES = ['Норма', 'Лёгкая анемия', 'Умеренная анемия', 'Тяжёлая анемия']

def _true_grade(y, thr, bounds):
    mild_lo, mod_lo = bounds
    return np.where(y >= thr, 0, np.where(y >= mild_lo, 1, np.where(y >= mod_lo, 2, 3)))

def _summ(y, gt, g, hbx, shown_anemia):
    conf = pd.crosstab(pd.Series(gt, name='истина'), pd.Series(g, name='прогноз')).reindex(
        index=range(4), columns=range(4), fill_value=0)
    rec = [conf.loc[k, k] / conf.loc[k].sum() for k in range(4) if conf.loc[k].sum() > 0]
    healthy = gt == 0
    return {'confusion': conf.values.tolist(), 'balanced_acc': float(np.mean(rec)),
            'exact': float((g == gt).mean()), 'MAE': float(np.abs(hbx - y).mean()),
            'bias': {name: float((hbx[gt == k] - y[gt == k]).mean()) for k, name in enumerate(GRADE_NAMES) if (gt == k).any()},
            # что реально увидит человек: «анемия» (а не «пограничный») при нормальном Hb
            'false_anemia_healthy': float(shown_anemia[healthy].mean()) if healthy.any() else np.nan,
            'missed_anemia': float((~shown_anemia[~healthy]).mean()) if (~healthy).any() else np.nan}

def _evaluate(y, pred, thr, bounds, groups, d_floor):
    """ЧЕСТНАЯ оценка: калибровка подбирается на одних пациентах и проверяется на других (кросс-фиттинг).
    'raw' — прежнее правило (точечная оценка против порогов ВОЗ, «анемия» при вероятности ≥ 70%)."""
    gt = _true_grade(y, thr, bounds)
    sig_raw = float(np.std(y - pred, ddof=1))
    p_raw = norm.cdf((thr - pred) / sig_raw)
    g_raw = np.array([0 if p >= t else 1 if p >= bounds[0] else 2 if p >= bounds[1] else 3 for p, t in zip(pred, thr)])
    shown_raw = p_raw >= 0.7
    g_cal = np.zeros(len(y), int); hb_cal = np.zeros(len(y)); p_cal = np.zeros(len(y))
    for tr, te in GroupKFold(n_splits=5).split(pred, y, groups):
        cal = _calibration(y[tr], pred[tr], thr[tr], bounds, d_floor)
        h, p = apply_cal(cal, pred[te], thr[te])
        hb_cal[te], p_cal[te] = h, p
        g_cal[te] = [grade_cal(cal, hh, tt, bounds) for hh, tt in zip(h, thr[te])]
    border = (g_cal <= 1) & (p_cal > 0.35) & (p_cal < 0.65)
    shown_cal = (g_cal > 0) & ~border
    res = {'raw': _summ(y, gt, g_raw, pred, shown_raw), 'calibrated': _summ(y, gt, g_cal, hb_cal, shown_cal),
           'borderline_share': float(border.mean()), 'd_floor': d_floor}
    an = (y < thr).astype(int)
    if 0 < an.sum() < len(an):
        res['AUC'] = float(roc_auc_score(an, p_cal))
    return res

def _identity_cal(y, pred, bounds):
    """«Без калибровки»: прежнее правило (пороги ВОЗ по точечной оценке, вероятность по нормальному закону)."""
    return {'c': 0.0, 'd': 1.0, 'd_floor': None, 'sigma': float(np.std(y - pred, ddof=1)),
            'cut_norm': 0.0, 'cut_mod': bounds[0], 'cut_sev': bounds[1], 'identity': True}

def _choose_calibration(y, pred, thr, bounds, groups):
    """Нужна ли калибровка и насколько сильная — решают ДАННЫЕ (на новых пациентах).
    Калибровка принимается, только если повышает долю верных степеней и НЕ увеличивает ложную
    «анемию» у здоровых; иначе остаётся прежнее правило."""
    best = None
    for d_floor in (1.0, 0.8, 0.65, 0.5):
        ev = _evaluate(y, pred, thr, bounds, groups, d_floor)
        c, r = ev['calibrated'], ev['raw']
        score = c['balanced_acc'] - 2.0 * max(0.0, c['false_anemia_healthy'] - r['false_anemia_healthy'])
        if best is None or score > best[0]:
            best = (score, d_floor, ev)
    score, d_floor, ev = best
    if score <= ev['raw']['balanced_acc'] + 0.01:            # калибровка не помогает — не применяем
        ev = dict(ev); ev['calibrated'] = ev['raw']; ev['borderline_share'] = np.nan
        return None, ev
    return d_floor, ev

def _fit_method(df, feats, label, bounds, extra_w=None, aug=None, stress=None, quiet=False):
    """Модель Hb для одного метода. aug/stress — таблицы того же вида (key, Hb, признаки) при другом свете."""
    X, y = df[feats].values, df.Hb.values
    thr, groups, keys = df.thr.values, _groups(df), df.key.astype(str).values
    w = _balance_weights(y) * (extra_w if extra_w is not None else 1.0)
    A_ = S_ = None
    if aug is not None and len(aug):
        aug = aug[aug.key.isin(df.key)].dropna(subset=feats)
        A_ = (aug[feats].values, aug.Hb.values, 0.5 * _balance_weights(aug.Hb.values), aug.key.values)
    if stress is not None and len(stress):
        stress = stress[stress.key.isin(df.key)].dropna(subset=feats)
        S_ = (stress[feats].values, stress.key.values) if len(stress) else None
    best = None
    for kind in ('ridge', 'gbr'):
        oof, oof_s = _oof(kind, X, y, w, groups, A_, S_, keys)
        clean = _band_mae(y, oof)
        light = _band_mae(stress.Hb.values, oof_s) if S_ is not None else clean
        score = 0.5 * clean + 0.5 * light
        if best is None or score < best[0]:
            best = (score, kind, oof, oof_s, clean, light)
    score, kind, oof, oof_s, clean, light = best
    d_floor, ev = _choose_calibration(y, oof, thr, bounds, groups)
    ev['MAE_light'] = float(np.abs(oof_s - stress.Hb.values).mean()) if S_ is not None else None
    if not quiet:
        print(f"   {label}: модель {kind}; ошибка Hb ±{ev['calibrated']['MAE'] * 10:.0f} г/л"
              + (f" (при другом свете ±{ev['MAE_light'] * 10:.0f})" if ev['MAE_light'] is not None else '')
              + f"; верная степень {ev['raw']['balanced_acc']:.0%} → {ev['calibrated']['balanced_acc']:.0%}; "
              f"ложная «анемия» у здоровых {ev['raw']['false_anemia_healthy']:.0%} → {ev['calibrated']['false_anemia_healthy']:.0%}")
    Xf, yf, wf = X, y, w
    if A_ is not None:
        Xf, yf, wf = np.vstack([X, A_[0]]), np.concatenate([y, A_[1]]), np.concatenate([w, A_[2]])
    rng_df = pd.concat([df[feats], aug[feats]]) if A_ is not None else df[feats]
    cal = _calibration(y, oof, thr, bounds, d_floor) if d_floor is not None else _identity_cal(y, oof, bounds)
    # ожидаемая ошибка метода для выбора «какой метод верить»: и при обычном, и при другом свете
    res_l = (stress.Hb.values - oof_s) if S_ is not None else (y - oof)
    cal['sigma_sel'] = float(np.sqrt(0.5 * cal['sigma'] ** 2 + 0.5 * np.var(res_l, ddof=1)))
    return {'model': _fit_w(kind, Xf, yf, wf), 'kind': kind, 'feats': feats, 'oof': oof, 'keys': keys,
            'prev': float(np.mean(y < thr)),
            'cal': cal, 'eval': ev, 'bounds_who': bounds, 'score': float(score), 'mae_clean': clean, 'mae_light': light,
            'bounds': {f: (float(np.percentile(rng_df[f], 1)), float(np.percentile(rng_df[f], 99))) for f in feats},
            'hb_range': (float(y.min()), float(y.max())), 'n': int(len(y))}

def _fit_best(df, sets, label, bounds, aug=None, stress=None, local_w=True, baseline=None, quiet=False):
    """Пробует наборы признаков и берёт тот, у которого меньше ошибка на НЕвиденных пациентах —
    в среднем при обычном свете и при другом свете/камере. Печатает сравнение (для защиты).
    ВНИМАНИЕ (аудит): ошибка выбранного варианта здесь занижена — по ней же он и выбран. Честная цифра —
    _nested_eval. sets может быть функцией sets(df): тогда, например, порог HHR выбирается по df."""
    if callable(sets):
        sets = sets(df)
    res = []
    for name, feats in sets.items():
        D = df.dropna(subset=feats).reset_index(drop=True)
        if len(D) < 40:
            continue
        ew = np.where(D.key.astype(str).str.startswith('LOCAL_'), 3.0, 1.0) if local_w else None
        m = _fit_method(D, feats, label, bounds, ew, aug, stress, quiet=True)
        m['feature_set'] = name
        res.append(m)
    if not res:
        return None
    best = min(res, key=lambda m: m['score'])
    if baseline in sets and stress is not None and not quiet:        # «как было до v10»: без обучения на другом свете
        D = df.dropna(subset=sets[baseline]).reset_index(drop=True)
        b = _fit_method(D, sets[baseline], label, bounds, None, None, stress, quiet=True)
        best['baseline'] = (b['mae_clean'], b['mae_light'])
    best['alternatives'] = {m['feature_set']: (m['mae_clean'], m['mae_light']) for m in res}
    if quiet:
        return best
    ev = best['eval']
    alt = '; '.join(f"{k}: ±{a * 10:.0f}/±{b * 10:.0f}" for k, (a, b) in best['alternatives'].items())
    print(f"   {label}: признаки «{best['feature_set']}», модель {best['kind']} (n={best['n']}); "
          f"ошибка Hb ±{ev['calibrated']['MAE'] * 10:.0f} г/л, при другом свете "
          f"±{(ev['MAE_light'] or ev['calibrated']['MAE']) * 10:.0f}; верная степень {ev['calibrated']['balanced_acc']:.0%}; "
          f"ложная «анемия» у здоровых {ev['calibrated']['false_anemia_healthy']:.0%}\n"
          f"      (сравнение наборов: ошибка, усреднённая по диапазонам Hb, обычный/другой свет, г/л — {alt})"
          + (f"\n      без обучения на другом свете (как до v10): ±{best['baseline'][0] * 10:.0f}/±{best['baseline'][1] * 10:.0f}"
             if best.get('baseline') else ''))
    return best

def _local_table():
    p = f'{LOCAL_DIR}/features.csv'
    return pd.read_csv(p) if os.path.exists(p) else None

REL_FEATS = ['rel_dA', 'rel_dB', 'rel_L', 'rel_dEI', 'rel_R', 'rel_G', 'rel_RG']

# ---------------------------------------------------------------- ЧЕСТНАЯ оценка: вложенная перекрёстная проверка
def _wilson(k, n, z=1.96):
    if n == 0:
        return (np.nan, np.nan)
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d; h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return (float(c - h), float(c + h))

def _apply_model(m, D):
    """Как в приложении (ячейка 4): признаки ограничены диапазоном обучения, Hb — диапазоном Hb обучения,
    затем калибровка. Возвращает (Hb, вероятность, степень, маска строк, где признаки есть)."""
    ok = D[m['feats']].notna().all(axis=1).to_numpy().copy()
    hb = np.full(len(D), np.nan); p = np.full(len(D), np.nan); g = np.full(len(D), -1)
    if ok.any():
        X = np.column_stack([np.clip(D[f].values[ok].astype(float), *m['bounds'][f]) for f in m['feats']])
        a, b = m['hb_range']
        raw = np.clip(m['model'].predict(X), a - 0.5, b + 0.5)
        hb[ok], p[ok] = apply_cal(m['cal'], raw, D.thr.values[ok])
        g[ok] = [grade_cal(m['cal'], h, t, m['bounds_who']) for h, t in zip(hb[ok], D.thr.values[ok])]
    return hb, p, g, ok

def _shown_anemia(g, p, identity):
    """То, что увидит человек (логика ячейки 4): «анемия», если степень ≥ 1 и результат не «пограничный»."""
    identity = np.asarray(identity, bool)
    lo, hi = np.where(identity, 0.3, 0.35), np.where(identity, 0.7, 0.65)
    border = (g <= 1) & (((p > lo) & (p < hi)) | ((g == 0) & (p >= hi)) | ((g == 1) & (p <= lo)))
    return (g >= 1) & ~border, border

def _metrics(y, thr, bounds, hb, p, g, shown, border, covered, hb_light=None, y_light=None):
    y, thr = np.asarray(y), np.asarray(thr)
    c = covered
    an = y < thr
    out = {'n': int(len(y)), 'covered': float(c.mean()), 'MAE': float(np.abs(hb[c] - y[c]).mean()),
           'bias': float((hb[c] - y[c]).mean())}
    d = hb[c] - y[c]
    out['LoA'] = (float(d.mean() - 1.96 * d.std(ddof=1)), float(d.mean() + 1.96 * d.std(ddof=1)))
    if 0 < an[c].sum() < c.sum():
        out['AUC'] = float(roc_auc_score(an[c], p[c]))
    tp = int((shown & an & c).sum()); P_ = int((an & c).sum())
    tn = int((~shown & ~an & c).sum()); N_ = int((~an & c).sum())
    out.update(sens=tp / max(P_, 1), sens_ci=_wilson(tp, P_), spec=tn / max(N_, 1), spec_ci=_wilson(tn, N_),
               borderline_share=float(border[c].mean()))
    gt = _true_grade(y[c], thr[c], bounds)
    rec = [np.mean(g[c][gt == k] == k) for k in range(4) if (gt == k).any()]
    out['balanced_acc'] = float(np.mean(rec))
    if hb_light is not None and len(hb_light):
        ok = ~np.isnan(hb_light)
        out['MAE_light'] = float(np.abs(hb_light[ok] - y_light[ok]).mean()) if ok.any() else None
    return out

def _nested_eval(df, sets, label, bounds, aug=None, stress=None, local_w=True, train_mask=None,
                 fallback_sets=None, keep_preds=False):
    """ЧЕСТНАЯ оценка всей процедуры: во внешнем цикле (5 фолдов по людям) на 4/5 данных ЗАНОВО выбираются
    порог HHR, набор признаков, Ridge/бустинг и калибровка; прогноз — для 1/5, которую процедура не видела.
    train_mask — какие строки можно брать в обучение (проверка всегда на всех). fallback_sets — запасная
    модель для строк без части признаков (как conj_auto_abs в приложении)."""
    df = df.reset_index(drop=True)
    grp = _groups(df)
    n = len(df)
    hb = np.full(n, np.nan); p = np.full(n, np.nan); g = np.full(n, -1); ident = np.zeros(n, bool)
    raw_store = {}
    light_hb, light_y = [], []
    for tr, te in GroupKFold(n_splits=5).split(df, groups=grp):
        tr_ = tr if train_mask is None else tr[np.asarray(train_mask)[tr]]
        Dtr, Dte = df.iloc[tr_], df.iloc[te]
        m = _fit_best(Dtr, sets, label, bounds, aug, stress, local_w, quiet=True)
        if m is None:
            continue
        models = [m]
        if fallback_sets is not None:
            mf = _fit_best(Dtr, fallback_sets, label, bounds, aug, stress, local_w, quiet=True)
            if mf is not None:
                models.append(mf)
        left = np.ones(len(te), bool)
        for mm in models:
            h_, p_, g_, ok = _apply_model(mm, Dte)
            ok = ok & left
            idx = te[ok]
            hb[idx], p[idx], g[idx] = h_[ok], p_[ok], g_[ok]
            ident[idx] = bool(mm['cal'].get('identity'))
            left &= ~ok
        if keep_preds:
            raw_store[tuple(te)] = models
        if stress is not None and len(stress):
            St = stress[stress.key.astype(str).isin(Dte.key.astype(str))]
            for mm in models[:1]:
                St_ = St.dropna(subset=mm['feats'])
                if len(St_):
                    h_, _, _, ok = _apply_model(mm, St_)
                    light_hb.append(h_[ok]); light_y.append(St_.Hb.values[ok])
    covered = ~np.isnan(hb)
    shown, border = _shown_anemia(g, np.nan_to_num(p), ident)
    res = _metrics(df.Hb.values, df.thr.values, bounds, hb, p, g, shown, border, covered,
                   np.concatenate(light_hb) if light_hb else None, np.concatenate(light_y) if light_y else None)
    res['oof_hb'] = hb
    res['keys'] = df.key.astype(str).values
    return res

def _fmt_nested(r):
    s = (f"n={r['n']}, оценка выдана {r['covered']:.0%}; MAE ±{r['MAE'] * 10:.1f} г/л "
         f"(Bland–Altman: сдвиг {r['bias'] * 10:+.1f}, границы {r['LoA'][0] * 10:+.0f}…{r['LoA'][1] * 10:+.0f} г/л)")
    if r.get('MAE_light') is not None:
        s += f", при другом свете ±{r['MAE_light'] * 10:.1f}"
    if r.get('AUC') is not None:
        s += f"; AUC {r['AUC']:.3f}"
    s += (f"; чувствительность {r['sens']:.0%} [{r['sens_ci'][0]:.0%}–{r['sens_ci'][1]:.0%}], "
          f"специфичность {r['spec']:.0%} [{r['spec_ci'][0]:.0%}–{r['spec_ci'][1]:.0%}]; "
          f"верная степень {r['balanced_acc']:.0%}; «пограничный» {r['borderline_share']:.0%}")
    return s

def _pick_hhr(D):
    """Порог HHR, лучше всего связанный с Hb, — ТОЛЬКО по переданным (обучающим) строкам."""
    cols = [c for c in D.columns if c.startswith('HHR_') and D[c].notna().sum() > 10]
    return max(cols, key=lambda c: abs(np.nan_to_num(D[c].corr(D.Hb))))

def _crop_sets(D, sex=True):
    h = _pick_hhr(D)
    ABS = CONJ_BASE + [h]; INV = ['EI', 'EI_lower', 'PVM_EI', h, 'r_norm', 'g_norm']
    sx = ['sex_male'] if sex else []
    return {'абсолютные': ABS + sx, 'отношения цветов': INV + sx}

def _conj_sets(D):
    h = _pick_hhr(D)
    ABS = CONJ_BASE + [h]; INV = ['EI', 'EI_lower', 'PVM_EI', h, 'r_norm', 'g_norm']
    return {'абсолютные': ABS + ['sex_male'], 'относительно белка': INV + REL_FEATS + ['sex_male'],
            'все': ABS + REL_FEATS + ['sex_male']}

def _conj_abs_sets(D):
    return {'абсолютные': CONJ_BASE + [_pick_hhr(D), 'sex_male']}

def _nested_comb(J, Aa, S, conj_sets, scl_sets, aug_c, str_c, aug_s, str_s, conj_mask):
    """Честная оценка «оба метода»: в каждом внешнем фолде обе модели и их объединение (стекинг по
    внутренним OOF-прогнозам) обучаются заново без проверяемых людей."""
    J = J.reset_index(drop=True)
    grp = _groups(J)
    hb = np.full(len(J), np.nan); p = np.full(len(J), np.nan); g = np.full(len(J), -1); ident = np.zeros(len(J), bool)
    for tr, te in GroupKFold(n_splits=5).split(J, groups=grp):
        ktr = set(J.key.iloc[tr].astype(str)); kte = J.key.iloc[te].astype(str)
        Atr = Aa[Aa.key.astype(str).isin(ktr) & conj_mask]; Str_ = S[S.key.astype(str).isin(ktr)]
        mc = _fit_best(Atr, conj_sets, '', ADULT_BOUNDS, aug_c, str_c, quiet=True)
        ms = _fit_best(Str_, scl_sets, '', ADULT_BOUNDS, aug_s, str_s, quiet=True)
        if mc is None or ms is None:
            continue
        oc = pd.DataFrame({'key': mc['keys'], 'oof_c': mc['oof']}); os_ = pd.DataFrame({'key': ms['keys'], 'oof_s': ms['oof']})
        Jt = oc.merge(os_, on='key').merge(J[['key', 'Hb', 'thr', 'grp']], on='key')
        if len(Jt) < 30:
            continue
        stack = LinearRegression().fit(Jt[['oof_c', 'oof_s']].values, Jt.Hb.values)
        d_floor, _ = _choose_calibration(Jt.Hb.values, stack.predict(Jt[['oof_c', 'oof_s']].values), Jt.thr.values,
                                         ADULT_BOUNDS, _groups(Jt))
        pj = stack.predict(Jt[['oof_c', 'oof_s']].values)
        cal = (_calibration(Jt.Hb.values, pj, Jt.thr.values, ADULT_BOUNDS, d_floor) if d_floor is not None
               else _identity_cal(Jt.Hb.values, pj, ADULT_BOUNDS))
        Ate = Aa.set_index(Aa.key.astype(str)).reindex(kte).reset_index(drop=True)
        Ste = S.set_index(S.key.astype(str)).reindex(kte).reset_index(drop=True)
        hc, _, _, okc = _apply_model({**mc, 'cal': _identity_cal(Jt.Hb.values, pj, ADULT_BOUNDS)}, Ate)
        hs, _, _, oks = _apply_model({**ms, 'cal': _identity_cal(Jt.Hb.values, pj, ADULT_BOUNDS)}, Ste)
        ok = okc & oks
        raw = stack.predict(np.column_stack([hc, hs])[ok]) if ok.any() else np.array([])
        if ok.any():
            idx = te[ok]
            hb[idx], p[idx] = apply_cal(cal, raw, J.thr.values[idx])
            g[idx] = [grade_cal(cal, h, t, ADULT_BOUNDS) for h, t in zip(hb[idx], J.thr.values[idx])]
            ident[idx] = bool(cal.get('identity'))
    covered = ~np.isnan(hb)
    shown, border = _shown_anemia(g, np.nan_to_num(p), ident)
    return _metrics(J.Hb.values, J.thr.values, ADULT_BOUNDS, hb, p, g, shown, border, covered)


def _with_meta(df, meta, rename):
    return df.rename(columns=rename).merge(meta, on='key')

def train_all():
    print(f'🎓 Подготовка данных и обучение (версия {VERSION}).')
    eda = load_eda()
    cp = load_cp()
    fseg = _extract(_conj_one, [(k, s) for k, s in zip(eda.key, eda.segment)], F_SEG, 'Конъюнктива, разметка врачей')
    fauto = _extract(_eye_one, [(k, o, s) for k, o, s in zip(eda.key, eda.original, eda.segment)], F_AUTO,
                     'Глаз целиком (тот же путь, что для ваших фото)')
    fcp = _extract(_conj_one, [(k, s) for k, s in zip(cp.key, cp.segment)], F_CP, 'Конъюнктива, дети') if cp is not None else None
    n_rule = int(fauto.filter(like='c_').notna().any(axis=1).sum()) if len(fauto) else 0
    seg, oof_masks = train_conj_segmenter(list(fauto.key.astype(str)))
    fauto, masks = auto_conj_features(fauto, seg, oof_masks)
    meta = eda[['key', 'Hb', 'sex', 'thr']].copy()
    meta['key'] = meta.key.astype(str)
    meta['sex_male'] = meta.sex.map({'M': 1.0, 'F': 0.0}).fillna(0.5)
    meta['grp'] = meta.key                     # в Eyes-defy одно фото = один пациент
    loc = _local_table()
    if loc is not None and len(loc):           # ИСПРАВЛЕНО (C): фото одного человека — одна группа
        loc['grp'] = loc['person'].astype(str) if 'person' in loc.columns else loc['key'].astype(str)
    st = {'version': VERSION, 'sklearn': sklearn.__version__, 'n_local': 0 if loc is None else int(len(loc)),
          'conj_segmenter': seg}

    # --- HHR: порог оттенка, лучше всего связанный с Hb. ИСПРАВЛЕНО (A): раньше выбирался по ВСЕМ пациентам
    # до перекрёстной проверки (утечка). Теперь _conj_sets/_crop_sets выбирают его по обучающей части.
    Aseg = fseg.merge(meta, on='key')
    hhr = _pick_hhr(Aseg)                      # только для показа; модели выбирают свой порог внутри фолдов
    st['hhr'] = hhr
    conj_sets, crop_sets = _conj_sets, _crop_sets
    scl_sets = {'все': SCL_BASE + ['sex_male'],
                'без абсолютного цвета': ['scl_blue_rel', 'scl_blue_ratio', 'vessel_density', 'vessel_contrast',
                                          'vessel_redness', 'sex_male']}

    # --- «другой свет, другая камера» для каждого снимка датасета
    keys_eye = list(fauto.key.astype(str))
    aug_eye = _aug_table(_aug_eye_one, [(k, masks.get(k)) for k in keys_eye], 'Глаз целиком', seg=seg)
    aug_eye['key'] = aug_eye.key.astype(str)
    cols_c = {c: c[2:] for c in fauto.columns if c.startswith('c_')}
    cols_c.update({c: c[2:] for c in aug_eye.columns if c.startswith('c_')})
    cols_s = {c: c[2:] for c in list(fauto.columns) + list(aug_eye.columns) if c.startswith('s_')}

    # --- 1) конъюнктива по фото глаза (основной метод; тот же путь, что для пользователя)
    Aa = _with_meta(fauto, meta, cols_c)
    good_det = Aa['overlap'].fillna(1.0) >= 0.3 if 'overlap' in Aa else pd.Series(True, index=Aa.index)
    found = Aa.dropna(subset=CONJ_BASE)
    st['detector'] = {'found': int(len(found)), 'of': int(len(eda)), 'eye_found': int(len(fauto)), 'conj_rule': n_rule,
                      'method': 'модель по разметке врачей' if (seg and seg['use']) else 'правила',
                      'overlap_median': float(found['overlap'].median()) if 'overlap' in found else None,
                      'iou_median': float(found['iou'].median()) if 'iou' in found else None,
                      # ИСПРАВЛЕНО (D): IoU по ВСЕМ пациентам с разметкой; глаз/конъюнктива не найдены = 0
                      'iou_median_all': float(np.median(np.concatenate([
                          Aa['iou'].dropna().values if 'iou' in Aa else np.array([]),
                          np.zeros(int(eda.segment.notna().sum()) - int(Aa['iou'].notna().sum() if 'iou' in Aa else 0))]))),
                      'good': int((found['overlap'].fillna(1.0) >= 0.3).sum()) if 'overlap' in found else int(len(found))}
    # ИСПРАВЛЕНО (B): промахи автопоиска (по разметке врача) убираются только из ОБУЧЕНИЯ; проверка — на всех
    Aa_all = Aa.dropna(subset=CONJ_BASE).reset_index(drop=True)
    Aa_all_good = (Aa_all['overlap'].fillna(1.0) >= 0.3).values if 'overlap' in Aa_all else np.ones(len(Aa_all), bool)
    Aa = Aa[good_det].reset_index(drop=True)
    if loc is not None and len(loc):                       # ваши фото с известным Hb — вес ×3
        L_ = loc.rename(columns=cols_c).rename(columns=cols_s)
        L_['key'] = L_.key.astype(str)
        L_['sex_male'] = L_.sex.map({'M': 1.0, 'F': 0.0}).fillna(0.5)
        L_['thr'] = [who_cutoff(sx, a, False) for sx, a in zip(L_.sex, L_.age)]
        L_['grp'] = loc['grp'].values
        Aa = pd.concat([Aa, L_], ignore_index=True)
        Aa_all = pd.concat([Aa_all, L_.dropna(subset=CONJ_BASE)], ignore_index=True)
        Aa_all_good = np.concatenate([Aa_all_good, np.ones(int(L_.dropna(subset=CONJ_BASE).shape[0]), bool)])
    A_aug = _with_meta(aug_eye[aug_eye.part == 'aug'], meta, cols_c)
    A_str = _with_meta(aug_eye[aug_eye.part == 'stress'], meta, cols_c)
    st['conj_auto'] = _fit_best(Aa, conj_sets, 'Конъюнктива по фото глаза', ADULT_BOUNDS, A_aug, A_str,
                                baseline='абсолютные')
    # запасная модель без признаков «относительно белка» — если на снимке белка почти не видно
    st['conj_auto_abs'] = None
    if st['conj_auto'] is not None and any(f in REL_FEATS for f in st['conj_auto']['feats']):
        st['conj_auto_abs'] = _fit_best(Aa, _conj_abs_sets, ' запасная (без белка)', ADULT_BOUNDS,
                                        A_aug, A_str)
    if st['conj_auto'] is not None:
        print('   ⏳ Честная оценка (вложенная проверка): конъюнктива по фото глаза…', flush=True)
        st['conj_auto']['eval_nested'] = _nested_eval(Aa_all, conj_sets, '', ADULT_BOUNDS, A_aug, A_str,
                                                      train_mask=Aa_all_good, fallback_sets=_conj_abs_sets)
        print('      ' + _fmt_nested(st['conj_auto']['eval_nested']))
        n_eda_found = int((~Aa_all.key.astype(str).str.startswith('LOCAL_')).sum())
        print(f"      (из {len(eda)} пациентов датасета конъюнктива автопоиском найдена у {n_eda_found}; "
              'ненайденные — это «переснимите», в MAE не входят)')

    # --- 2) конъюнктива по уже вырезанной области (режим «вырезанная конъюнктива», взрослые)
    aug_seg = _aug_table(_aug_seg_one, list(zip(eda.key.astype(str), eda.segment)), 'Вырезанная конъюнктива, взрослые')
    S_aug = aug_seg[aug_seg.part == 'aug'].merge(meta, on='key') if len(aug_seg) else None
    S_str = aug_seg[aug_seg.part == 'stress'].merge(meta, on='key') if len(aug_seg) else None
    st['conj_seg'] = _fit_best(Aseg, crop_sets, 'Конъюнктива вырезанная (взрослые)', ADULT_BOUNDS, S_aug, S_str, False)
    if st['conj_seg'] is not None:
        st['conj_seg']['eval_nested'] = _nested_eval(Aseg, crop_sets, '', ADULT_BOUNDS, S_aug, S_str, False)
        print('      честно: ' + _fmt_nested(st['conj_seg']['eval_nested']))

    # --- 3) склера
    S = _with_meta(fauto, meta, cols_s)
    if loc is not None and len(loc):
        Ls = loc.rename(columns=cols_s).rename(columns=cols_c)
        Ls['key'] = Ls.key.astype(str)
        Ls['sex_male'] = Ls.sex.map({'M': 1.0, 'F': 0.0}).fillna(0.5)
        Ls['thr'] = [who_cutoff(sx, a, False) for sx, a in zip(Ls.sex, Ls.age)]
        Ls['grp'] = loc['grp'].values
        S = pd.concat([S, Ls], ignore_index=True)
    Sc_aug = _with_meta(aug_eye[aug_eye.part == 'aug'], meta, cols_s)
    Sc_str = _with_meta(aug_eye[aug_eye.part == 'stress'], meta, cols_s)
    st['sclera'] = _fit_best(S, scl_sets, 'Склера', ADULT_BOUNDS, Sc_aug, Sc_str, baseline='все')
    if st['sclera'] is not None:
        S_eval = S.dropna(subset=['scl_blue_rel', 'scl_blue_ratio', 'vessel_density']).reset_index(drop=True)
        st['sclera']['eval_nested'] = _nested_eval(S_eval, scl_sets, '', ADULT_BOUNDS, Sc_aug, Sc_str)
        print('      честно: ' + _fmt_nested(st['sclera']['eval_nested']))
    S_ok = S.dropna(subset=SCL_BASE)
    st['scl_blue_p10'] = float(np.percentile(S_ok.scl_blue_rel, 10)) if len(S_ok) else None

    # --- 4) оба метода: оптимальное объединение (стекинг) + своя калибровка
    st['comb'] = None
    if st['conj_auto'] is not None and st['sclera'] is not None:
        oc = pd.DataFrame({'key': st['conj_auto']['keys'], 'oof_c': st['conj_auto']['oof']})
        os_ = pd.DataFrame({'key': st['sclera']['keys'], 'oof_s': st['sclera']['oof']})
        J = oc.merge(os_, on='key').merge(pd.concat([meta[['key', 'Hb', 'thr', 'grp']],
                                                     Aa[['key', 'Hb', 'thr', 'grp']]]).drop_duplicates('key'), on='key')
        if len(J) >= 40:
            Xj = J[['oof_c', 'oof_s']].values
            oof_j = np.zeros(len(J))
            for tr, te in GroupKFold(n_splits=5).split(Xj, J.Hb, _groups(J)):
                oof_j[te] = LinearRegression().fit(Xj[tr], J.Hb.values[tr]).predict(Xj[te])
            stack = LinearRegression().fit(Xj, J.Hb.values)
            d_floor, ev = _choose_calibration(J.Hb.values, oof_j, J.thr.values, ADULT_BOUNDS, _groups(J))
            cal = (_calibration(J.Hb.values, oof_j, J.thr.values, ADULT_BOUNDS, d_floor) if d_floor is not None
                   else _identity_cal(J.Hb.values, oof_j, ADULT_BOUNDS))
            cal['sigma_sel'] = cal['sigma']
            st['comb'] = {'stack': (float(stack.intercept_), float(stack.coef_[0]), float(stack.coef_[1])),
                          'cal': cal, 'eval': ev, 'oof': oof_j, 'y': J.Hb.values, 'n': int(len(J)),
                          'prev': float(np.mean(J.Hb.values < J.thr.values))}
            print('   ⏳ Честная оценка: оба метода…', flush=True)
            st['comb']['eval_nested'] = _nested_comb(J, Aa_all, S, conj_sets, scl_sets, A_aug, A_str, Sc_aug, Sc_str,
                                                     pd.Series(Aa_all_good, index=Aa_all.index))
            print('      честно: ' + _fmt_nested(st['comb']['eval_nested']))
            print(f"   Оба метода вместе (оптимистично, без вложенной проверки): ошибка Hb ±{ev['calibrated']['MAE'] * 10:.0f} г/л; "
                  f"верная степень {ev['calibrated']['balanced_acc']:.0%}; "
                  f"ложная «анемия» у здоровых {ev['calibrated']['false_anemia_healthy']:.0%}")

    # --- 5) дети до 5 лет (CP-AnemiC, вырезанная конъюнктива)
    st['conj_child'] = None
    if fcp is not None and len(fcp) >= 40:
        C = fcp.merge(cp[['key', 'Hb', 'thr']], on='key').reset_index(drop=True)
        C['sex_male'] = 0.5
        aug_cp = _aug_table(_aug_seg_one, list(zip(cp.key.astype(str), cp.segment)), 'Конъюнктива, дети')
        cpm = cp[['key', 'Hb', 'thr']].assign(sex_male=0.5)
        C_aug = aug_cp[aug_cp.part == 'aug'].merge(cpm, on='key') if len(aug_cp) else None
        C_str = aug_cp[aug_cp.part == 'stress'].merge(cpm, on='key') if len(aug_cp) else None
        child_sets = lambda D: _crop_sets(D, sex=False)
        st['conj_child'] = _fit_best(C, child_sets, 'Конъюнктива, дети до 5 лет', CHILD_BOUNDS, C_aug, C_str, False)
        if st['conj_child'] is not None:
            st['conj_child']['eval_nested'] = _nested_eval(C, child_sets, '', CHILD_BOUNDS, C_aug, C_str, False)
            print('      честно: ' + _fmt_nested(st['conj_child']['eval_nested']))
    st['datasets'] = {'Eyes-defy-anemia': int(len(eda)), 'CP-AnemiC': int(len(fcp)) if fcp is not None else 0,
                      'ваши фото с Hb': st['n_local']}
    joblib.dump(st, STATE_FILE)
    print(f'💾 Модели сохранены: {STATE_FILE}')
    return st

_t0 = time.time()
STATE = None
if os.path.exists(STATE_FILE):
    try:
        STATE = joblib.load(STATE_FILE)
        loc_ = _local_table()
        if STATE.get('version') != VERSION:
            print(f"ℹ️ Файл моделей старой версии ({STATE.get('version')}) — обучаю версию {VERSION} (один раз).")
            STATE = None
            raise StopIteration
        if STATE.get('sklearn') != sklearn.__version__:
            # модели простые — обычно работают и в новой версии scikit-learn; проверяем пробным прогнозом
            try:
                for k in ('conj_auto', 'conj_auto_abs', 'conj_seg', 'sclera', 'conj_child'):
                    if STATE.get(k):
                        STATE[k]['model'].predict(np.array([[np.mean(b) for b in STATE[k]['bounds'].values()]]))
                print(f"ℹ️ Модели сохранены в scikit-learn {STATE.get('sklearn')}, сейчас {sklearn.__version__} — работают.")
                STATE['sklearn'] = sklearn.__version__
            except Exception:
                print('ℹ️ Colab обновил библиотеки — модели нужно переобучить.'); STATE = None
        if STATE is None:
            pass
        elif (0 if loc_ is None else len(loc_)) != STATE.get('n_local', 0):
            print('ℹ️ Добавлены ваши фото с известным Hb — переобучаю с ними (≈5–10 мин).'); STATE = None
    except StopIteration:
        pass
    except Exception as e:
        print('ℹ️ Сохранённые модели не читаются, переобучаю:', str(e)[:80]); STATE = None
if STATE is None:
    STATE = train_all()
    if not globals().get('ON_DRIVE', True):           # режим без Диска: отдаём файл моделей на компьютер
        try:
            from google.colab import files
            files.download(STATE_FILE)
            print(f'💾 Скачан файл {os.path.basename(STATE_FILE)} — сохраните его. В следующий раз выберите его '
                  'в ячейке 1, и обучение не понадобится.')
        except Exception:
            pass
print(f"✅ Модели готовы за {time.time() - _t0:.0f} с. Данные: {STATE['datasets']}")
CONJ_SEG = STATE.get('conj_segmenter')               # поиск конъюнктивы для ячеек 4 и 7
det = STATE.get('detector')
if det:
    print(f"   Фото датасета: глаз найден у {det.get('eye_found', det['found'])}/{det['of']}, конъюнктива — у {det['found']} "
          f"(поиск: {det.get('method', 'правила')}); доля найденной области внутри разметки врача (медиана) "
          f"{det['overlap_median'] or 0:.0%}, IoU среди найденных {det.get('iou_median') or 0:.0%}, "
          f"IoU по всем (ненайденные = 0) {det.get('iou_median_all', float('nan')):.0%}")
if CONJ_SEG:
    print(f"   Совпадение с врачами на невиденных фото (IoU, медиана, ненайденная = 0): модель {CONJ_SEG['iou_learned']:.0%} "
          f"против правил {CONJ_SEG['iou_rule']:.0%}; найдено {CONJ_SEG.get('found_learned', 0):.0%} / "
          f"{CONJ_SEG.get('found_rule', 0):.0%} → {'используется модель' if CONJ_SEG['use'] else 'используются правила'}")
for k, name in (('comb', 'Оба метода'), ('conj_auto', 'Конъюнктива'), ('sclera', 'Склера'),
                ('conj_seg', 'Вырезанная'), ('conj_child', 'Дети')):
    m = STATE.get(k)
    if m and m.get('eval_nested'):
        print(f"   {name:11s} (честно, вложенная проверка): " + _fmt_nested(m['eval_nested'])
              + (f"; признаки «{m['feature_set']}»" if m.get('feature_set') else ''))
    elif m:
        e = m['eval']
        print(f"   {name:11s}: ошибка Hb ±{e['calibrated']['MAE'] * 10:.0f} г/л (оптимистично: без вложенной проверки)")


## 📷 Анализ фото
Пол и возраст — в полях справа от кода. Можно выбрать 1–4 фото. Результат: короткая карточка + кнопка «Подробный анализ»; тот же итог сохраняется в `results/result.json` (для интерфейса приложения).

In [ ]:
#@title 📷 ЯЧЕЙКА 4 — АНАЛИЗ ФОТО ГЛАЗА (заполните поля справа и запустите) { display-mode: "form" }
Пол = 'Женский'  #@param ["Женский", "Мужской"]
Возраст = 25  #@param {type:"integer"}
Беременность = False  #@param {type:"boolean"}
#@markdown Имя или код человека — чтобы персональная калибровка (ячейка 5) не переносилась на других людей:
Кто = 'я'  #@param {type:"string"}
# Можно выбрать 1–4 фото (лучше 2–3 снимка оттянутого нижнего века): результаты усредняются.

from scipy.stats import norm
assert 'STATE' in globals(), 'Сначала выполните ячейки 1–3 («Среда выполнения → Выполнить все»)'
SEX = 'M' if Пол == 'Мужской' else 'F'
AGE, PREGNANT = int(Возраст), bool(Беременность)
PERSON = (Кто or 'я').strip().lower()
THR = who_cutoff(SEX, AGE, PREGNANT)
CHILD = AGE < 5 and STATE.get('conj_child') is not None
BOUNDS = (10.0, 7.0) if (PREGNANT or AGE < 5) else (11.0, 8.0)     # ВОЗ: границы лёгкой и умеренной анемии
# Модель конъюнктивы: для фото глаза — обученная тем же путём (поиск глаза), для вырезанной — по разметке
CONJ_EYE_KEY = 'conj_child' if CHILD else ('conj_auto' if STATE.get('conj_auto') else 'conj_seg')
CONJ_CROP_KEY = 'conj_child' if CHILD else 'conj_seg'
SCL = None if CHILD else STATE.get('sclera')            # модель склеры обучена на взрослых
CONJ = STATE[CONJ_EYE_KEY]
NOTES = []                                               # честные оговорки о применимости моделей
if AGE < 5 and not CHILD:
    NOTES.append('модель для детей до 5 лет не обучена (нет архива CP-AnemiC) — используется взрослая, точность ниже')
elif 5 <= AGE < 15:
    NOTES.append('для детей 5–14 лет отдельной модели нет — используется взрослая; порог ВОЗ взят для вашего возраста')
if AGE < 1:
    NOTES.append('для детей младше 6 месяцев порога ВОЗ нет; использован порог 6–59 мес. (110 г/л) — ориентировочно')
if PREGNANT:
    NOTES.append('беременных в обучающих данных нет — порог ВОЗ 110 г/л учтён, но точность модели не проверена')

try:
    from google.colab import files
    print('Выберите 1–4 фото глаза с оттянутым нижним веком:')
    PHOTOS = list(files.upload().keys())
except ImportError:
    PHOTOS = ['test_eye.jpg']

def predict_hb(m, fdict):
    """Прогноз Hb; признаки ограничены диапазоном обучающих данных (защита от абсурдных значений).
    Возвращает (Hb г/дл, доля СИЛЬНО нетипичных признаков, доля признаков вне диапазона обучения)."""
    x, odd, out = [], 0, 0
    for k in m['feats']:
        lo, hi = m['bounds'][k]
        v, pad = float(fdict[k]), 0.5 * (hi - lo)
        odd += (v < lo - pad) or (v > hi + pad)
        out += (v < lo) or (v > hi)
        x.append(min(max(v, lo), hi))
    a, b = m['hb_range']
    n = len(m['feats'])
    return float(np.clip(m['model'].predict([x])[0], a - 0.5, b + 0.5)), odd / n, out / n

def _conj_estimate(r, rgb, mask, key=None, sclera=None):
    """Оценка по конъюнктиве (сырой выход модели; калибровка — при объединении). True — если получилось.
    На фото глаза добавляются признаки «относительно белка»; если белка почти не видно, а модель
    их требует, — берётся запасная модель без них."""
    key = key or CONJ_EYE_KEY
    fc = eye_conj_features(rgb, mask, sclera) if sclera is not None else compute_indices(rgb, mask)
    if fc is None:
        return False
    if any(f not in fc for f in STATE[key]['feats'] if f != 'sex_male'):
        if key == 'conj_auto' and STATE.get('conj_auto_abs'):
            key = 'conj_auto_abs'
        else:
            return False
    fc['sex_male'] = 1.0 if SEX == 'M' else 0.0
    r['hb_conj'], odd, r['atyp_conj'] = predict_hb(STATE[key], fc)
    r['conj_key'] = key
    r['fc'] = fc
    if odd > 0.3:
        r['warns'].append('цвет конъюнктивы нетипичен для обучающих снимков — точность ниже')
    return True

def analyze_photo(path):
    """Три режима: 1) вырезанная конъюнктива -> только конъюнктива;
    2) глаз, где видна только склера -> только склера; 3) видны обе -> обе (общий итог)."""
    r = {'file': os.path.basename(path), 'ok': False, 'problems': [], 'warns': [], 'mode': None}
    # ---- режим 1: файл — уже вырезанная конъюнктива (прозрачный/чёрный фон)
    seg = conj_crop_from_file(path)
    if seg is not None:
        rgb_s, mask_s, how = seg
        r.update(crop=rgb_s, crop_mask=mask_s, source=how)
        if _conj_estimate(r, rgb_s, mask_s, CONJ_CROP_KEY):
            r.update(ok=True, mode='конъюнктива')
        else:
            r['problems'].append('на вырезанной конъюнктиве слишком мало чистой ткани')
        return r
    try:
        rgb = load_image(path, 2400)
    except ValueError as e:
        r['problems'].append(str(e)); return r
    # Лист белой бумаги в кадре (необязательно) — самый точный эталон цвета света и камеры
    paper = white_reference_gains(rgb)
    eye, source, _ = best_eye(rgb, gains=paper, require_conj=False)
    if eye is None and paper is not None:              # «лист» оказался не листом — ищем как обычно
        paper = None
        eye, source, _ = best_eye(rgb, require_conj=False)
    if eye is None:
        crop = conj_tissue_crop(rgb)                          # крупный кадр одной конъюнктивы без фона
        if crop is not None:
            rgb_s, mask_s, how = crop
            r.update(crop=rgb_s, crop_mask=mask_s, source=how)
            if _conj_estimate(r, rgb_s, mask_s, CONJ_CROP_KEY):
                r.update(ok=True, mode='конъюнктива')
                r['warns'].append('фото распознано как вырезанная конъюнктива (глаза в кадре нет)')
                return r
        v = rgb.mean()
        r['problems'].append('слишком темно — снимите при хорошем освещении' if v < 0.1 else
                             'пересвет — уберите вспышку' if v > 0.93 else
                             'глаз не найден — снимите глаз крупнее, чтобы был виден белок или оттянутое веко')
        return r
    r.update(rgb=rgb, eye=eye, source=source)
    if eye.get('exposure_gain', 1.0) > 2.0:
        r['warns'].append('снимок тёмный — яркость выровнена программно, точность ниже')
    if paper is not None:
        r['info'] = 'найден белый лист — цвет света выровнен по нему'
    if eye.get('wb_fallback') is not None:
        r['warns'].append('сильный цветной свет — цвет выровнен программно по коже, точность ниже')
    # ---- конъюнктива (если веко оттянуто)
    conj_ok = False
    if eye['conj'] is not None:
        p_c, w_c, _ = quality_check(eye)
        r['warns'] += w_c
        if not p_c:
            conj_ok = _conj_estimate(r, eye['crop'], eye['conj'], sclera=eye.get('sclera'))
        else:
            r['conj_problems'] = p_c
    # ---- склера (если видна; модель склеры — для взрослых)
    scl_ok = False
    if SCL is not None and not quality_check_sclera(eye):
        fs, vessel, bg = sclera_features(eye['crop'], eye['sclera'], eye['iris_crop'])
        if fs is not None:
            fs['sex_male'] = 1.0 if SEX == 'M' else 0.0
            hb_s, odd_s, atyp_s = predict_hb(SCL, fs)
            r.update(fs=fs, vessel=vessel, scl_bg=bg)
            if odd_s <= 0.3:
                r['hb_scl'], r['atyp_scl'], scl_ok = hb_s, atyp_s, True
            else:
                r['warns'].append('склера нетипична для обучающих снимков — метод «склера» не учтён')
    if conj_ok and scl_ok:
        r['mode'] = 'конъюнктива + склера'
    elif conj_ok:
        r['mode'] = 'конъюнктива'
    elif scl_ok:
        r['mode'] = 'склера'
        if eye['conj'] is None:
            r['warns'].append('веко не оттянуто — оценка только по склере (точнее, если оттянуть нижнее веко)')
    else:
        r['problems'] += r.get('conj_problems', []) or quality_check_sclera(eye) or \
            (['оценка по склере для детей до 5 лет недоступна — оттяните нижнее веко'] if SCL is None else
             ['не удалось оценить ни конъюнктиву, ни склеру — переснимите крупнее'])
        return r
    r['ok'] = True
    return r

RES = [analyze_photo(p) for p in PHOTOS]
good = [r for r in RES if r['ok']]
RESULT = None

# ---------------- вывод: сначала коротко и понятно, подробности — по кнопке
try:
    import ipywidgets as W
    from IPython.display import display, HTML
    _WIDGETS = True
except ImportError:
    _WIDGETS = False

def _card(color, title, lines):
    body = ''.join(f'<div style="margin-top:6px;font-size:16px">{l}</div>' for l in lines)
    return (f'<div style="border-left:8px solid {color};background:#fafafa;padding:14px 18px;'
            f'border-radius:8px;max-width:760px;font-family:sans-serif">'
            f'<div style="font-size:22px;font-weight:bold;color:{color}">{title}</div>{body}</div>')

def _show(summary_html, summary_text, details_fn):
    """Короткая карточка + кнопка «Подробный анализ» (в Colab); без виджетов — всё текстом."""
    if _WIDGETS:
        display(HTML(summary_html))
        if details_fn is None:
            return
        out = W.Output(layout=W.Layout(display='none'))
        btn = W.Button(description='🔍 Подробный анализ', button_style='info',
                       layout=W.Layout(width='280px', height='42px'))
        with out:
            details_fn()
        def _toggle(_):
            hidden = out.layout.display == 'none'
            out.layout.display = None if hidden else 'none'
            btn.description = '▲ Скрыть подробности' if hidden else '🔍 Подробный анализ'
        btn.on_click(_toggle)
        display(btn, out)
    else:
        print(summary_text)
        if details_fn is not None:
            print('\n----- ПОДРОБНЫЙ АНАЛИЗ -----')
            details_fn()

if not PHOTOS:
    _show(_card('#757575', 'Фото не выбрано', ['Запустите ячейку ещё раз и выберите фото глаза.']),
          'Фото не выбрано — запустите ячейку ещё раз.', None)
elif not good:
    reason = RES[0]['problems'][0] if RES[0]['problems'] else 'фото не подходит для анализа'
    _show(_card('#c62828', '📸 Нужно переснять фото', [reason[0].upper() + reason[1:] + '.',
                'Снимите глаз крупно: нижнее веко оттянуто, дневной свет, без вспышки.']),
          '📸 Нужно переснять фото: ' + reason, None)
else:
    # ---- 1) сырые оценки по фото -> медиана; 2) калибровка каждого метода (снимает сжатие к среднему)
    # Калибровка «растягивает» прогноз, чтобы тяжёлые случаи не сжимались к среднему. Но для снимков,
    # нетипичных для обучающих (другая камера/свет), растяжка УСИЛИВАЕТ сдвиг цвета и превращает всех
    # в «умеренную/тяжёлую». Поэтому растягиваем частично и только для типичных снимков.
    def _hb_cal(cal, raw, atyp=0.0):
        # ИСПРАВЛЕНО (аудит): было lam = 0.5 — «половина» калибровки, которую НИКТО не проверял (точность считалась
        # для полной калибровки). Теперь для типичных снимков — ровно то, что проверено; для нетипичных — без растяжки.
        full = (raw - cal['c']) / cal['d']
        lam = 1.0 if atyp <= 0.15 else 0.0
        return raw + lam * (full - raw)
    def _prob(cal, hb):
        z = hb - THR
        if 'p_a' in cal and not cal.get('identity'):
            return float(1 / (1 + np.exp(-(cal['p_a'] + cal['p_b'] * z))))
        return float(norm.cdf(-z / max(cal['sigma'], 0.8)))
    keys = [r['conj_key'] for r in good if 'hb_conj' in r]
    keyC = max(set(keys), key=keys.count) if keys else None
    rc = [r['hb_conj'] for r in good if 'hb_conj' in r and r.get('conj_key') == keyC]
    rs = [r['hb_scl'] for r in good if 'hb_scl' in r]
    raw_c = float(np.median(rc)) if rc else None
    raw_s = float(np.median(rs)) if rs else None
    at_c = float(np.median([r.get('atyp_conj', 0) for r in good if 'hb_conj' in r and r.get('conj_key') == keyC])) if rc else 1.0
    at_s = float(np.median([r.get('atyp_scl', 0) for r in good if 'hb_scl' in r])) if rs else 1.0
    hb_c = _hb_cal(STATE[keyC]['cal'], raw_c, at_c) if raw_c is not None else None
    hb_s = _hb_cal(SCL['cal'], raw_s, at_s) if raw_s is not None else None
    n_warn = len({w for r in good for w in r['warns']})
    infl = min(1.0 + 0.1 * n_warn, 1.4)                      # предупреждения -> шире интервал
    # ---- ВЫБОР МЕТОДА. У каждой оценки своя ожидаемая ошибка: ошибка метода на проверке (пациенты,
    # которых модель не видела, при обычном и при другом свете) × поправка за качество ЭТОГО снимка
    # (предупреждения, непохожесть на обучающие). Если конъюнктива и склера согласуются в пределах
    # своих ошибок — объединяем (так точнее любой одной). Если расходятся — верим той, чья ошибка меньше.
    def _sig(cal, atyp, n_w):
        return max(cal.get('sigma_sel', cal['sigma']) * (1 + 0.1 * n_w) * (1 + 1.5 * atyp), 0.5)
    w_c = len({w for r in good if 'hb_conj' in r for w in r['warns']})
    w_s = len({w for r in good if 'hb_scl' in r for w in r['warns']})
    sig_c = _sig(STATE[keyC]['cal'], at_c, w_c) if hb_c is not None else None
    sig_s = _sig(SCL['cal'], at_s, w_s) if hb_s is not None else None
    disagree = False
    if hb_c is not None and hb_s is not None:
        tol = max(1.0, float(np.hypot(sig_c, sig_s)))        # г/дл: «похожие значения» с учётом ошибок
        disagree = abs(hb_c - hb_s) > tol
        if not disagree:
            if STATE.get('comb') and keyC == 'conj_auto':
                b0, b1, b2 = STATE['comb']['stack']
                CAL = STATE['comb']['cal']
                hb = _hb_cal(CAL, b0 + b1 * raw_c + b2 * raw_s, max(at_c, at_s))
            else:                                            # взвешивание по точности каждого метода
                wc_, ws_ = 1 / sig_c ** 2, 1 / sig_s ** 2
                hb = (wc_ * hb_c + ws_ * hb_s) / (wc_ + ws_)
                CAL = dict(STATE[keyC]['cal'] if sig_c <= sig_s else SCL['cal'])
                CAL['sigma'] = float(1 / np.sqrt(wc_ + ws_))
            method = 'конъюнктива + склера'
            WHY = (f'конъюнктива ({hb_c * 10:.0f} г/л) и склера ({hb_s * 10:.0f} г/л) согласуются — '
                   'взят общий результат')
        else:
            use_c = sig_c <= sig_s
            CAL, hb, method = (STATE[keyC]['cal'], hb_c, 'конъюнктива') if use_c else (SCL['cal'], hb_s, 'склера')
            WHY = (f'конъюнктива ({hb_c * 10:.0f} г/л) и склера ({hb_s * 10:.0f} г/л) расходятся на '
                   f'{abs(hb_c - hb_s) * 10:.0f} г/л — выбран метод «{method}»: его ожидаемая ошибка для этого снимка '
                   f'±{min(sig_c, sig_s) * 10:.0f} г/л против ±{max(sig_c, sig_s) * 10:.0f}')
    elif hb_c is not None:
        CAL, hb, method = STATE[keyC]['cal'], hb_c, 'конъюнктива'
        WHY = 'склера не оценивалась: ' + ('для детей до 5 лет модели склеры нет' if SCL is None else
                                            'белка видно мало, он нетипичен или фото — вырезанная конъюнктива')
    else:
        CAL, hb, method = SCL['cal'], hb_s, 'склера'
        WHY = 'конъюнктива не оценивалась: нижнее веко не оттянуто или видно слишком мало'
    at_main = at_c if hb_c is not None else at_s
    typical = at_main <= 0.15
    unreliable = at_main > 0.4          # почти половина признаков вне диапазона обучения — выводы по ним ненадёжны
    sig = max(CAL.get('sigma_sel', CAL['sigma']) * infl, 0.8)   # с учётом ошибки при другом свете
    hb_raw = hb
    # Персональная калибровка (ячейка 5): поправка по вашему же анализу крови
    import json
    CAL_FILE = f'{DRIVE_DIR}/personal_calibration.json'
    _pc = json.load(open(CAL_FILE)) if os.path.exists(CAL_FILE) else {}
    if isinstance(_pc, list):                  # старый формат v10: одна калибровка на всех — не применяем к людям
        _pc = {}
    pcal = _pc.get(PERSON, [])                 # ИСПРАВЛЕНО (аудит): поправка только для ЭТОГО человека
    if pcal:
        off = float(np.mean([e['offset'] for e in pcal]))
        hb = hb + off
        method += f' + персональная калибровка ({off * 10:+.0f} г/л, по {len(pcal)} анализ.)'
    hb = float(np.clip(hb, 4.0, 19.0))
    p = _prob(CAL, hb)
    # ---- степень анемии по ОТКАЛИБРОВАННЫМ границам (подобраны по данным, честная перекрёстная проверка)
    shift = -1.0 if (PREGNANT or (AGE < 5 and not CHILD)) else 0.0      # у беременных границы ВОЗ ниже на 1 г/дл
    if hb >= THR + CAL['cut_norm']:
        g = 0
    elif hb >= CAL['cut_mod'] + shift:
        g = 1
    elif hb >= CAL['cut_sev'] + shift:
        g = 2
    else:
        g = 3
    GRADE_COL = ['#2e7d32', '#f9a825', '#ef6c00', '#c62828']
    grade = ['Норма', 'Лёгкая анемия', 'Умеренная анемия', 'Тяжёлая анемия'][g]
    gcol = GRADE_COL[g]
    # «Умеренная/тяжёлая» — только при надёжном сигнале: вероятность ≥ 85%, снимок типичен для
    # обучающих данных и (если есть оба метода) конъюнктива и склера согласны. Иначе — честно
    # «возможна анемия» без уверенной степени: по снимку с другой камеры/светом степень не определить.
    agree = not disagree
    severity_capped = False
    if g >= 2 and not (p >= 0.85 and typical and agree):
        g, severity_capped = 1, True
        grade, gcol = 'Возможна анемия', GRADE_COL[1]
    lo_p, hi_p = (0.3, 0.7) if CAL.get('identity') else (0.35, 0.65)
    # «Пограничный»: вероятность у границы ИЛИ уровень Hb и вероятность противоречат друг другу
    # (Hb в норме, а вероятность анемии высокая — или наоборот). Такое честно не называем ни нормой, ни анемией.
    borderline = g <= 1 and (lo_p < p < hi_p or (g == 0 and p >= hi_p) or (g == 1 and p <= lo_p))
    if unreliable:
        verdict, vcol, zone, emoji = 'Оценка ненадёжна — снимок не похож на обучающие', '#757575', 'НЕ ОПРЕДЕЛЁН', '⚪'
        advice = ('Цвета на этом снимке сильно отличаются от снимков, на которых училась модель (камера, свет, '
                  'фильтр). Переснимите при дневном свете без вспышки и фильтров — или сдайте общий анализ крови.')
    elif borderline:
        verdict, vcol, zone, emoji = 'Пограничный результат', '#f9a825', 'СРЕДНИЙ', '🟡'
        advice = ('По фото нельзя точно отличить норму от лёгкой анемии — сдайте общий анализ крови '
                  '(его можно расшифровать в ячейке 8).')
    elif g == 0:
        verdict, vcol, zone, emoji = 'Норма — анемия маловероятна', '#2e7d32', 'НИЗКИЙ', '🟢'
        advice = 'Признаков анемии по фото нет. Если есть слабость, головокружение или одышка — всё равно сдайте анализ крови.'
    else:
        verdict = ('Возможна анемия — степень по фото не определить' if severity_capped else grade + ' (вероятно)')
        vcol, zone, emoji = gcol, 'ВЫСОКИЙ' if not severity_capped else 'СРЕДНИЙ', '🔴' if not severity_capped else '🟠'
        advice = ('Сдайте общий анализ крови в ближайшее время и покажите результат врачу. '
                  'Когда будет анализ — впишите его в ячейку 8: она подскажет возможную причину.')
    lo, hi = hb - 1.96 * sig, hb + 1.96 * sig
    with_eye = [r for r in good if 'eye' in r]
    best = max(with_eye, key=lambda r: (('hb_conj' in r) + ('hb_scl' in r), r['eye']['open_score'])) if with_eye else good[0]
    fc = next((r['fc'] for r in good if 'fc' in r), None)
    fs_best = best.get('fs')
    flags = []
    if fs_best is not None and STATE.get('scl_blue_p10') is not None and fs_best['scl_blue_rel'] < STATE['scl_blue_p10']:
        flags.append('🔵 Склера голубоватая — по пилотному исследованию это косвенный признак дефицита железа '
                     '(проверяется анализом на ферритин).')
    RESULT = {'hb': hb, 'hb_raw': hb_raw, 'hb_conj': hb_c, 'hb_scl': hb_s, 'sigma': sig, 'p_anemia': p,
              'verdict': verdict, 'zone': zone, 'thr': THR, 'n_photos': len(good), 'reliable': not unreliable,
              'method': method, 'severity_capped': severity_capped, 'person': PERSON}

    METHOD_TXT = {'конъюнктива + склера': 'по конъюнктиве и склере (общий результат)',
                  'конъюнктива': 'только по конъюнктиве', 'склера': 'только по склере'}[method.split(' + персон')[0]]
    HB_TXT = f'≈ {hb * 10:.0f} г/л' + (' (ненадёжно)' if unreliable else '')
    # Честно: насколько этот метод ошибается на проверке (пациенты, которых модель не видела)
    if method.startswith('конъюнктива + склера'):
        _m = STATE['comb'] if (STATE.get('comb') and keyC == 'conj_auto') else (STATE[keyC] if sig_c <= sig_s else SCL)
    else:
        _m = STATE[keyC] if method.startswith('конъюнктива') else SCL
    # ИСПРАВЛЕНО (аудит): показываем честную точность (вложенная проверка), а не ошибку того варианта,
    # который выбран по этой же ошибке
    RESULT['train_prev'] = _m.get('prev')        # доля анемии в обучающих данных этого метода (для ячейки 5)
    _evn = _m.get('eval_nested')
    if _evn:
        MAE_GL, MAE_LIGHT = _evn['MAE'] * 10, (_evn.get('MAE_light') or 0) * 10
    else:
        _ev = _m['eval']
        MAE_GL = _ev['calibrated']['MAE'] * 10
        MAE_LIGHT = (_ev.get('MAE_light') or 0) * 10
    ACC_TXT = (f'Точность метода на проверке: ±{MAE_GL:.0f} г/л'
               + (f' (при другом свете/камере ±{MAE_LIGHT:.0f})' if MAE_LIGHT else '')
               + (' — это низкая точность, результат ориентировочный' if max(MAE_GL, MAE_LIGHT) > 15 else ''))
    # Итог в виде словаря/JSON — то же самое будет показывать интерфейс приложения
    RESULT.update({
        'hb_g_l': round(hb * 10), 'hb_interval_g_l': [round(lo * 10), round(hi * 10)],
        'hb_conj_g_l': None if hb_c is None else round(hb_c * 10), 'hb_sclera_g_l': None if hb_s is None else round(hb_s * 10),
        'threshold_g_l': round(THR * 10), 'grade': grade, 'risk_zone': zone, 'advice': advice, 'emoji': emoji,
        'tissue': METHOD_TXT, 'method_reason': WHY, 'accuracy_g_l': round(MAE_GL), 'typical_photo': bool(typical),
        'conj_region': next((r['eye'].get('conj_method') for r in good if 'eye' in r and 'hb_conj' in r), None),
        'warnings': sorted({w for r in good for w in r['warns']}), 'notes': NOTES,
        'photos_used': len(good), 'photos_total': len(RES)})
    try:
        json.dump({k: v for k, v in RESULT.items() if not isinstance(v, (np.ndarray,))}, open(f'{OUT}/result.json', 'w'),
                  ensure_ascii=False, indent=1, default=float)
    except Exception:
        pass
    summary = _card(vcol, f'{emoji} {verdict}', [
        f'Гемоглобин: <b>{HB_TXT}</b> &nbsp;(норма для вас — от {THR * 10:.0f} г/л)',
        f'Определено {METHOD_TXT}',
        f'<span style="color:#555;font-size:14px">Почему: {WHY}</span>',
        f'<span style="color:#555;font-size:14px">{ACC_TXT}</span>',
        f'👉 {advice}',
        '<span style="color:#777;font-size:13px">Оценка по фото — это скрининг, а не диагноз. '
        'Точный гемоглобин, ферритин и тип анемии — только анализ крови.</span>'])
    summary_txt = (f'{emoji} {verdict}\nГемоглобин: {HB_TXT} (норма для вас — от {THR * 10:.0f} г/л)\n'
                   f'Определено {METHOD_TXT}\nПочему: {WHY}\n{ACC_TXT}\n'
                   f'👉 {advice}\nОценка по фото — скрининг, а не диагноз.')

    def details():
        fig = plt.figure(figsize=(20, 11))
        if 'eye' in best:
            rgb, eye = best['rgb'], best['eye']
            ov = (eye['crop'] * 255).astype(np.uint8).copy()
            if 'scl_bg' in best:
                ov[best['scl_bg']] = (0.45 * ov[best['scl_bg']] + 0.55 * np.array([0, 190, 255])).astype(np.uint8)
                ov[best['vessel']] = np.array([255, 0, 60], np.uint8)
            if eye['conj'] is not None and 'hb_conj' in best:
                ov[eye['conj']] = (0.45 * ov[eye['conj']] + 0.55 * np.array([0, 230, 0])).astype(np.uint8)
            cxc, cyc, rcc = eye['iris_crop']
            if not eye.get('iris_hidden'):                 # радужку рисуем, только если она реально видна
                cv2.circle(ov, (int(cxc), int(cyc)), int(rcc), (255, 220, 0), 2)
            full8 = (rgb * 255).astype(np.uint8).copy()
            x0, y0, x1, y1 = eye['box']
            cv2.rectangle(full8, (x0, y0), (x1, y1), (255, 220, 0), max(2, rgb.shape[1] // 300))
            ax1 = fig.add_subplot(2, 3, 1); ax1.imshow(full8); ax1.set_title(f"Глаз найден ({best['source']})"); ax1.axis('off')
            ax2 = fig.add_subplot(2, 3, 2); ax2.imshow(ov); ax2.axis('off')
            ax2.set_title('Конъюнктива (зелёный) · склера (голубой) · сосуды (красный)\n'
                          + ('радужка закрыта веком — глаз найден по конъюнктиве' if eye.get('iris_hidden')
                             else 'радужка (жёлтый)'))
            ax3 = fig.add_subplot(2, 3, 3); ax3.axis('off')
            if eye['conj'] is not None and 'hb_conj' in best:
                im3 = ax3.imshow(np.where(eye['conj'], erythema_map(eye['crop']), np.nan), cmap='inferno', vmin=0, vmax=0.5)
                ax3.set_title('Карта покраснения конъюнктивы (EI)'); plt.colorbar(im3, ax=ax3, fraction=0.046)
            else:
                ax3.text(0.5, 0.5, 'Конъюнктива не видна —\nоценка только по склере', ha='center', va='center', fontsize=14)
        else:
            crop8 = (best['crop'] * 255).astype(np.uint8)
            ov = crop8.copy(); m_ = best['crop_mask']
            ov[m_] = (0.6 * ov[m_] + 0.4 * np.array([0, 230, 0])).astype(np.uint8)
            ax1 = fig.add_subplot(2, 3, 1); ax1.imshow(crop8); ax1.set_title(best['source']); ax1.axis('off')
            ax2 = fig.add_subplot(2, 3, 2); ax2.imshow(ov); ax2.set_title('Анализируемая ткань (зелёный)'); ax2.axis('off')
            ax3 = fig.add_subplot(2, 3, 3); ax3.axis('off')
            im3 = ax3.imshow(np.where(m_, erythema_map(best['crop']), np.nan), cmap='inferno', vmin=0, vmax=0.5)
            ax3.set_title('Карта покраснения конъюнктивы (EI)'); plt.colorbar(im3, ax=ax3, fraction=0.046)
        ax4 = fig.add_subplot(2, 1, 2)
        mild, sev = BOUNDS
        for a_, b_, c_, t_ in [(4, sev, '#c62828', 'тяжёлая'), (sev, mild, '#ef6c00', 'умеренная'),
                               (mild, THR, '#f9a825', 'лёгкая'), (THR, 18, '#2e7d32', 'норма')]:
            if b_ > a_:
                ax4.axvspan(a_ * 10, b_ * 10, color=c_, alpha=0.18)
                ax4.text((a_ + b_) * 5, 3.55, t_, ha='center', fontsize=10)
        rows_ = [('По конъюнктиве', hb_c, STATE[keyC]['cal']['sigma'] * infl, '#1565c0')] if hb_c is not None else []
        if hb_s is not None:
            rows_.append(('По склере', hb_s, SCL['cal']['sigma'] * infl, '#00838f'))
        rows_.append(('ИТОГ', hb, sig, 'k'))
        for i, (name, v_, s_, c_) in enumerate(rows_):
            yv = 2.6 - i * (2.6 / max(len(rows_) - 1, 1)) if len(rows_) > 1 else 1.3
            ax4.errorbar(v_ * 10, yv, xerr=1.96 * s_ * 10, fmt='o', color=c_, capsize=6, lw=2.5, ms=10)
            ax4.text(v_ * 10, yv + 0.3, f'{name}: {v_ * 10:.0f} г/л', ha='center', fontsize=12, color=c_,
                     fontweight='bold' if name == 'ИТОГ' else 'normal')
        ax4.set_xlim(40, 180); ax4.set_ylim(-0.2, 3.9); ax4.set_yticks([])
        ax4.set_xlabel('Гемоглобин, г/л (отрезки — 95%-й интервал погрешности метода)')
        ax4.set_title(f'Hb ≈ {hb * 10:.0f} г/л  →  {verdict}', color=vcol, fontsize=15, fontweight='bold')
        plt.tight_layout(); plt.savefig(f'{OUT}/my_result.png', dpi=130, bbox_inches='tight'); plt.show()

        print('ПОДРОБНЫЙ АНАЛИЗ')
        print('=' * 80)
        print(f'Гемоглобин (итог):      ≈ {hb * 10:.0f} г/л, 95%-й интервал {lo * 10:.0f}–{hi * 10:.0f} г/л')
        print(f'   по конъюнктиве:      ' + (f'{hb_c * 10:.0f} г/л' if hb_c is not None else 'не оценивалась (не видна)'))
        print(f'   по склере:           ' + (f'{hb_s * 10:.0f} г/л' if hb_s is not None else 'не оценивался'))
        print(f'   метод итога:         {method} — {WHY}')
        if sig_c is not None or sig_s is not None:
            print('   ожидаемая ошибка:    ' + ' · '.join(t for t in (
                f'конъюнктива ±{sig_c * 10:.0f} г/л' if sig_c is not None else '',
                f'склера ±{sig_s * 10:.0f} г/л' if sig_s is not None else '') if t))
        for r in good:
            if r.get('info'):
                print(f"   {r['file']}: {r['info']}")
        print(f'Порог анемии (ВОЗ):     {THR * 10:.0f} г/л   ({"дети до 5 лет" if CHILD else "беременные" if PREGNANT else "мужчины" if SEX == "M" else "женщины"})')
        print(f'Вероятность анемии:     {p:.0%}  →  риск {zone}')
        print(f'Степень (калиброванные границы): {grade}')
        at_show = at_c if hb_c is not None else at_s
        print(f"Похожесть снимка на обучающие: {'да' if typical else 'НЕТ'} ({at_show:.0%} признаков вне диапазона обучения"
              + ('; > 40% — вывод не делается' if unreliable else '') + ')')
        if severity_capped:
            why = [w for w, f in (('вероятность < 85%', p < 0.85), ('снимок нетипичен (другая камера/свет)', not typical),
                                  ('конъюнктива и склера расходятся', not agree)) if f]
            print('   Степень не уточняется, потому что: ' + '; '.join(why) + '. Точную степень покажет анализ крови.')
        if fc is not None:
            print(f"Индексы конъюнктивы:    EI {fc['EI']:.3f} · a* {fc['a_star']:.1f} · L* {fc['L_star']:.1f} · "
                  f"{STATE['hhr']} {fc[STATE['hhr']]:.3f} · PVM_R {fc['PVM_R']:.0f}")
        if fs_best is not None:
            f_ = fs_best
            print(f"Индексы склеры:         голубизна {f_['scl_blue_rel']:+.1f} · сосуды {f_['vessel_density']:.1%} · "
                  f"краснота сосудов {f_['vessel_redness']:+.1f}")
        print(f'Фото:                   пригодных {len(good)} из {len(RES)}; модель «{"дети до 5 лет" if CHILD else "взрослые"}»')
        for r in RES:
            if r['problems']:
                print(f"   ❌ {r['file']}: " + '; '.join(r['problems']))
            elif r['warns']:
                print(f"   ⚠️ {r['file']}: " + '; '.join(r['warns']))
        for fl in flags:
            print(fl)
        if 'eye' in best and 'hb_conj' in best:
            e_ = best['eye']
            segm = STATE.get('conj_segmenter')
            how = e_.get('conj_method', 'правила (цвет и геометрия)')
            if segm and segm.get('use') and how.startswith('модель'):
                how += f" (совпадение с врачами на невиденных фото {segm['iou_learned']:.0%}, у правил {segm['iou_rule']:.0%})"
            print(f'Область конъюнктивы:    найдена — {how}; площадь {e_["conj"].sum() / e_["iris_crop"][2] ** 2:.1f} r²')
        for nt in NOTES:
            print('ℹ️ ' + nt)
        print('Ферритин, MCV, B12 и др. по фото глаза НЕ определяются: Hb из анализа — в ячейку 5, '
              'полный анализ (ферритин, MCV, B12…) — в ячейку 8.')

    _show(summary, summary_txt, details)


## Необязательно: уточнение и отчёт о точности

In [ ]:
#@title ЯЧЕЙКА 5 — (необязательно) АНАЛИЗ КРОВИ и осмотр: точная интерпретация Hb, ферритина, MCV и др.
# Анализ крови — самый точный источник и имеет приоритет над фото.
SIGNS = {                       # 1 = есть. Лучше, если оценивает другой человек при дневном свете
    'palmar_crease_pallor': 0,  # складки ладони бледные при разогнутых пальцах
    'palmar_pallor': 0,         # ладони бледные
}
# Результаты общего анализа крови (если есть). Неизвестное оставьте None.
LABS = dict(Hb=None,        # как в бланке: 118 (г/л) или 11.8 (г/дл) — единицы определяются автоматически
            MCV=None, RBC=None, RDW=None, ferritin=None, CRP=None, B12=None, folate=None)

SAVE_AS_CALIBRATION = True   # сохранить разницу «анализ − фото», чтобы следующие оценки по фото были точнее
RESET_CALIBRATION = False    # True — удалить сохранённую персональную калибровку

import json
from scipy.stats import norm
CAL_FILE = f'{DRIVE_DIR}/personal_calibration.json'
PERSON = globals().get('PERSON', 'я')
_pc = json.load(open(CAL_FILE)) if os.path.exists(CAL_FILE) else {}
_pc = {} if isinstance(_pc, list) else _pc          # старый формат v10 (общая на всех) — сбрасывается
if RESET_CALIBRATION and PERSON in _pc:
    _pc.pop(PERSON); json.dump(_pc, open(CAL_FILE, 'w')); print(f'🗑 Персональная калибровка «{PERSON}» удалена.')
# ИСПРАВЛЕНО (аудит): Hb в г/л (как в бланке и в ячейке 8) или в г/дл. Раньше ввод «118» читался как 118 г/дл
if LABS['Hb'] is not None and LABS['Hb'] > 25:
    LABS['Hb'] = LABS['Hb'] / 10.0
print('=' * 78)
if LABS['Hb'] is not None:
    grade, lines = interpret_labs(SEX, AGE, PREGNANT, **{k: v for k, v in LABS.items() if v is not None})
    print('🩸 ЗАКЛЮЧЕНИЕ ПО АНАЛИЗУ КРОВИ (самый точный источник):')
    for l in lines:
        print('   •', l)
    print('   ➜ Подробный профиль возможной причины (железо / B12 / воспаление / талассемия) — в ячейке 8.')
    if globals().get('RESULT'):
        diff = RESULT['hb'] - LABS['Hb']
        print(f"\n📷 Оценка по фото была {RESULT['hb'] * 10:.0f} г/л, анализ — {LABS['Hb'] * 10:.0f} г/л "
              f"(разница {diff * 10:+.0f} г/л; типичная погрешность метода ±{RESULT['sigma'] * 10:.0f} г/л)")
        if SAVE_AS_CALIBRATION:
            cal = _pc.get(PERSON, [])
            cal.append({'offset': float(LABS['Hb'] - RESULT['hb_raw']), 'date': time.strftime('%Y-%m-%d')})
            _pc[PERSON] = cal[-5:]                           # храним 5 последних, ОТДЕЛЬНО для каждого человека
            json.dump(_pc, open(CAL_FILE, 'w'))
            off = np.mean([e['offset'] for e in cal[-5:]])
            print(f'💾 Персональная калибровка «{PERSON}» сохранена: следующие оценки по фото будут скорректированы на '
                  f'{off * 10:+.0f} г/л. Она верна для ЭТОГО телефона и освещения. Анализ сдан давно (> 2 недель)? '
                  'Тогда лучше не сохранять.')
elif globals().get('RESULT') and not RESULT.get('reliable', True):
    print('📷 Снимок в ячейке 4 не похож на обучающие — вероятность по фото не считается (было бы неправдой).\n'
          '   Переснимите при дневном свете без вспышки или впишите анализ крови выше.')
elif globals().get('RESULT'):
    # ИСПРАВЛЕНО (аудит): 1) вероятность по фото — это апостериорная вероятность при доле анемии в ОБУЧАЮЩЕМ
    # датасете; её переводим в отношение правдоподобия и применяем к априорной вероятности для группы человека
    # (PRIORS, ВОЗ 2019). 2) Бледность ладоней и складок ладони — один и тот же признак бледности, они не независимы:
    # берётся только самый сильный LR (раньше LR перемножались).
    p = float(np.clip(RESULT['p_anemia'], 0.01, 0.99))
    prev = RESULT.get('train_prev')
    grp_ = ('pregnant' if PREGNANT else 'child_6_59m' if AGE < 5 else
            'woman_15_49' if (SEX == 'F' and 15 <= AGE <= 49) else 'other')
    steps = []
    if prev is not None and 0 < prev < 1:
        lr_photo = (p / (1 - p)) / (prev / (1 - prev))
        odds = PRIORS[grp_] / (1 - PRIORS[grp_])
        steps.append((f'Априорно для вашей группы ({grp_}, ВОЗ 2019' + (', допущение' if grp_ == 'other' else '') + ')',
                      odds / (1 + odds)))
        odds *= lr_photo
        steps.append((f"По фото ({RESULT.get('method', 'конъюнктива')}; LR={lr_photo:.2f})", odds / (1 + odds)))
    else:
        odds = p / (1 - p)
        steps.append(('По фото (' + RESULT.get('method', 'конъюнктива') + ')', p))
    present = [SIGN_LR[c] for c, v in SIGNS.items() if v]
    if present:
        lr, name = max(present)
        odds *= lr
        steps.append((f'{name} (LR={lr}; признаки бледности не перемножаются)', odds / (1 + odds)))
    print('   Оговорка: бледность по фото и бледность ладоней частично зависимы — итог может быть завышен.')
    print('📊 ВЕРОЯТНОСТЬ АНЕМИИ (фото + осмотр, теорема Байеса):')
    for name, v in steps:
        print(f'   {v:6.0%}  ←  {name}')
    pf = steps[-1][1]
    print(f"\n   Итог: {pf:.0%} → риск {'НИЗКИЙ' if pf < 0.2 else 'СРЕДНИЙ' if pf < 0.5 else 'ВЫСОКИЙ'}")
else:
    print('Сначала выполните ячейку 4 (фото) или впишите результаты анализа крови.')
print('=' * 78)
print('⚠️ Не является диагнозом. Тип анемии и лечение определяет врач; не принимайте железо без анализа.')


## 🩸 Этап 2: возможная причина анемии
Камера выявляет **признаки** анемии, но **причину** (дефицит железа, B12/фолата, воспаление, талассемия) по фото определить нельзя. Если есть анализ крови — впишите его в ячейку 8: система покажет, на какой профиль он похож и что проверить дальше. Если анализа нет — заполните анкету, система подскажет, какие анализы сдать.

In [ ]:
#@title 🩸 ЯЧЕЙКА 8 — ПРОФИЛЬ ВОЗМОЖНОЙ ПРИЧИНЫ АНЕМИИ (по анализу крови) { display-mode: "form" }
# Этап 2 после фото: камера выявляет ПРИЗНАКИ анемии, а ПРИЧИНУ помогают понять показатели крови.
# Это НЕ диагноз: система показывает, на какой профиль похожи анализы и что проверить дальше.
# Впишите то, что есть в бланке анализа. Неизвестное оставьте 0.
# Источники правил: ВОЗ (пороги Hb 2011 — как в обучающих данных; ферритин 2020); NHLBI — талассемия, B12-дефицит;
# MSD/Merck Manual — оценка анемии (MCV, MCH, RDW, ферритин, B12, фолат); индекс Ментцера (1973).
Пол_ = 'Женский'  #@param ["Женский", "Мужской"]
Возраст_ = 25  #@param {type:"integer"}
Беременность_ = False  #@param {type:"boolean"}
#@markdown **Общий анализ крови (ОАК)**
Hb_г_л = 0  #@param {type:"number"}
MCV_фл = 0  #@param {type:"number"}
MCH_пг = 0  #@param {type:"number"}
RBC_10_12_л = 0  #@param {type:"number"}
RDW_процент = 0  #@param {type:"number"}
Ретикулоциты_процент = 0  #@param {type:"number"}
#@markdown **Обмен железа и витамины (если сдавали)**
Ферритин_нг_мл = 0  #@param {type:"number"}
Насыщение_трансферрина_процент = 0  #@param {type:"number"}
CRP_мг_л = 0  #@param {type:"number"}
B12_пг_мл = 0  #@param {type:"number"}
Фолат_нг_мл = 0  #@param {type:"number"}
#@markdown **Если анализов нет — анкета (отметьте, что верно)**
Ранее_был_низкий_ферритин_или_железо = False  #@param {type:"boolean"}
Обильные_менструации_или_кровопотери = False  #@param {type:"boolean"}
Вегетарианское_или_бедное_мясом_питание = False  #@param {type:"boolean"}
Болезни_ЖКТ_или_операции_на_желудке = False  #@param {type:"boolean"}
Хронические_воспалительные_болезни_или_болезни_почек = False  #@param {type:"boolean"}
Анемия_или_талассемия_у_родственников = False  #@param {type:"boolean"}
Онемение_или_покалывание_в_руках_ногах = False  #@param {type:"boolean"}

def cause_profile(sex, age, pregnant, hb=None, mcv=None, mch=None, rbc=None, rdw=None, retic=None,
                  ferritin=None, tsat=None, crp=None, b12=None, folate=None):
    """Правила для взрослых. Возвращает (статус анемии, список профилей с обоснованием, что проверить)."""
    v = lambda x: x if (x is not None and x > 0) else None
    hb, mcv, mch, rbc, rdw, retic = v(hb), v(mcv), v(mch), v(rbc), v(rdw), v(retic)
    ferritin, tsat, crp, b12, folate = v(ferritin), v(tsat), v(crp), v(b12), v(folate)
    out, todo = [], []
    if hb is None:
        return None, [], ['общий анализ крови (Hb, MCV, MCH, RBC, RDW)']
    hb_dl = hb / 10.0
    thr = who_cutoff(sex, age, pregnant)
    anemia = hb_dl < thr
    status = (f'Hb {hb:.0f} г/л — ' + ('АНЕМИЯ' if anemia else 'норма') +
              f' (порог ВОЗ для вашей группы {thr * 10:.0f} г/л)')
    if anemia:
        grade, _ = who_grade(hb_dl, sex, age, pregnant)
        status += f'; степень: {grade.lower()}'
    if age < 12:
        todo.append('у детей нормы MCV/RBC зависят от возраста — профиль ниже ориентировочный, нужен педиатр')
    inflamed = crp is not None and crp > 5
    micro = mcv is not None and mcv < 80
    macro = mcv is not None and mcv > 100
    normo = mcv is not None and 80 <= mcv <= 100
    mentzer = mcv / rbc if (mcv and rbc) else None

    # --- 1. Дефицит железа
    why = []
    if ferritin is not None:
        # ИСПРАВЛЕНО (аудит): порог ВОЗ 2020 зависит от возраста: < 5 лет — 12 (при воспалении 30), ≥ 5 лет — 15 (70)
        f_cut = (30 if age < 5 else 70) if inflamed else (12 if age < 5 else 15)
        if ferritin < f_cut:
            why.append(f'ферритин {ferritin:g} < {f_cut} нг/мл' + (' (порог ВОЗ при воспалении)' if inflamed else ''))
        elif ferritin < 30 and not inflamed:
            why.append(f'ферритин {ferritin:g} — низко-нормальный (часто используемая клиническая граница 30)')
    if tsat is not None and tsat < 20:
        why.append(f'насыщение трансферрина {tsat:g}% < 20%')
    if micro:
        why.append(f'MCV {mcv:g} фл < 80 (микроцитоз)')
    if mch is not None and mch < 27:
        why.append(f'MCH {mch:g} пг < 27 (гипохромия)')
    if rdw is not None and rdw > 14.5:
        why.append(f'RDW {rdw:g}% > 14.5 (эритроциты разного размера)')
    if mentzer is not None and mentzer > 13:
        why.append(f'индекс Ментцера {mentzer:.1f} > 13')
    strong = ferritin is not None and ferritin < ((30 if age < 5 else 70) if inflamed else (12 if age < 5 else 15))
    if strong or len(why) >= 3:
        out.append(('Профиль железодефицита', 'высокое' if strong else 'среднее', why,
                    'обсудить с врачом причину потери/нехватки железа (питание, менструации, ЖКТ); '
                    'не начинать приём железа без назначения'))
        if ferritin is None:
            todo.append('ферритин (главный показатель запасов железа)')

    # --- 2. Носительство талассемии / гемоглобинопатия
    why = []
    if micro:
        why.append(f'микроцитоз (MCV {mcv:g} фл)')
    if rbc is not None and rbc > 5.0:
        why.append(f'эритроцитов много для такого Hb (RBC {rbc:g} ×10¹²/л)')
    if mentzer is not None and mentzer < 13:
        why.append(f'индекс Ментцера {mentzer:.1f} < 13')
    if rdw is not None and rdw <= 14.5 and micro:
        why.append(f'RDW {rdw:g}% в норме при микроцитозе (эритроциты одинаково мелкие)')
    if ferritin is not None and ferritin >= 30 and micro:
        why.append(f'ферритин {ferritin:g} нормальный — дефицит железа не объясняет микроцитоз')
    if micro and len(why) >= 3:
        conf = 'среднее' if (ferritin is not None and ferritin >= 30) else 'низкое (сначала исключить дефицит железа)'
        out.append(('Возможная гемоглобинопатия (например, носительство талассемии)', conf, why,
                    'электрофорез/ВЭЖХ гемоглобина; при необходимости — генетический тест; '
                    'семейный анамнез. Важно: дефицит железа и талассемия могут сочетаться'))

    # --- 3. Анемия воспаления / хронического заболевания
    why = []
    if inflamed:
        why.append(f'CRP {crp:g} мг/л > 5 (есть воспаление)')
    if ferritin is not None and ferritin >= 100:
        why.append(f'ферритин {ferritin:g} высокий/нормальный')
    if tsat is not None and tsat < 20:
        why.append(f'насыщение трансферрина {tsat:g}% низкое')
    if normo:
        why.append(f'MCV {mcv:g} фл — нормоцитарная картина')
    if anemia and inflamed and (ferritin is None or ferritin >= 30) and len(why) >= 2:
        mixed = ferritin is not None and 30 <= ferritin < 100
        out.append(('Профиль анемии воспаления / хронического заболевания' + (' (возможно, смешанный с дефицитом железа)' if mixed else ''),
                    'среднее', why, 'выяснить источник воспаления, проверить функцию почек (креатинин), обсудить с терапевтом'))

    # --- 4. Дефицит B12 / фолата
    why = []
    b12_low = b12 is not None and b12 < 200
    fol_low = folate is not None and folate < 3
    if macro:
        why.append(f'MCV {mcv:g} фл > 100 (макроцитоз)')
    if b12_low:
        why.append(f'B12 {b12:g} пг/мл < 200')
    elif b12 is not None and b12 < 300:
        why.append(f'B12 {b12:g} пг/мл — пограничный (200–300), дефицит не исключён')
    if fol_low:
        why.append(f'фолат {folate:g} нг/мл < 3')
    if rdw is not None and rdw > 14.5 and macro:
        why.append(f'RDW {rdw:g}% повышен')
    if b12_low or fol_low or (macro and len(why) >= 2):
        out.append(('Профиль дефицита витамина B12 и/или фолиевой кислоты',
                    'высокое' if (b12_low or fol_low) and macro else 'среднее', why,
                    'B12 и фолат (если не сдавали); при пограничных значениях — метилмалоновая кислота/гомоцистеин; '
                    'врач выясняет причину (питание, всасывание, препараты)'))
    if macro and b12 is None:
        todo.append('витамин B12 и фолиевая кислота')

    # --- 5. Нормоцитарная анемия: кровопотеря / гемолиз / костный мозг / почки
    if anemia and normo and not any('воспаления' in o[0] for o in out):
        why = [f'MCV {mcv:g} фл — нормоцитарная картина']
        if retic is not None:
            why.append(f'ретикулоциты {retic:g}% ' + ('повышены — костный мозг активно работает (кровопотеря или гемолиз?)'
                                                    if retic > 2.5 else 'не повышены — недостаточная выработка эритроцитов'))
            nxt = ('искать источник кровопотери; анализы на гемолиз (билирубин, ЛДГ, гаптоглобин)' if retic > 2.5
                   else 'креатинин (почки), ферритин/B12/фолат (смешанный дефицит), консультация гематолога')
        else:
            nxt = 'ретикулоциты, креатинин, ферритин, B12/фолат — чтобы сузить причину'
            todo.append('ретикулоциты')
        out.append(('Нормоцитарная анемия — нужно дообследование', 'низкое', why, nxt))

    if not anemia and not out:
        out.append(('Анемии по анализу нет', 'высокое', [f'Hb {hb:.0f} г/л не ниже порога ВОЗ'],
                    'если ферритин низкий при нормальном Hb — это латентный дефицит железа, обсудите с врачом'))
    if anemia and not out:
        out.append(('Недостаточно данных для профиля', '—', ['есть анемия, но показателей для определения причины мало'],
                    'сдать: ' + ', '.join([n for n, v_ in (('MCV', mcv), ('MCH', mch), ('RBC', rbc), ('RDW', rdw)) if v_ is None]
                                          + ([] if ferritin else ['ферритин']) +
                                          ([] if b12 else ['B12']) + ([] if folate else ['фолат']) +
                                          ([] if crp else ['CRP']))))
    if anemia and ferritin is None and not any('ферритин' in t for t in todo):
        todo.append('ферритин')
    rank = {'высокое': 0, 'среднее': 1, 'низкое': 2}
    out.sort(key=lambda o: rank.get(o[1].split()[0], 3))
    return status, out, todo

def questionnaire_advice(flags):
    """Без анализов причину не определить — анкета подсказывает, ЧТО сдать и какой профиль проверить первым."""
    tips = []
    if flags['iron_hist'] or flags['blood_loss'] or flags['diet'] or flags['gi']:
        tips.append('Факторы риска ДЕФИЦИТА ЖЕЛЕЗА → сдать ОАК + ферритин')
    if flags['diet'] or flags['gi'] or flags['neuro']:
        tips.append('Факторы риска ДЕФИЦИТА B12 → сдать ОАК + витамин B12 (+ фолат)')
    if flags['chronic']:
        tips.append('Хроническое заболевание → ОАК + ферритин + CRP + креатинин')
    if flags['family']:
        tips.append('Анемия/талассемия у родственников → ОАК (MCV, RBC); при микроцитозе с нормальным ферритином — '
                    'электрофорез гемоглобина')
    return tips or ['Явных факторов риска нет → при признаках анемии на фото начните с общего анализа крови (ОАК)']

# ---------------- вывод
sx = 'M' if Пол_ == 'Мужской' else 'F'
print('=' * 84)
if globals().get('RESULT'):
    print(f"📷 Скрининг по фото (ячейка 4): {RESULT['verdict']}, Hb ≈ {RESULT['hb'] * 10:.0f} г/л"
          + ('' if RESULT.get('reliable', True) else ' (ненадёжно)'))
status, profiles, todo = cause_profile(sx, int(Возраст_), bool(Беременность_), Hb_г_л, MCV_фл, MCH_пг, RBC_10_12_л,
                                       RDW_процент, Ретикулоциты_процент, Ферритин_нг_мл,
                                       Насыщение_трансферрина_процент, CRP_мг_л, B12_пг_мл, Фолат_нг_мл)
if status is None:
    print('🩸 Анализа крови нет — тип анемии по фото и симптомам определить НЕЛЬЗЯ.')
    print('   По анкете — что сдать в первую очередь:')
    for t in questionnaire_advice({'iron_hist': Ранее_был_низкий_ферритин_или_железо,
                                   'blood_loss': Обильные_менструации_или_кровопотери,
                                   'diet': Вегетарианское_или_бедное_мясом_питание,
                                   'gi': Болезни_ЖКТ_или_операции_на_желудке,
                                   'chronic': Хронические_воспалительные_болезни_или_болезни_почек,
                                   'family': Анемия_или_талассемия_у_родственников,
                                   'neuro': Онемение_или_покалывание_в_руках_ногах}):
        print('   •', t)
else:
    print('🩸', status)
    print('-' * 84)
    for i, (name, conf, why, nxt) in enumerate(profiles, 1):
        print(f'{i}. {name}   [соответствие: {conf}]')
        for w in why:
            print(f'     ✓ {w}')
        print(f'     ➜ что дальше: {nxt}')
    if todo:
        print('-' * 84)
        print('Для более точного профиля не хватает:', '; '.join(dict.fromkeys(todo)))
print('=' * 84)
print('⚠️ Это профиль для обсуждения с врачом, а НЕ диагноз. Тип анемии устанавливает врач после обследования.')


In [ ]:
#@title ЯЧЕЙКА 6 — (для защиты) ТОЧНОСТЬ: что было и что стало после калибровки
# Все цифры — на пациентах, которых модель НЕ видела при обучении (перекрёстная проверка по пациентам).
GN = ['Норма', 'Лёгкая', 'Умеренная', 'Тяжёлая']
rows = []
for k, name in (('comb', 'Конъюнктива + склера'), ('conj_auto', 'Конъюнктива (фото глаза)'), ('sclera', 'Склера'),
                ('conj_seg', 'Конъюнктива (вырезанная)'), ('conj_child', 'Дети до 5 лет')):
    m = STATE.get(k)
    if not m:
        continue
    e = m['eval']
    en = m.get('eval_nested') or {}
    rows.append({'Метод': name, 'Пациентов': m['n'],
                 'ЧЕСТНО: ошибка Hb, г/л': en.get('MAE', np.nan) * 10,
                 'ЧЕСТНО: AUC': en.get('AUC', np.nan),
                 'ЧЕСТНО: чувствит.': en.get('sens', np.nan), 'ЧЕСТНО: специф.': en.get('spec', np.nan),
                 'ЧЕСТНО: оценка выдана': en.get('covered', np.nan),
                 'Ошибка Hb, г/л (оптимистично)': e['calibrated']['MAE'] * 10,
                 'Верная степень: было': e['raw']['balanced_acc'], 'стало': e['calibrated']['balanced_acc'],
                 'Ложная «анемия» у здоровых: было': e['raw']['false_anemia_healthy'],
                 'стало ': e['calibrated']['false_anemia_healthy'],
                 'Пропуск анемии: было': e['raw']['missed_anemia'], 'стало  ': e['calibrated']['missed_anemia'],
                 '«Пограничный»': e.get('borderline_share', np.nan), 'AUC (оптимистично)': e.get('AUC', np.nan)})
T = pd.DataFrame(rows)
NUM = ('Метод', 'Пациентов', 'ЧЕСТНО: ошибка Hb, г/л', 'Ошибка Hb, г/л (оптимистично)', 'ЧЕСТНО: AUC', 'AUC (оптимистично)')
fmt = {c: '{:.0%}' for c in T.columns if c not in NUM}
fmt.update({'ЧЕСТНО: ошибка Hb, г/л': '±{:.1f}', 'Ошибка Hb, г/л (оптимистично)': '±{:.1f}',
            'ЧЕСТНО: AUC': '{:.3f}', 'AUC (оптимистично)': '{:.3f}'})
display(T.style.format(fmt, na_rep='—'))
T.to_csv(f'{OUT}/accuracy_before_after.csv', index=False)
print('ЧЕСТНО — вложенная перекрёстная проверка: порог HHR, набор признаков, модель и калибровка выбираются без\n'
      'проверяемых людей. «Оптимистично» — цифры v10: вариант выбран по той же ошибке, что показана (для сравнения).\n'
      'Говорить на защите — только «ЧЕСТНО».')
print('«Верная степень» — сбалансированная точность по 4 классам (норма/лёгкая/умеренная/тяжёлая):\n'
      'каждый класс весит одинаково, поэтому редкие тяжёлые случаи не «тонут» среди нормальных.\n'
      '«Ложная анемия у здоровых» — доля людей с нормальным Hb, которым система показала «анемию»\n'
      '(«пограничный результат» сюда не входит — он честно говорит «сдайте анализ»).')
det = STATE.get('detector')
if det:
    print(f"Автопоиск на фото датасета: глаз найден у {det.get('eye_found', det['found'])} из {det['of']}, "
          f"конъюнктива — у {det['found']} (поиск: {det.get('method', 'правила')}); "
          f"доля найденной области внутри разметки врача (медиана) {det['overlap_median'] or 0:.0%}, "
          f"IoU {det.get('iou_median') or 0:.0%}.")
print('\nУстойчивость к свету и камере (стресс-тест: снимки проверяемых пациентов при других лампах,\n'
      'экспозиции и обработке телефона; ошибка Hb, усреднённая по диапазонам Hb, г/л — обычный / другой свет):')
for k, name in (('conj_auto', 'Конъюнктива (фото глаза)'), ('sclera', 'Склера'), ('conj_seg', 'Вырезанная конъюнктива'),
                ('conj_child', 'Дети до 5 лет')):
    m = STATE.get(k)
    if m and m.get('alternatives'):
        alt = '; '.join(f"«{a}» ±{c * 10:.0f}/±{l * 10:.0f}" for a, (c, l) in m['alternatives'].items())
        print(f"   {name}: выбран набор «{m['feature_set']}» — {alt}"
              + (f"; без обучения на другом свете (до v10) ±{m['baseline'][0] * 10:.0f}/±{m['baseline'][1] * 10:.0f}"
                 if m.get('baseline') else ''))
sg = STATE.get('conj_segmenter')
if sg:
    print(f"Поиск конъюнктивы, проверка на {sg['n']} фото пациентов, которых модель не видела: совпадение с врачами "
          f"(IoU, медиана) — модель {sg['iou_learned']:.0%}, правила {sg['iou_rule']:.0%} → "
          f"{'в приложении работает модель' if sg['use'] else 'в приложении работают правила'}.")

key = 'comb' if STATE.get('comb') else 'conj_auto' if STATE.get('conj_auto') else 'conj_seg'
e = STATE[key]['eval']
fig, ax = plt.subplots(1, 3, figsize=(21, 5.5))
for a, part, title in ((ax[0], 'raw', 'БЫЛО: прогноз без калибровки'), (ax[1], 'calibrated', 'СТАЛО: после калибровки')):
    M = np.array(e[part]['confusion'], float)
    Mn = M / np.maximum(M.sum(1, keepdims=True), 1)
    a.imshow(Mn, cmap='Greens', vmin=0, vmax=1)
    for i in range(4):
        for j in range(4):
            a.text(j, i, f'{int(M[i, j])}\n{Mn[i, j]:.0%}', ha='center', va='center', fontsize=10)
    a.set_xticks(range(4)); a.set_xticklabels(GN); a.set_yticks(range(4)); a.set_yticklabels(GN)
    a.set_xlabel('Прогноз по фото'); a.set_ylabel('Истина (анализ крови)')
    a.set_title(f"{title}\nверная степень {e[part]['balanced_acc']:.0%}")
b_raw, b_cal = e['raw']['bias'], e['calibrated']['bias']
names = [n for n in ['Норма', 'Лёгкая анемия', 'Умеренная анемия', 'Тяжёлая анемия'] if n in b_raw]
xx = np.arange(len(names))
ax[2].bar(xx - 0.2, [b_raw[n] * 10 for n in names], 0.4, label='было', color='#bdbdbd')
ax[2].bar(xx + 0.2, [b_cal[n] * 10 for n in names], 0.4, label='стало', color='#1565c0')
ax[2].axhline(0, color='k'); ax[2].set_xticks(xx); ax[2].set_xticklabels([n.split()[0] for n in names])
ax[2].set_ylabel('Сдвиг оценки Hb, г/л (+ завышает, − занижает)'); ax[2].legend()
ax[2].set_title('Систематический сдвиг по степеням анемии')
plt.tight_layout(); plt.savefig(f'{OUT}/slide_accuracy_before_after.png', dpi=160, bbox_inches='tight'); plt.show()


## Повысить точность для вашего телефона
Добавляйте фото людей, у которых есть свежий анализ крови (Hb). После «Выполнить все» модели переобучатся с этими данными.

In [ ]:
#@title ЯЧЕЙКА 7 — ДОБАВИТЬ фото с ИЗВЕСТНЫМ гемоглобином (самый надёжный способ повысить точность) { display-mode: "form" }
# Нужны: фото глаза (как для анализа) + Hb из анализа крови, сданного НЕ раньше чем за 7 дней до фото.
# Фото и признаки сохраняются в папку local_data (на Диске или в local_data.zip без Диска). При следующем
# «Выполнить все» модели автоматически переобучатся с вашими данными (их вес ×3: они сняты
# ВАШИМ телефоном и при ВАШЕМ свете — именно этого не хватает исследовательским датасетам).
Hb_из_анализа_г_л = 0  #@param {type:"integer"}
Пол_человека = 'Женский'  #@param ["Женский", "Мужской"]
Возраст_человека = 25  #@param {type:"integer"}

import shutil, time
if Hb_из_анализа_г_л < 40 or Hb_из_анализа_г_л > 220:
    print('Впишите Hb из анализа крови в г/л (например 118) и запустите ячейку снова.')
else:
    os.makedirs(LOCAL_DIR, exist_ok=True)
    try:
        from google.colab import files
        print('Выберите 1–3 фото глаза ЭТОГО человека:')
        up = list(files.upload().keys())
    except ImportError:
        up = []
    sx = 'M' if Пол_человека == 'Мужской' else 'F'
    rows = []
    PERSON_ID = f"LOCALP_{time.strftime('%Y%m%d_%H%M%S')}"   # ИСПРАВЛЕНО (аудит): все фото этого запуска — один человек,
    for p in up:                                           # одна группа в перекрёстной проверке (без утечки)
        key = f"LOCAL_{time.strftime('%Y%m%d_%H%M%S')}_{len(rows)}"
        dst = f'{LOCAL_DIR}/{key}{os.path.splitext(p)[1].lower() or ".jpg"}'
        shutil.copy(p, dst)
        f = _eye_one(key, dst)                           # тот же путь, что при анализе
        if f is None or not any(k.startswith(('c_', 's_')) for k in f):
            print(f'❌ {p}: глаз не найден (или не видны ни веко, ни белок) — это фото не добавлено'); continue
        f.update(Hb=Hb_из_анализа_г_л / 10.0, sex=sx, age=int(Возраст_человека), photo=dst, person=PERSON_ID)
        rows.append(f)
        print(f"✅ {p}: добавлено ({'конъюнктива' if 'c_EI' in f else ''}{' + ' if 'c_EI' in f and 's_scl_L' in f else ''}{'склера' if 's_scl_L' in f else ''})")
    if rows:
        path = f'{LOCAL_DIR}/features.csv'
        old = pd.read_csv(path) if os.path.exists(path) else pd.DataFrame()
        allf = pd.concat([old, pd.DataFrame(rows)], ignore_index=True)
        allf.to_csv(path, index=False)
        print(f'\n📦 В вашей базе {len(allf)} фото. Нажмите «Выполнить все» — модели переобучатся с ними.')
        if not globals().get('ON_DRIVE', True):          # без Диска папка исчезнет при перезапуске Colab
            try:
                shutil.make_archive(f'{DRIVE_DIR}/local_data', 'zip', DRIVE_DIR, 'local_data')
                from google.colab import files
                files.download(f'{DRIVE_DIR}/local_data.zip')
                print('💾 Скачан local_data.zip — сохраните его и в следующий раз выберите в ячейке 1 '
                      'вместе с файлом моделей.')
            except Exception:
                pass
        print('   Цель: 30–50 человек с РАЗНЫМ Hb (и с анемией, и без) — тогда точность для вашего телефона '
              'станет заметно выше.')
